# PTCG — the `v5_s2` recipe on **every** host-released episode dataset

This notebook rebuilds the shipped agent's policy net from scratch on Kaggle,
using the same behavioural-cloning recipe that produced `out/policy_v5_s2.npz`,
but over **every daily episode dataset the competition hosts have released**
(`kaggle/pokemon-tcg-ai-battle-episodes-YYYY-MM-DD`, 2026-06-16 onwards) instead
of the four days `v5_s2` was trained on.

**Nothing about the agent changes except the corpus.** The trainer and the corpus
builder below are embedded verbatim from the repo, and the training command is
the day-25 recipe unchanged:

```
scripts/train_policy.py --ds <corpus> --epochs 12 --bs 1024 --loss listwise \
    --state-h 512,256 --head-h 256,128 --pool --opt-cols 37 --seed 2
```

`--opt-cols 37` is load-bearing: the builder writes 46-wide option features and
the v6 attribute block is *appended*, so 37 slices exactly the v5 layout. A net
trained without it cannot share an ensemble with `policy_v5`.

---

## What it does, in order

1. Rebuilds the repo tree under `/kaggle/working/ptcg` from `%%writefile` cells.
2. Pulls the `cg` engine (`kiyotah/cg-lib`) — the featurizer needs its card and
   attack tables.
3. Probes every daily dataset from `START_DATE` to today, reads each
   `manifest.csv`, and takes the top `EPISODES_PER_DAY` episodes by `avg_score`.
4. Per day: download the replays → build shards → **delete the replays**, so
   peak disk stays ~2 GB instead of ~1.3 TB.
5. Trains, verifies the export loads through `sa/policynet.py`, and leaves the
   net + corpus in `/kaggle/working` to download.

## ⚠ "Every dataset" means every DAY, top-N per day. Read this before raising N.

Every day's dataset holds ~4,600 episodes and **~21.5 GB of JSON**. All 59 days
is **~1.3 TB and ~42 M training rows**. That is not a Kaggle-sized job and no
setting of the knobs below makes it one — the ceiling is RAM, not the hosts:

`train_policy.Data` concatenates the whole corpus into memory, and it holds the
per-shard arrays *and* the concatenated copy alive at the same time. Measured on
`artifacts/pds_all`: **~4.0 KB/row resident, ~7.8 KB/row at the load peak.**
42 M rows would need ~330 GB. Kaggle gives you ~29 GB.

| `EPISODES_PER_DAY` | episodes | rows | load peak | build | train (12 ep) |
|---|---|---|---|---|---|
| 150 | ~8,850 | ~1.3 M | ~10 GB | ~25 min | ~1.5 h CPU / ~35 min GPU |
| **300** (default) | ~17,700 | ~2.7 M | **~21 GB** | ~45 min | ~3 h CPU / ~1.2 h GPU |
| 400 | ~23,600 | ~3.5 M | ~28 GB | ~1 h | **OOM risk on a 29 GB box** |
| 600 | ~35,400 | ~5.3 M | ~41 GB | ~1.5 h | will not load |

For reference, `v5_s2` itself was trained on **4 days × 400 episodes ≈ 250 K
rows**. The default here is ~11× that corpus, drawn from 15× as many days.

Episodes are taken **top-N by `avg_score`**, which is the same selection rule
`scripts/fetch_top_episodes.py --max 400` uses for the repo's own dumps — so
this is the v5_s2 corpus's own rule applied to every released day, not a new one.

**Set `SMOKE = True` for a ~5 minute end-to-end rehearsal before committing a
multi-hour session.**

## Requirements

* **Internet ON** (Settings → Internet). kagglehub cannot reach the datasets
  without it, and the failure looks like every date being "missing".
* **GPU T4 x2** recommended (`DEVICE = "cuda"`). `DEVICE = "cpu"` reproduces
  `v5_s2`'s numerics exactly but takes ~3 h for the default corpus.
* Nothing needs to be attached as a data source, and **no API key or secret is
  needed** — on Kaggle, kagglehub authenticates from the token file the kernel
  already has (`KAGGLE_API_V1_TOKEN_PATH`).

⚠ The scaffold cell sets `DISABLE_KAGGLE_CACHE=1`. Do not remove it — see the
comment there; without it this notebook silently trains on an empty corpus.


In [ ]:
# ─────────────────────────────────────────────────────────────── CONFIG ──────
SMOKE            = False   # True = 2 days x 12 episodes x 1 epoch, ~5 min rehearsal

ROOT             = "/kaggle/working/ptcg"
START_DATE       = "2026-06-16"   # the first daily dataset the hosts published
END_DATE         = None           # None = today (UTC). Dates with no dataset are skipped.
EPISODES_PER_DAY = 0              # top-N of each day's manifest by avg_score. 0 = ALL.
                                  # ⚠ read the RAM table in the header before raising this
                                  # for a run that TRAINS; a build-only run is unconstrained.
BUILD_PROCS      = 4              # parallel builder processes (Kaggle gives 4 vCPU)
TOP_PCT          = 0              # 0 = every episode. 10 = keep only the top 10%
                                  # of episodes by avg_score. `gid` IS the
                                  # episode id, so this is a row mask over the
                                  # SAME corpus -- no rebuild.
                                  # ⚠ the cut is applied to the SHARDS by the
                                  # filter cell, NOT passed to the trainer as a
                                  # row mask -- see that cell for why.
ARCHETYPE        = ""             # "" = rank every episode together (RUN 1).
                                  # "grimmsnarl" = keep only episodes the
                                  # DEMONSTRATOR played that deck in, and rank
                                  # TOP_PCT within them (RUN 2). ⚠ the quality
                                  # bar then floats to that deck's own rating
                                  # distribution -- the cutoff will NOT be the
                                  # global one.
KEEP_GIDS        = ""             # set by the cut cell below, cleared by the filter cell
STREAM           = False          # load shards a buffer at a time. REQUIRED above
                                  # ~4M rows: the in-RAM path wants 7.8 KB/row
                                  # (313 GB at 40.1M) against a 34 GB box.
                                  # ⚠ shuffling becomes buffer-local -- declare it.
STREAM_BUFFER    = 8              # shards resident per buffer under STREAM.
                                  # ⚠ this counts SHARDS, so what it means
                                  # depends on rows/shard. It is comparable to
                                  # the unfiltered run ONLY because the filter
                                  # cell repacks to the same ~57k rows/shard.
MAX_HOURS        = 0.0            # >0: stop after the first epoch past this many
                                  # hours and exit 0. Kaggle DISCARDS output when
                                  # it kills a kernel at the 12 h cap, so a run
                                  # that might overrun must stop itself.
INIT_NET         = ""             # "" = train from scratch. A filename (e.g.
                                  # "policy_v5_s2_hostall.npz") warm-starts from
                                  # that net, found by glob under /kaggle/input.
                                  # ⚠ WARM START, NOT RESUME: --init restores
                                  # WEIGHTS only. AdamW's moment estimates reset,
                                  # so a chained 4+8 is NOT a contiguous 12 --
                                  # declare the chain in any result that uses it.
SKIP_TRAIN       = False          # True = build the corpus and stop (for scale tests
                                  # and for the build half of a split pipeline)
THROUGHPUT_TEST  = True           # measure mount read scaling before building

# ── the v5_s2 recipe, verbatim (HANDOFF, "THE DAY-25 PLAN") ──────────────────
EPOCHS    = 12
BS        = 1024
LOSS      = "listwise"
STATE_H   = "512,256"
HEAD_H    = "256,128"
OPT_COLS  = 37        # ⚠ load-bearing: slices the v5 option layout out of the 46-wide corpus
SEED      = 2
POOL      = True
# ─────────────────────────────────────────────────────────────────────────────

DEVICE       = "cuda"   # "cpu" == v5_s2's exact numerics; "cuda" is ~3x faster
CORPUS       = "artifacts/pds_hostall"
NET_NAME     = "policy_v5_s2_hostall.npz"
RATINGS_ZIP  = ""       # optional Kaggle LB export (.zip/.csv) -> enables val_top1@1120+.
                        # Attach it as a dataset and point at the file; leave "" to skip.

if SMOKE:
    EPISODES_PER_DAY, EPOCHS = 12, 1
    CORPUS, NET_NAME = "artifacts/pds_smoke", "policy_smoke.npz"

print(f"corpus  {CORPUS}   net  {NET_NAME}   episodes/day {EPISODES_PER_DAY}")


In [ ]:
import os, sys, shutil, subprocess

# ⚠ Episodes come from ATTACHED datasets (/kaggle/input), never downloaded.
# Per-file kagglehub downloads are rate-limited to ~60 requests per ~20 min
# (429 on DownloadDataset, and it fails serial too -- it is a request budget,
# not a concurrency problem). A 17,700-file run measured 98.9% failures and
# still exited 0. Declare the days in kernel-metadata `dataset_sources`
# instead: Kaggle mounts them before the kernel starts, at zero disk cost and
# zero API calls. The cap is ~50 sources per kernel, so 59 days needs two.
# kagglehub is still used for the cg engine only -- one small request.

for sub in ("src/ptcg/env", "agents/sa", "scripts", "data", "replays",
            "artifacts", "out"):
    os.makedirs(f"{ROOT}/{sub}", exist_ok=True)
os.chdir(ROOT)                      # every %%writefile below is relative to here
# CORPUS is a repo-relative dir when this run BUILDS one, and an absolute
# /kaggle/input path when it trains off a corpus another kernel built.
CORPUS_DIR = CORPUS if os.path.isabs(CORPUS) else f"{ROOT}/{CORPUS}"
print("cwd:", os.getcwd())

# The repo puts `src` and `agents` on sys.path; the scripts do it themselves,
# but the verification cells at the bottom import from this kernel.
for sub in ("src", "agents", ""):
    p = os.path.join(ROOT, sub) if sub else ROOT
    if p not in sys.path:
        sys.path.insert(0, p)


## The repo, verbatim

Every cell below is a byte-identical copy of a file in the repo, embedded at generation time by
`scripts/make_kaggle_notebook.py`. **Do not edit them here** — edit the source file and regenerate,
or the net this notebook trains stops being comparable to `policy_v5_s2.npz`.


In [ ]:
%%writefile src/ptcg/__init__.py



In [ ]:
%%writefile src/ptcg/config.py
"""Repo paths and SDK discovery."""
from __future__ import annotations

from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = ROOT / "data"
DIST_DIR = ROOT / "dist"
OUT_DIR = ROOT / "out"


def find_sdk_dir() -> Path | None:
    """Directory that *contains* the `cg/` package (so it can go on sys.path)."""
    for api in sorted(DATA_DIR.glob("**/cg/api.py")):
        return api.parents[1]
    return None


def find_sample_deck() -> Path | None:
    for deck in sorted(DATA_DIR.glob("**/deck.csv")):
        return deck
    return None


In [ ]:
%%writefile src/ptcg/env/__init__.py



In [ ]:
%%writefile src/ptcg/env/sdk.py
"""Load the licensed `cg` engine from data/ and expose it as modules.

The engine is a single-battle-at-a-time ctypes wrapper, so everything here is
process-global: `load()` puts the SDK dir on sys.path and imports `cg`.
"""
from __future__ import annotations

import sys
from functools import lru_cache

from ptcg import config

_loaded = None


def load():
    """Import and return the `cg` package (idempotent)."""
    global _loaded
    if _loaded is not None:
        return _loaded
    sdk_dir = config.find_sdk_dir()
    if sdk_dir is None:
        raise RuntimeError("cg engine not found under data/ (need **/cg/api.py)")
    if str(sdk_dir) not in sys.path:
        sys.path.insert(0, str(sdk_dir))
    import cg  # noqa: F401
    import cg.api  # noqa: F401
    import cg.game  # noqa: F401
    _loaded = cg
    return cg


def api():
    return load().api


def game():
    return load().game


@lru_cache(maxsize=1)
def card_db() -> dict:
    """cardId -> CardData for every card the engine knows."""
    return {c.cardId: c for c in api().all_card_data()}


@lru_cache(maxsize=1)
def attack_db() -> dict:
    """attackId -> Attack."""
    return {a.attackId: a for a in api().all_attack()}


In [ ]:
%%writefile agents/sa/__init__.py



In [ ]:
%%writefile agents/sa/cards.py
"""Static card/attack data pulled once from the engine, dict-indexed."""
from __future__ import annotations

import json

from cg.sim import lib

# EnergyType ints (mirror cg.api without importing it)
COLORLESS = 0
RAINBOW = 10
TEAM_ROCKET = 11

_cards: dict[int, dict] | None = None
_attacks: dict[int, dict] | None = None


def cards() -> dict[int, dict]:
    global _cards
    if _cards is None:
        _cards = {c["cardId"]: c
                  for c in json.loads(lib.AllCard().decode())}
    return _cards


def attacks() -> dict[int, dict]:
    global _attacks
    if _attacks is None:
        _attacks = {a["attackId"]: a
                    for a in json.loads(lib.AllAttack().decode())}
    return _attacks


def card(cid: int) -> dict:
    return cards().get(cid) or {}


def is_pokemon(cid: int) -> bool:
    return card(cid).get("cardType") == 0


def is_basic_pokemon(cid: int) -> bool:
    c = card(cid)
    return c.get("cardType") == 0 and bool(c.get("basic"))


def is_basic_energy(cid: int) -> bool:
    return card(cid).get("cardType") == 5


def prize_value(cid: int) -> int:
    """Prizes the opponent takes when this Pokemon is KO'd."""
    c = card(cid)
    if c.get("megaEx"):
        return 3
    if c.get("ex"):
        return 2
    return 1


def energy_satisfied(cost: list[int], have: list[int]) -> bool:
    """Can `have` (attached energy units) pay `cost`? Colorless is wildcard;
    RAINBOW counts as any type; TEAM_ROCKET as psychic(5)/darkness(7)."""
    have = list(have)
    for e in cost:
        if e == COLORLESS:
            continue
        if e in have:
            have.remove(e)
        elif RAINBOW in have:
            have.remove(RAINBOW)
        elif e in (5, 7) and TEAM_ROCKET in have:
            have.remove(TEAM_ROCKET)
        else:
            return False
    n_colorless = sum(1 for e in cost if e == COLORLESS)
    return len(have) >= n_colorless


def best_usable_damage(cid: int, energies: list[int]) -> int:
    """Max printed damage among this Pokemon's attacks payable with
    `energies`. Text effects (bonus damage, coins) are not modeled."""
    best = 0
    atk_db = attacks()
    for aid in card(cid).get("attacks") or []:
        a = atk_db.get(aid)
        if a and energy_satisfied(a["energies"], energies):
            best = max(best, a.get("damage") or 0)
    return best


In [ ]:
%%writefile agents/sa/textdmg.py
"""Expected-damage estimation from attack text (pattern-based, approximate).

Used only inside the heuristic eval's threat terms; playouts always see real
simulated damage. Estimates are compiled once per attackId.
"""
from __future__ import annotations

import re
from functools import lru_cache

from . import cards as cdb

_ENERGY_LETTER = {"G": 1, "R": 2, "W": 3, "L": 4, "P": 5, "F": 6, "D": 7,
                  "M": 8, "N": 9}


@lru_cache(maxsize=4096)
def _analyze(aid: int) -> tuple:
    """-> tuple of (kind, per, arg) modifiers for this attack's text."""
    a = cdb.attacks().get(aid) or {}
    text = (a.get("text") or "").replace("\n", " ")
    mods: list[tuple] = []

    m = re.search(r"Flip (\d+) coins?\. This attack does (\d+) damage for each"
                  r" heads", text)
    if m:
        mods.append(("flat", 0.5 * int(m.group(1)) * int(m.group(2)), None))
    if re.search(r"Flip a coin until you get tails", text):
        m2 = re.search(r"(\d+) damage for each heads", text)
        if m2:
            mods.append(("flat", float(m2.group(1)), None))
    m = re.search(r"Flip a coin\. If heads, this attack does (\d+) more damage",
                  text)
    if m:
        mods.append(("flat", 0.5 * int(m.group(1)), None))

    for m in re.finditer(
            r"does (\d+)(?: more)? damage for each (?:\{(\w)\} )?Energy "
            r"attached to (this Pok\S+mon|all of your[^,.]*|your opponent"
            r"\S+s Active Pok\S+mon|both Active Pok\S+mon)", text):
        per = int(m.group(1))
        etype = _ENERGY_LETTER.get(m.group(2)) if m.group(2) else None
        scope = m.group(3)
        if scope.startswith("this"):
            mods.append(("energy_self", per, etype))
        elif scope.startswith("all of your"):
            mods.append(("energy_all_mine", per, etype))
        elif scope.startswith("both"):
            mods.append(("energy_both_actives", per, etype))
        else:
            mods.append(("energy_opp_active", per, etype))

    m = re.search(r"does (\d+)(?: more)? damage for each of your Benched",
                  text)
    if m:
        mods.append(("bench_mine", int(m.group(1)), None))
    m = re.search(r"does (\d+)(?: more)? damage for each of your opponent"
                  r"\S+s Benched", text)
    if m:
        mods.append(("bench_opp", int(m.group(1)), None))

    m = re.search(r"does (\d+)(?: more)? damage for each damage counter on "
                  r"(this Pok\S+mon|your opponent\S+s Active)", text)
    if m:
        mods.append(("counters_self" if m.group(2).startswith("this")
                     else "counters_opp", int(m.group(1)), None))

    m = re.search(r"does (\d+) damage for each card in your opponent\S+s hand",
                  text)
    if m:
        mods.append(("opp_hand", int(m.group(1)), None))
    m = re.search(r"does (\d+) damage for each Prize card your opponent has "
                  r"taken", text)
    if m:
        mods.append(("opp_prizes_taken", int(m.group(1)), None))

    m = re.search(r"Discard (?:up to (\d+)|all) (?:\{(\w)\} )?Energy.{0,40}?"
                  r"does (\d+) damage for each", text)
    if m:
        cap = int(m.group(1)) if m.group(1) else 99
        etype = _ENERGY_LETTER.get(m.group(2)) if m.group(2) else None
        mods.append(("discard_energy", int(m.group(3)), (cap, etype)))

    # generic conditional bonus: assume it's live half the time
    if not mods:
        m = re.search(r"If [^.]{3,80}, this attack does (\d+) more damage",
                      text)
        if m:
            mods.append(("flat", 0.5 * int(m.group(1)), None))

    return tuple(mods)


def _count_energy(energies: list[int], etype: int | None) -> int:
    if etype is None:
        return len(energies)
    return sum(1 for e in energies if e == etype or e >= 10)


def estimate(aid: int, attacker: dict, mypl: dict, oppl: dict) -> float:
    """Expected damage of attack `aid` used by `attacker` (a pokemon dict on
    mypl's side) against oppl's active, before weakness/resistance."""
    a = cdb.attacks().get(aid) or {}
    dmg = float(a.get("damage") or 0)
    for kind, per, arg in _analyze(aid):
        if kind == "flat":
            dmg += per
        elif kind == "energy_self":
            dmg += per * _count_energy(attacker["energies"], arg)
        elif kind == "energy_all_mine":
            total = 0
            for pk in list(mypl["active"]) + list(mypl["bench"]):
                if pk is not None:
                    total += _count_energy(pk["energies"], arg)
            dmg += per * total
        elif kind == "energy_opp_active":
            act = oppl["active"][0] if oppl["active"] else None
            if act:
                dmg += per * _count_energy(act["energies"], arg)
        elif kind == "energy_both_actives":
            for pl in (mypl, oppl):
                act = pl["active"][0] if pl["active"] else None
                if act:
                    dmg += per * _count_energy(act["energies"], arg)
        elif kind == "bench_mine":
            dmg += per * sum(1 for pk in mypl["bench"] if pk is not None)
        elif kind == "bench_opp":
            dmg += per * sum(1 for pk in oppl["bench"] if pk is not None)
        elif kind == "counters_self":
            dmg += per * max(0, (attacker["maxHp"] - attacker["hp"]) // 10)
        elif kind == "counters_opp":
            act = oppl["active"][0] if oppl["active"] else None
            if act:
                dmg += per * max(0, (act["maxHp"] - act["hp"]) // 10)
        elif kind == "opp_hand":
            dmg += per * oppl["handCount"]
        elif kind == "opp_prizes_taken":
            dmg += per * (6 - len(oppl["prize"]))
        elif kind == "discard_energy":
            cap, etype = arg
            dmg += per * min(cap, _count_energy(attacker["energies"], etype))
    return dmg


def best_estimated_damage(attacker: dict, mypl: dict, oppl: dict) -> float:
    """Max expected damage over the attacker's payable attacks."""
    best = 0.0
    for aid in cdb.card(attacker["id"]).get("attacks") or []:
        a = cdb.attacks().get(aid)
        if a and cdb.energy_satisfied(a["energies"], attacker["energies"]):
            best = max(best, estimate(aid, attacker, mypl, oppl))
    return best


In [ ]:
%%writefile agents/sa/targeting.py
"""Aim chip damage at the Pokemon it can actually finish.

Measured defect (`scripts/opportunity_audit.py`, 80 games): when the clone
chooses which of the opponent's Pokemon to point an effect at, it picks the
lowest-HP candidate 25.7% of the time for Adrena-Brain's counter move and 42.1%
for Shadow Bullet's bench snipe. With 2-4 candidates on board that is chance.

The cause is in the features, not the weights -- but ⚠ **the original statement of
it here was WRONG, corrected 2026-07-30** (`report/EVIDENCE.md` §8f). It read "no
HP and no damage", which is false: `features.py` has always given the net per-slot
HP, damage fraction, attached energy and prize value for all 12 slots.

**The actual defect is narrower.** The v2 per-option vector encoded position only
as *area* flags (active / bench / hand) and **never encoded `opt["index"]`**. So
two options naming two different benched Pokemon were identical vectors apart from
the card-id embedding -- and two options naming **two copies of the same card were
bitwise identical inputs with different right answers.** The net could see the
board; it could not see which option pointed where. That is why these rules win:
they restore a missing *binding*, not missing arithmetic.

`optfeat` v3 gives the net that binding directly (target HP, dies-to-30, own-type
energy, our damage into it, and the slot index). Whether it makes these rules
redundant is ROADMAP B1, measured head-to-head in the arena -- not by val
accuracy, which has failed to predict strength five times (rule 3).

Both effects deal exactly 30: Shadow Bullet's "30 damage to 1 of your
opponent's Benched Pokemon", and Adrena-Brain's "move up to 3 damage counters".
So the rule is: if anything dies to 30, kill the one worth the most prizes;
otherwise concentrate on the one closest to dying.

Deliberately narrow. It fires only when *every* option in the select resolves
to an opponent Pokemon, which leaves the mixed selects (Adrena-Brain's "from 1
of YOUR Pokemon" source pick) to the net.

`energy_spread` is the same idea one select over: the net cannot see how much
energy an ATTACH target already carries, so it stacks a second {D} on a
Munkidori that already has one instead of arming a bare one.
"""
from __future__ import annotations

from . import cards
from .textdmg import estimate

# SelectContext ints (mirror cg.api without importing it)
DAMAGE_COUNTER = 13
DAMAGE_COUNTER_ANY = 14
DAMAGE = 15
CHIP_CONTEXTS = (DAMAGE_COUNTER, DAMAGE_COUNTER_ANY, DAMAGE)

# AreaType ints
_HAND, _ACTIVE, _BENCH = 2, 4, 5

MAIN = 0            # SelectContext.MAIN
SWITCH = 3          # SelectContext.SWITCH -- what Boss's Orders drags with
REMOVE_DAMAGE_COUNTER = 16   # Adrena-Brain's SOURCE pick, all options ours
MAX_MOVE = 3        # Adrena-Brain moves "up to 3 damage counters"
OPT_PLAY = 7        # OptionType.PLAY
OPT_ATTACH = 8      # OptionType.ATTACH
BOSS_ORDERS = 1182
MUNKIDORI = 112
POFFIN = 1086       # Buddy-Buddy Poffin -- E11
DARK_ENERGY = 7     # Basic {D} Energy, card id
DARK_TYPE = 7       # ... and energy type; pk["energies"] holds types

# Shadow Bullet's bench snipe and Adrena-Brain's 3 counters are both 30.
CHIP_DAMAGE = 30

# Pokemon whose ABILITY prevents damage from our {ex} attacks, so our only way
# to remove them is damage counters. Verified in-engine 2026-07-30 over 60 games
# (`scripts/p3_crustle_probe.py`): 209 of 224 attack-damage events onto Crustle
# logged `value: 0` (93.3% prevented), while 1,298 of 1,386 damage-counter events
# landed (93.7%). See `report/EVIDENCE.md` §8d.
#
# ⚠ Hardcoded card ids because the card db exposes no ability text for 345
# (`abilities: None`), so the condition cannot be read off the card, and
# `best_damage` does not model prevention either -- it happily reports 180 for
# Shadow Bullet into a Crustle. V10 hardcodes 344/345 the same way. The general
# version would learn it in-game from the event log (an attack that logged 0
# against this card id), which is the upgrade path if a second wall appears.
WALL_POKEMON = frozenset({345})   # Crustle -- Mysterious Rock Inn

# --- E21: Petrel's fetch (day 30) --------------------------------------------
FETCH = 7            # SelectContext of a "search your deck" resolution
PETREL = 1219        # Team Rocket's Petrel -- "search your deck for a Trainer"
SPIKEMUTH = 1259     # Spikemuth Gym (Stadium, x4)
TOOL_SCRAPPER = 1137  # "discard up to 2 Pokemon Tools" (x1)


def _pokemon_at(state: dict, player: int, area: int, index: int) -> dict | None:
    try:
        pl = state["players"][player]
        if area == _ACTIVE:
            act = pl["active"]
            return act[0] if act and act[0] is not None else None
        if area == _BENCH:
            bench = pl["bench"]
            if 0 <= index < len(bench):
                return bench[index]
    except (KeyError, IndexError, TypeError):
        return None
    return None


def chip_target(obs: dict, wall_defer: bool = False) -> list[int] | None:
    """Ranked option indices for an opponent-targeting select, else None.

    `wall_defer` (`bc:<label>,wall`) is the matchup branch measured on
    2026-07-30. This rule ranks "dies to 30 first, most prizes among those, then
    lowest HP" -- which is right when prizes are the currency, and **wrong
    against a deck whose Active cannot be damaged by our attacks at all**. There,
    damage counters are the only way to remove the blocker, and spending them to
    farm a 1-prize Dwebble loses the game slowly.

    Measured (n=2000 each, fixed `rule:crustle` opponent): `bc` scores 0.559 and
    `bc:x,noChip` scores **0.685** -- this rule is worth **-0.126** in that
    matchup while being worth +0.077 head-to-head in the mirror. The cause is
    measured too: with the rule on, 235 counter-placement events land on Dwebble
    and 1,386 on Crustle; with it off, Dwebble drops to **24** and Crustle rises
    to **1,583** at a higher mean (12.9 -> 15.0). `report/EVIDENCE.md` §8c.

    So when their Active is a wall, hand the select back to the net -- which was
    measured to concentrate counters correctly on its own. Deliberately the
    one-line version of the fix: a bespoke wall-aware ranker is only worth
    building if this fails to recover the -0.126 (HANDOFF §3.3).
    """
    sel = obs.get("select") or {}
    if sel.get("context") not in CHIP_CONTEXTS:
        return None
    options = sel.get("option") or []
    if len(options) < 2:
        return None
    state = obs.get("current") or {}
    me = state.get("yourIndex")
    if me is None:
        return None

    if wall_defer:
        try:
            opp_active = state["players"][1 - me]["active"]
            active = opp_active[0] if opp_active else None
        except (KeyError, IndexError, TypeError):
            active = None
        if active is not None and active.get("id") in WALL_POKEMON:
            return None  # counters are our only out here: let the net aim them

    scored: list[tuple[tuple, int]] = []
    for i, opt in enumerate(options):
        player = opt.get("playerIndex")
        if player is None or player == me:
            return None  # mixed or own-side select: leave it to the net
        pk = _pokemon_at(state, player, opt.get("area"), opt.get("index") or 0)
        if pk is None:
            return None  # something we cannot read: leave it to the net
        hp = pk.get("hp")
        if hp is None:
            return None
        kills = hp <= CHIP_DAMAGE
        # kills first, most prizes among those, then closest to dying
        scored.append(((0 if kills else 1,
                        -cards.prize_value(pk["id"]) if kills else 0,
                        hp), i))

    scored.sort()
    return [i for _, i in scored]


def _hand_card_id(state: dict, me: int, index: int) -> int:
    try:
        hand = state["players"][me]["hand"]
        card = hand[index]
        return card["id"] if card else 0
    except (KeyError, IndexError, TypeError):
        return 0


def _dark(pk: dict) -> int:
    return sum(1 for e in (pk.get("energies") or []) if e == DARK_TYPE)


def energy_spread(obs: dict, chosen: list[int]) -> list[int] | None:
    """Redirect a wasted second {D} on a Munkidori to a bare one.

    Verified in-engine over 40 games: Adrena-Brain is once **per Pokemon**
    (we activated it twice in a turn 35 times, and a slot that had used it was
    never re-offered), and its "has any {D} Energy attached" is a **threshold,
    not a cost** -- the energy is never consumed (n=138, unchanged every time).
    So two Munkidori holding one {D} each move 6 damage counters a turn; one
    Munkidori holding two moves 3, and the second {D} does nothing else either:
    Munkidori's only attack is Mind Bend, cost {P}{C}, and this deck runs zero
    Psychic energy. Nor is Munkidori a *Marnie's* Pokemon, so Grimmsnarl ex's
    Punk Up cannot attach to it -- the 1-per-turn hand attach is the only
    source, which is what makes spending it on a no-op expensive.

    Measured on the shipped clone (`opportunity_audit.py`, 150 games): when a
    select offered both a bare and an already-loaded Munkidori and the clone
    attached to one of them, it picked the loaded one **143 times to 94** --
    worse than a coin flip, because `optfeat` gives it no attached-energy count.

    Narrow on purpose: this only reorders *which Munkidori* gets the energy the
    net already decided to attach. It never creates or suppresses an attach.
    """
    sel = obs.get("select") or {}
    if sel.get("context") != MAIN:
        return None
    options = sel.get("option") or []
    if not chosen:
        return None
    pick = chosen[0]
    if not 0 <= pick < len(options):
        return None
    state = obs.get("current") or {}
    me = state.get("yourIndex")
    if me is None:
        return None

    bare: list[int] = []
    loaded: list[int] = []
    for i, opt in enumerate(options):
        if opt.get("type") != OPT_ATTACH:
            continue
        if (opt.get("area") or _HAND) != _HAND:
            continue
        if _hand_card_id(state, me, opt.get("index") or 0) != DARK_ENERGY:
            continue
        pk = _pokemon_at(state, me, opt.get("inPlayArea"),
                         opt.get("inPlayIndex") or 0)
        if pk is None or pk.get("id") != MUNKIDORI:
            continue
        (bare if _dark(pk) == 0 else loaded).append(i)

    if pick not in loaded or not bare:
        return None
    return [bare[0]] + [i for i in chosen[1:] if i != bare[0]]


def counter_source(obs: dict, chosen: list[int], rank) -> list[int] | None:
    """Take Adrena-Brain's counters off a Pokemon that HAS three of them.

    Adrena-Brain moves "up to 3 damage counters" from one of our Pokemon to one
    of theirs, and the source is its own select (REMOVE_DAMAGE_COUNTER, all
    options ours). How many it then moves is capped by what the source actually
    carries -- the follow-up REMOVE_DAMAGE_COUNTER_COUNT select offers "1,2,3"
    off a source with 3+ counters but only "1,2" off a source with 2. The clone
    already takes the maximum on that second select **100% of the time**
    (n=481, 120 games), so all the loss is here, one select earlier.

    Measured on the shipped clone (120 games, 291 source selects with >= 2
    options): in **59 of them (20.3%)** it picked a source that moves fewer
    counters than an available alternative -- 10 or 20 damage where 30 was on
    the table. With the rule that goes to 0, and activations that move the full
    3 counters rise from 67.1% to 76.5%.

    arena: `bc:s,src` vs `bc` = **0.534 [0.513, 0.556], n=2000**, mirror.

    This is the `energy_spread` shape, not the `boss_converts` shape: the
    heavily damaged source is better in BOTH directions at once -- it transfers
    more damage AND it heals the Pokemon that actually needed healing -- so
    there is no trade being made and no judgment to override. `optfeat` simply
    has no HP or damage per option, so the net cannot see which is which.

    Deliberately minimal, exactly like `energy_spread`: it never changes
    whether counters move, only which of our Pokemon they come off, and among
    the sources that can pay the full 3 it keeps the net's own preference.
    """
    sel = obs.get("select") or {}
    if sel.get("context") != REMOVE_DAMAGE_COUNTER:
        return None
    options = sel.get("option") or []
    if len(options) < 2 or not chosen:
        return None
    pick = chosen[0]
    if not 0 <= pick < len(options):
        return None
    state = obs.get("current") or {}
    me = state.get("yourIndex")
    if me is None:
        return None

    movable: list[int] = []
    for opt in options:
        if opt.get("playerIndex") != me:
            return None  # not the own-side source pick: leave it to the net
        pk = _pokemon_at(state, me, opt.get("area"), opt.get("index") or 0)
        if pk is None:
            return None
        hp, mx = pk.get("hp"), pk.get("maxHp")
        if hp is None or mx is None:
            return None
        movable.append(min(max(0, (mx - hp) // 10), MAX_MOVE))

    best = max(movable)
    if movable[pick] >= best:
        return None  # the net already picked a source that pays in full
    top = {i for i, m in enumerate(movable) if m == best}
    for i in rank():
        if i in top:
            return [i] + [j for j in chosen[1:] if j != i]
    return None


# --- Boss's Orders: drag something we can actually kill -------------------

def best_damage(active: dict, mypl: dict, oppl: dict, target: dict) -> float:
    """Best damage `active` can pay for right now, applied to `target`.

    `estimate` is text-pattern-based and approximate in general, but every
    attack this deck can pay for is flat damage (Shadow Bullet 180, Corkscrew
    Punch 60, Frost Smash 60), so here it is exact. Where it is not, it
    under-reads, which only makes the callers below more conservative."""
    if not active:
        return 0.0
    atk_type = cards.card(active["id"]).get("energyType")
    weak = cards.card(target["id"]).get("weakness")
    mult = 2.0 if (weak is not None and weak == atk_type) else 1.0
    best = 0.0
    for aid in cards.card(active["id"]).get("attacks") or []:
        a = cards.attacks().get(aid)
        if a and cards.energy_satisfied(a["energies"], active["energies"]):
            best = max(best, estimate(aid, active, mypl, oppl) * mult)
    return best


def drag_target(obs: dict, prefer_high_hp: bool = False) -> list[int] | None:
    """Ranked option indices for Boss's Orders' drag, else None.

    Boss's Orders resolves through a **SWITCH** select (not TO_ACTIVE, which is
    our own post-KO promotion) whose options are the opponent's benched
    Pokemon. Same blind spot as `chip_target`: no HP in the features, so the
    net cannot tell which of them dies to the attack we are about to make.

    Measured on the shipped clone (300 games): given a KO-able bench target it
    took the best available KO 85 times out of 99 -- 12 drags of a Pokemon that
    survives, 2 that took fewer prizes than were on offer.

    Rank: dies to our attack first, most prizes among those, then closest to
    dying. Same guard as `chip_target` -- every option must be an opponent's
    benched Pokemon, so our own retreats and switches are left to the net.

    `prefer_high_hp` (`bc:drag,dragHi`) flips the tiebreak **inside the KO-able
    group only**: they all die this turn, so "closest to dying" buys nothing
    there, and the user's argument is that the big one is a developing threat
    while a small basic is cheap for them to replace. The non-KO-able fallback
    keeps ascending HP -- there "closest to dying" is the whole point, and
    flipping it would be a different intervention."""
    sel = obs.get("select") or {}
    if sel.get("context") != SWITCH:
        return None
    options = sel.get("option") or []
    if len(options) < 2:
        return None
    state = obs.get("current") or {}
    me = state.get("yourIndex")
    if me is None:
        return None
    try:
        mypl, oppl = state["players"][me], state["players"][1 - me]
    except (KeyError, IndexError, TypeError):
        return None
    active = mypl["active"][0] if mypl.get("active") else None

    scored: list[tuple[tuple, int]] = []
    for i, opt in enumerate(options):
        if opt.get("playerIndex") in (None, me) or opt.get("area") != _BENCH:
            return None
        pk = _pokemon_at(state, 1 - me, _BENCH, opt.get("index") or 0)
        if pk is None or pk.get("hp") is None:
            return None
        kills = best_damage(active, mypl, oppl, pk) >= pk["hp"]
        hp = -pk["hp"] if (kills and prefer_high_hp) else pk["hp"]
        scored.append(((0 if kills else 1,
                        -cards.prize_value(pk["id"]) if kills else 0,
                        hp), i))

    scored.sort()
    return [i for _, i in scored]


def full_rank(net, obs: dict) -> list[int]:
    """The net's complete ranking, not the top-k that `choose` returns.

    A veto needs the runner-up, and every MAIN select measured here has
    maxCount == 1. Plain sort rather than argsort so this module stays
    numpy-free."""
    scores = net.scores(obs)
    return sorted(range(len(scores)), key=lambda i: -float(scores[i]))


def boss_veto(obs: dict, chosen: list[int], rank) -> list[int] | None:
    """Suppress Boss's Orders when their bench holds nothing we can KO.

    **This is the third Boss's Orders intervention, and the only untested one.**
    P4a measured *forcing* the play when it converts (0.493) and *aiming* the
    drag (0.489), both null. Neither touched the case here: the play happening
    at all when it buys nothing. Measured with `scripts/p5_audit.py --matches
    200`: **35 of 108 Boss's Orders plays (32.4%) had no KO-able target on the
    opponent's bench at all.** Those hand the opponent a free promotion --- the
    user watched one drag get evolved into their main attacker the next turn.

    Shape: this deletes an option rather than picking a side in a trade, but the
    option is only *conditionally* dominated (a drag can still strand their
    attacker in the Active, which `best_damage` cannot see), so it sits between
    the P4b class and the P4a class. Per rule 10 it lives or dies by its A/B.

    `rank` is a zero-argument callable returning the net's FULL ranking. It is
    needed because MAIN selects have maxCount == 1 --- `choose` hands back one
    index, so vetoing it leaves nothing to play instead. Called only when the
    veto actually fires, which is why it is lazy.
    """
    sel = obs.get("select") or {}
    if sel.get("context") != MAIN:
        return None
    options = sel.get("option") or []
    if not chosen:
        return None
    pick = chosen[0]
    if not 0 <= pick < len(options):
        return None
    state = obs.get("current") or {}
    me = state.get("yourIndex")
    if me is None:
        return None
    opt = options[pick]
    if opt.get("type") != OPT_PLAY:
        return None
    if _hand_card_id(state, me, opt.get("index") or 0) != BOSS_ORDERS:
        return None
    try:
        mypl, oppl = state["players"][me], state["players"][1 - me]
    except (KeyError, IndexError, TypeError):
        return None
    active = mypl["active"][0] if mypl.get("active") else None
    if not active:
        return None
    for pk in oppl.get("bench") or []:
        if pk and pk.get("hp") is not None \
                and best_damage(active, mypl, oppl, pk) >= pk["hp"]:
            return None  # the drag buys a prize: let it through

    # Nothing on their bench dies. Fall through to the net's next choice,
    # skipping every other Boss's Orders copy in hand for the same reason.
    vetoed = {i for i, o in enumerate(options)
              if o.get("type") == OPT_PLAY
              and _hand_card_id(state, me, o.get("index") or 0) == BOSS_ORDERS}
    rest = [i for i in rank() if i not in vetoed]
    return rest[:len(chosen)] or None


def boss_converts(obs: dict) -> list[int] | None:
    """Play Boss's Orders when the drag turns a nothing turn into a prize.

    The frequency question was already closed -- we play Boss's Orders on 32.4%
    of legal turns against the demonstrators' 31.4%. The open one was *when*.
    Over 300 games there were 157 turns where our attack would not KO the
    opponent's Active but would KO something on their bench, and the clone
    played Boss's Orders on only 58 of them (36.9%; it plays it on 25.7% of all
    other legal turns, so it does discriminate -- just barely).

    Fires only on that exact shape: we can attack, the Active survives, a
    benched Pokemon does not. It costs the turn's Supporter, which is why it
    stays pinned to the case where the payoff is a guaranteed prize."""
    sel = obs.get("select") or {}
    if sel.get("context") != MAIN:
        return None
    options = sel.get("option") or []
    state = obs.get("current") or {}
    me = state.get("yourIndex")
    if me is None:
        return None
    boss = [i for i, o in enumerate(options)
            if o.get("type") == OPT_PLAY
            and _hand_card_id(state, me, o.get("index") or 0) == BOSS_ORDERS]
    if not boss:
        return None
    try:
        mypl, oppl = state["players"][me], state["players"][1 - me]
    except (KeyError, IndexError, TypeError):
        return None
    active = mypl["active"][0] if mypl.get("active") else None
    opp_active = oppl["active"][0] if oppl.get("active") else None
    if not active or not opp_active:
        return None
    if best_damage(active, mypl, oppl, opp_active) >= opp_active["hp"]:
        return None  # the Active already dies; the drag would trade down
    for pk in oppl.get("bench") or []:
        if pk and best_damage(active, mypl, oppl, pk) >= pk["hp"]:
            return [boss[0]]
    return None


def _prize_if_ko(attacker, mypl, oppl, target) -> int:
    """Prizes we take by attacking `target`, or 0 if it survives."""
    if target is None or target.get("hp") is None:
        return 0
    if best_damage(attacker, mypl, oppl, target) < target["hp"]:
        return 0
    return cards.prize_value(target["id"])


def _snipe_prizes(oppl, exclude_index=None) -> int:
    """Best prize Shadow Bullet's 30 bench snipe takes, 0 if none dies to it."""
    best = 0
    for i, pk in enumerate(oppl.get("bench") or []):
        if pk is None or pk.get("hp") is None or i == exclude_index:
            continue
        if pk["hp"] <= CHIP_DAMAGE:
            best = max(best, cards.prize_value(pk["id"]))
    return best


def boss_prize_veto(obs: dict, chosen: list[int], rank) -> list[int] | None:
    """Don't play Boss's Orders when ATTACKING NOW takes strictly more prizes.

    **The fifth Boss's Orders intervention, and `EVIDENCE` §6 said not to write
    it. §6 is wrong, and here is the distinction it missed.** The four nulls all
    answered *which* Pokemon to drag (`drag_target`, `prefer_high_hp`) or
    *whether the drag itself converts* (`boss_converts`, `boss_veto`). **Not one
    of them compares the drag against the attack we already have.**

    The defect this targets was measured on 54 REAL ladder games of the shipped
    v3 agent (`scripts/p8_optv3_replays.py`): of 31 drags where attacking was a
    genuine alternative, **9 (29%) were misplays, and 5 of those threw away a
    DOUBLE KO** -- Shadow Bullet is 180 to the Active *plus 30 to a bench*, so a
    <=30 HP bench sitter means attacking takes two prizes. In every one of those
    five we dragged the very Pokemon we could have sniped for free, converting a
    2-prize turn into a 1-prize turn:

        eg 89011961 t11: could KO Crustle hp=80 (1p) + snipe Dwebble hp=10;
                         dragged Dwebble
        eg 89021174 t9:  could KO Alakazam hp=80 (1p) + snipe Abra hp=20;
                         dragged Abra

    **Why this is the DOMINATED column (rule 11's 3-for-3 side) and the other
    four were not:** both branches are pure arithmetic -- prize values and
    damage-vs-HP, no judgment about tempo or what they might evolve into. We are
    not choosing between two goods; we are deleting a strictly worse option.

    ⚠ The comparison is honest on both sides: a drag can double-KO too (drag a
    KO-able target, snipe a different <=30 HP bench sitter), so `drag_best`
    excludes the dragged Pokemon from its own snipe. The veto fires only on
    **strictly** greater, so ties go to the net.
    """
    sel = obs.get("select") or {}
    if sel.get("context") != MAIN or not chosen:
        return None
    options = sel.get("option") or []
    pick = chosen[0]
    if not 0 <= pick < len(options):
        return None
    opt = options[pick]
    if opt.get("type") != OPT_PLAY:
        return None
    state = obs.get("current") or {}
    me = state.get("yourIndex")
    if me is None:
        return None
    if _hand_card_id(state, me, opt.get("index") or 0) != BOSS_ORDERS:
        return None
    try:
        mypl, oppl = state["players"][me], state["players"][1 - me]
    except (KeyError, IndexError, TypeError):
        return None
    active = mypl["active"][0] if mypl.get("active") else None
    opp_active = oppl["active"][0] if oppl.get("active") else None
    if not active or not opp_active:
        return None

    # What attacking RIGHT NOW is worth: the Active if it dies, plus the snipe.
    attack_now = _prize_if_ko(active, mypl, oppl, opp_active) + _snipe_prizes(oppl)

    # What the best drag is worth: that target if it dies, plus a snipe onto a
    # DIFFERENT bench sitter (the dragged one is no longer benched).
    drag_best = 0
    for i, pk in enumerate(oppl.get("bench") or []):
        if pk is None:
            continue
        got = _prize_if_ko(active, mypl, oppl, pk)
        if got:
            drag_best = max(drag_best, got + _snipe_prizes(oppl, exclude_index=i))

    if attack_now <= drag_best:
        return None  # dragging is at least as good -- leave it to the net

    vetoed = {i for i, o in enumerate(options)
              if o.get("type") == OPT_PLAY
              and _hand_card_id(state, me, o.get("index") or 0) == BOSS_ORDERS}
    rest = [i for i in rank() if i not in vetoed]
    return rest[:len(chosen)] or None


def poffin_force(obs: dict, chosen: list[int]) -> list[int] | None:
    """Play Buddy-Buddy Poffin when the bench has room — E11.

    **The first candidate this project found where WE are the worse player at
    something ordering-free.** `p70_perturn_sweep.py` ranks every option class
    by its per-TURN gap (rule 21) instead of its per-decision gap, and this was
    invisible to the per-decision ranking because the clone is never
    *confidently wrong* here — it simply never gets round to it. Share of
    available turns in which the card is actually played, conditioned on our own
    board occupancy, mirror only (`EVIDENCE` §8bl, `docs/experiments/E11-poffin.md`):

        board  4:  1150+ pilots 70.2%,  our clone 29.4%
        board  5:  1150+ pilots 46.9%,  our clone  7.2%

    Worth **0.80 plays/game**, over the 0.5 sizing gate, and the confound is
    checked: both sides decline at the same mean board size (4.46 vs 4.45), so
    it is the behaviour that differs, not the mix of situations.

    Shape: benching a 70 HP basic is a **tradeoff** (development against giving
    the mirror's own Shadow Bullet snipe another target), so rule 11 would
    normally forbid building it. The governing precedent is `boss_veto`'s —
    rule 10, "it lives or dies by its A/B" — and the A/B is byte-identical-net
    with the rule toggled, so the ±13 Elo seed nuisance cancels exactly.

    ⚠ **Deliberately conservative at board 5.** The experts are themselves a
    coin flip there (46.9%), so forcing that bucket would overshoot the
    behaviour being copied. Fires only with **>= 2 free slots**. Widening it is
    a separate experiment, not a knob to turn after reading the result.
    """
    sel = obs.get("select") or {}
    if sel.get("context") != MAIN:
        return None
    options = sel.get("option") or []
    if not chosen or not 0 <= chosen[0] < len(options):
        return None
    state = obs.get("current") or {}
    try:
        me = state["yourIndex"]
        mypl = state["players"][me]
    except (KeyError, IndexError, TypeError):
        return None

    # Already playing it: nothing to force.
    if _hand_card_id(state, me, options[chosen[0]].get("index") or 0) == POFFIN:
        return None

    # Board occupancy: the Active plus every filled bench slot, out of 6.
    bench = mypl.get("bench") or []
    filled = (1 if (mypl.get("active") and mypl["active"][0]) else 0)
    filled += sum(1 for pk in bench if pk)
    if filled > 4:            # fewer than 2 free slots -- see the docstring
        return None

    for i, o in enumerate(options):
        if (o.get("type") == OPT_PLAY
                and _hand_card_id(state, me, o.get("index") or 0) == POFFIN):
            return [i]
    return None


# --- E21: the Petrel fetch ---------------------------------------------------

def petrel_fetch(obs: dict, stadium: bool = False,
                 scrapper: bool = False) -> list[int] | None:
    """Inject board facts into Petrel's fetch — the one select with NO board.

    **Why this select and not another.** §8br's structural addendum: a fetch
    option's feature vector contains *nothing* about the board. The whole v3
    target block (`hp`, damage taken, dies-to-30, energy count, best_damage) is
    identically zero, because it resolves a Pokemon at (player, area, index) and
    the deck is not an in-play area. **One fetch option differs from another
    only by its card embedding and card type.** Everything situational has to
    arrive through `srepr`, which is concatenated identically to every option
    and can discriminate only through the head MLP's interaction term.

    ⚠ **These rules are NOT derived from what the experts fetch**, and that is
    deliberate: §8u measured that agreement with the FIELD predicts strength
    while agreement with the EXPERT anti-predicts it, and E11 copied a sized,
    ordering-free expert gap (0.80 plays/game) and measured **0.487**. Both
    conditions below come from card text plus the board:

    * `stadium` — Spikemuth Gym is the only Stadium in the 60 and our entire
      evolution line is Marnie's, so it is a repeatable engine tutor for us and
      much weaker for most opponents. Fetch it when **no Stadium is in play, or
      the one in play is theirs** — i.e. when playing it both starts our engine
      and removes theirs. Sized at **0.461 firings/game** over our 76 ladder
      games, the largest rate anything in this seam has had.
    * `scrapper` — Tool Scrapper does *nothing* unless a Tool is attached. So
      fetch it only when a Tool is on THEIR board. Sized at 0.171/game.
      ⚠ Under the 0.5 gate; included because an n=2,000 mirror A/B costs
      ~6 min here, so the gate's original cost model (arena time is scarce)
      does not bind. Rule 14 gates what is worth BUILDING, not what is
      affordable to measure.

    ⛔ Gated on `select.effect` being Petrel itself, so Poke Pad's Supporter
    search and Night Stretcher's discard recovery are untouched — they cannot
    reach a Stadium anyway, and a rule that fires on selects it was not sized
    on is measuring something other than what was sized.

    Returns a single-element order (maxCount is 1 here and minCount is 0, so
    declining is legal and one index is a complete answer), or None to leave the
    select entirely to the net.
    """
    sel = obs.get("select") or {}
    if (sel.get("context") or 0) != FETCH:
        return None
    eff = sel.get("effect")
    if not isinstance(eff, dict) or eff.get("id") != PETREL:
        return None
    options = sel.get("option") or []
    if len(options) < 2:
        return None
    state = obs.get("current") or {}
    try:
        me = state["yourIndex"]
        opp = state["players"][1 - me]
    except (KeyError, IndexError, TypeError):
        return None

    want = None
    if scrapper:
        tools = 0
        for where in ("active", "bench"):
            for pk in (opp.get(where) or []):
                if pk:
                    tools += len((pk or {}).get("tools") or [])
        if tools:
            want = TOOL_SCRAPPER
    if want is None and stadium:
        stad = state.get("stadium") or []
        if (not stad) or (stad[0] or {}).get("playerIndex") != me:
            want = SPIKEMUTH
    if want is None:
        return None

    # The card an option names, via the NET'S OWN extractor -- p76's lesson:
    # a PLAY option carries no `area`, and a hand-rolled mapping produced a
    # table that E11's own measurement contradicted outright.
    from .optfeat import option_features
    for i, o in enumerate(options):
        try:
            if int(option_features(obs, o)[1] or 0) == want:
                return [i]
        except Exception:  # noqa: BLE001
            continue
    return None


In [ ]:
%%writefile agents/sa/features.py
"""State featurization shared by the value-net trainer and the agent.

`featurize(state, me)` -> (dense float32 vector, id-bag int32 arrays).
The id bags are embedded by the net (sum of embedding rows), so the feature
layout here and the net architecture must move together. Bump VERSION when
changing either.
"""
from __future__ import annotations

import numpy as np

from . import cards as cdb
from .textdmg import best_estimated_damage

VERSION = 3
N_CARD_IDS = 1300          # card id space (ids are 1..1267 today)
N_SLOTS = 12               # my active, my bench x5, opp active, opp bench x5
PER_SLOT = 18
N_GLOBAL = 26
DENSE_DIM = N_GLOBAL + N_SLOTS * PER_SLOT

# --- the v4 block (day 12), APPENDED at the very end of the state vector ----
# `scripts/p18_missing_state_audit.py` enumerated every field of the
# observation that `featurize()` never reads and measured how much each varies
# at a real decision point. It killed three candidates that had been on the
# plan for two days -- opponent hand size, prizes remaining and turn number are
# ALL already encoded above (lines "put(...)" below) -- and two it found
# itself: `remainDamageCounter` is 0 at 100% of decisions and `remainEnergyCost`
# at 99.1%, so neither can explain a single miss.
#
# What survived, with the miss mass it targets (v3 net, 12,939 held-out rows):
#   * turnActionCount -- 20 distinct values, modal share 17% in MAIN.
#     MAIN is 2,629 of 3,902 misses. The net re-scores a barely-changed board
#     several times per turn with no idea how deep into the turn it is.
#   * the select's EFFECT card -- which card caused this select. Modal share
#     26% in TO_HAND (674 misses): the same context means "tutor a Trainer"
#     (Petrel), "take a Supporter" (Poke Pad), "recover from discard" (Night
#     Stretcher) or "search anything" (Ultra Ball), and the net scores each
#     option INDEPENDENTLY, so it never sees the option set that would reveal
#     which. This is the one input that tells it what kind of choice it is.
#   * the stadium -- 7 distinct, 61% Spikemuth Gym, and Area Zero Underdepths
#     changes the bench size. Absent entirely, including from every id bag.
#   * `retreated` / `stadiumPlayed` -- the two missing members of the
#     once-per-turn quartet whose other two (`supporterPlayed`,
#     `energyAttached`) have been encoded since v1. `retreated` is 43%
#     non-modal in SWITCH.
#
# ⚠ APPENDED, NEVER INSERTED -- and the append lands after `seld`, which is the
# LAST block of the state vector (see policynet.scores / train_policy.forward).
# A v3 net simply slices to its own `state_in` and reads byte-identical input,
# which is what lets v3 and v4 run head-to-head in one process (rule 4).
N_EXTRA = 8                # dense scalars
N_XSLOT = 2                # card ids embedded through the existing slot table:
#                            (stadium in play, the select's effect card)

# --- ablating the v4 block (day 13) -----------------------------------------
# The block shipped whole, so nothing said WHICH member bought the 37 Elo. Each
# name below is a drop-one arm; the trainer zeroes those columns of the corpus
# and records the surviving mask in the npz, so `policynet` reproduces exactly
# the input the net was trained on WITHOUT rebuilding a corpus or changing a
# single layer width. Same arch, same init, same rows -- only the content of a
# few columns differs, which is a tighter control than removing the dimensions.
# Indices are (xdense col, ...) then (N_EXTRA + xslot col, ...).
X_GROUPS: dict[str, tuple[int, ...]] = {
    "turnAction": (0,),
    "retreated": (1,),
    "stadiumPlayed": (2,),
    "stadium": (3, N_EXTRA + 0),      # the in-play flag AND the card id
    "benchMax": (4,),
    "tools": (5, 6),
    "poolSize": (7,),
    "effect": (N_EXTRA + 1,),         # which card caused this select
}


def extra_feats(state: dict, sel: dict,
                me: int) -> tuple[np.ndarray, np.ndarray]:
    """The v4 block: (dense scalars, card ids to embed). See the note above."""
    x = np.zeros(N_EXTRA, dtype=np.float32)
    mypl, oppl = state["players"][me], state["players"][1 - me]
    stad = state.get("stadium") or []

    def tools(pl) -> int:
        n = 0
        for pk in ([pl["active"][0] if pl["active"] else None]
                   + list(pl["bench"])):
            if pk:
                n += len(pk.get("tools") or [])
        return n

    x[0] = min(int(state.get("turnActionCount") or 0), 24) / 24.0
    x[1] = 1.0 if state.get("retreated") else 0.0
    x[2] = 1.0 if state.get("stadiumPlayed") else 0.0
    x[3] = 1.0 if stad else 0.0
    x[4] = (mypl.get("benchMax") or 5) / 8.0
    x[5] = min(tools(mypl), 4) / 4.0
    x[6] = min(tools(oppl), 4) / 4.0
    x[7] = min(len(sel.get("deck") or []), 60) / 60.0

    eff = sel.get("effect")
    ids = np.zeros(N_XSLOT, dtype=np.int32)
    ids[0] = stad[0]["id"] if stad else 0
    ids[1] = (eff or {}).get("id", 0) if isinstance(eff, dict) else 0
    ids[ids >= N_CARD_IDS] = 0
    return x, ids

# --- the v6 card-attribute block (day 20), APPENDED after the v4 block -------
# E6 (docs/experiments/embeddings/E6-identity-channel.md) priced the identity
# channel by permuting embedding rows on the frozen v5 net: scrambling only the
# OPPONENT's card ids costs 0.838 -> 0.587 against rule:crustle, whose four
# Pokemon are all in vocabulary, and 0.625 -> 0.607 against rule:v10, whose six
# are all OUT of it. We do not read Mega Lucario badly; we cannot read it at
# all, because `slot_emb` has no trained row for any of its Pokemon.
#
# A per-card embedding row can only ever describe cards the corpus contained.
# These attributes come from the card DB, which covers all 1,267 cards, so an
# unseen Pokemon arrives as "Fighting, weak to Psychic, has an ability" instead
# of an untrained N(0, 1) vector. That is the only channel here that transfers.
#
# Sized BEFORE building (rule 14, scripts/p55_attr_sizing.py, at the decision):
#   energyType   10 distinct  modal 0.438  H/Hmax 0.720   at the opp active
#   weakness      9 distinct  modal 0.443  H/Hmax 0.732
#   hasAbility    2 distinct  modal 0.717  H/Hmax 0.860
#   weak-to-our-active's-type fires on 12.1% of decisions
# and the gate KILLED two candidates before they cost anything: `aceSpec` is a
# single value across the whole corpus, and `pokemonType`/`evolutionType` are
# fully redundant -- the six flags at slot +4..+9 give 12 distinct signatures
# and none maps to more than one value of either. Shipping those would have
# repeated EVIDENCE 8ab, where five leftover columns measured -22 Elo against
# having no block at all.
#
# `resistance` is marginal (3 distinct, modal 0.835) so it gets one flag, not
# a one-hot.
N_ATTR_ETYPE = 11          # energyType, 0..10
N_ATTR_WEAK = 9            # weakness: index 0 = none, else the type 1..8
PER_SLOT_ATTR = N_ATTR_ETYPE + N_ATTR_WEAK + 3   # + ability, resist, weakHit
N_ATTR = N_SLOTS * PER_SLOT_ATTR

# Same drop-one machinery as X_GROUPS, over columns of the attr vector. The
# block ships whole, so without these nothing would say WHICH member paid.
_A_ETYPE, _A_WEAK = 0, N_ATTR_ETYPE
_A_ABILITY = _A_WEAK + N_ATTR_WEAK
_A_RESIST, _A_WEAKHIT = _A_ABILITY + 1, _A_ABILITY + 2


def _a_group(lo: int, hi: int) -> tuple[int, ...]:
    return tuple(s * PER_SLOT_ATTR + i
                 for s in range(N_SLOTS) for i in range(lo, hi))


A_GROUPS: dict[str, tuple[int, ...]] = {
    "attrEnergyType": _a_group(_A_ETYPE, _A_ETYPE + N_ATTR_ETYPE),
    "attrWeakness": _a_group(_A_WEAK, _A_WEAK + N_ATTR_WEAK),
    "attrAbility": _a_group(_A_ABILITY, _A_ABILITY + 1),
    "attrResist": _a_group(_A_RESIST, _A_RESIST + 1),
    "attrWeakHit": _a_group(_A_WEAKHIT, _A_WEAKHIT + 1),
}


def attr_feats(state: dict, me: int) -> np.ndarray:
    """The v6 block: card attributes for all 12 slots, same slot order as
    `featurize` (my active, my bench x5, opp active, opp bench x5)."""
    a = np.zeros(N_ATTR, dtype=np.float32)
    mypl, oppl = state["players"][me], state["players"][1 - me]

    def active_type(pl) -> int | None:
        act = pl["active"][0] if pl["active"] else None
        return cdb.card(act["id"]).get("energyType") if act else None

    # the type that would be ATTACKING each side's slots
    facing = (active_type(oppl), active_type(mypl))

    slots = ([mypl["active"][0] if mypl["active"] else None]
             + [(mypl["bench"][i] if i < len(mypl["bench"]) else None)
                for i in range(5)]
             + [oppl["active"][0] if oppl["active"] else None]
             + [(oppl["bench"][i] if i < len(oppl["bench"]) else None)
                for i in range(5)])

    for si, pk in enumerate(slots):
        if pk is None:
            continue
        c = cdb.card(pk["id"])
        base = si * PER_SLOT_ATTR

        et = c.get("energyType") or 0
        if 0 <= et < N_ATTR_ETYPE:
            a[base + _A_ETYPE + et] = 1.0

        weak = c.get("weakness") or 0
        if 0 <= weak < N_ATTR_WEAK:
            a[base + _A_WEAK + weak] = 1.0

        a[base + _A_ABILITY] = 1.0 if c.get("skills") else 0.0
        a[base + _A_RESIST] = 1.0 if c.get("resistance") else 0.0
        # does whatever is facing this slot hit it for weakness?
        att = facing[0 if si < 6 else 1]
        a[base + _A_WEAKHIT] = 1.0 if (weak and att == weak) else 0.0

    return a


# id bags: per-slot card id (12), my hand, my discard, opp discard, opp known
BAG_NAMES = ("slots", "my_hand", "my_discard", "opp_discard")


def _slot_feats(pk: dict | None, mypl: dict, oppl: dict, out: np.ndarray,
                base: int) -> int:
    """Write PER_SLOT features for one pokemon slot; return its card id."""
    if pk is None:
        return 0
    c = cdb.card(pk["id"])
    hp = pk["hp"]
    max_hp = pk["maxHp"] or 1
    out[base + 0] = 1.0
    out[base + 1] = hp / 300.0
    out[base + 2] = max_hp / 300.0
    out[base + 3] = 1.0 - hp / max_hp
    out[base + 4] = 1.0 if c.get("basic") else 0.0
    out[base + 5] = 1.0 if c.get("stage1") else 0.0
    out[base + 6] = 1.0 if c.get("stage2") else 0.0
    out[base + 7] = 1.0 if c.get("ex") else 0.0
    out[base + 8] = 1.0 if c.get("megaEx") else 0.0
    out[base + 9] = 1.0 if c.get("tera") else 0.0
    out[base + 10] = min(len(pk["energies"]), 6) / 6.0
    out[base + 11] = (c.get("retreatCost") or 0) / 4.0
    out[base + 12] = 1.0 if pk["appearThisTurn"] else 0.0
    out[base + 13] = min(best_estimated_damage(pk, mypl, oppl), 400) / 400.0
    out[base + 14] = len(pk["tools"]) / 1.0 if pk["tools"] else 0.0
    out[base + 15] = cdb.prize_value(pk["id"]) / 3.0
    out[base + 16] = _cost_satisfaction(pk)
    own = c.get("energyType")
    out[base + 17] = min(sum(1 for e in pk["energies"]
                             if e == own or e >= 10), 4) / 4.0
    return pk["id"]


def _cost_satisfaction(pk: dict) -> float:
    """How close the attached energy comes to paying this Pokemon's cheapest
    attack (1.0 = can attack now)."""
    best = 0.0
    have = pk["energies"]
    for aid in cdb.card(pk["id"]).get("attacks") or []:
        a = cdb.attacks().get(aid)
        if not a:
            continue
        cost = a["energies"]
        if not cost:
            return 1.0
        if cdb.energy_satisfied(cost, have):
            return 1.0
        best = max(best, min(len(have), len(cost)) / len(cost))
    return best


def featurize(state: dict, me: int) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    opp = 1 - me
    mypl = state["players"][me]
    oppl = state["players"][opp]

    dense = np.zeros(DENSE_DIM, dtype=np.float32)
    g = 0

    def put(v):
        nonlocal g
        dense[g] = v
        g += 1

    put(min(state["turn"], 40) / 40.0)
    put(1.0 if state["firstPlayer"] == me else 0.0)
    put((6 - len(mypl["prize"])) / 6.0)      # my prizes taken
    put((6 - len(oppl["prize"])) / 6.0)
    put(len(mypl["prize"]) / 6.0)
    put(len(oppl["prize"]) / 6.0)
    put(min(mypl["deckCount"], 60) / 60.0)
    put(min(oppl["deckCount"], 60) / 60.0)
    put(1.0 if mypl["deckCount"] == 0 else 0.0)
    put(1.0 if oppl["deckCount"] == 0 else 0.0)
    put(min(mypl["handCount"], 12) / 12.0)
    put(min(oppl["handCount"], 12) / 12.0)
    put(1.0 if state["supporterPlayed"] else 0.0)
    put(1.0 if state["energyAttached"] else 0.0)
    for pl in (mypl, oppl):
        put(1.0 if pl["poisoned"] else 0.0)
        put(1.0 if pl["burned"] else 0.0)
        put(1.0 if pl["asleep"] else 0.0)
        put(1.0 if pl["paralyzed"] else 0.0)
        put(1.0 if pl["confused"] else 0.0)
    put(sum(1 for pk in mypl["bench"] if pk is not None) / 5.0)
    put(sum(1 for pk in oppl["bench"] if pk is not None) / 5.0)
    assert g == N_GLOBAL, g

    slot_ids = np.zeros(N_SLOTS, dtype=np.int32)
    slots = ([mypl["active"][0] if mypl["active"] else None]
             + [(mypl["bench"][i] if i < len(mypl["bench"]) else None)
                for i in range(5)]
             + [oppl["active"][0] if oppl["active"] else None]
             + [(oppl["bench"][i] if i < len(oppl["bench"]) else None)
                for i in range(5)])
    for si, pk in enumerate(slots):
        side_my, side_opp = (mypl, oppl) if si < 6 else (oppl, mypl)
        cid = _slot_feats(pk, side_my, side_opp,
                          dense, N_GLOBAL + si * PER_SLOT)
        slot_ids[si] = cid if cid < N_CARD_IDS else 0

    def bag(cards_list) -> np.ndarray:
        return np.asarray([c["id"] for c in cards_list
                           if c is not None and c["id"] < N_CARD_IDS],
                          dtype=np.int32)

    bags = {
        "slots": slot_ids,
        "my_hand": bag(mypl["hand"] or []),
        "my_discard": bag(mypl["discard"]),
        "opp_discard": bag(oppl["discard"]),
    }
    return dense, bags


In [ ]:
%%writefile agents/sa/optfeat.py
"""Per-option features for the policy net.

Resolves what card/attack an option refers to and produces:
  * dense per-option vector (type one-hot + misc scalars + TARGET STATE)
  * card id (for embedding), attack id (for embedding)
Layout must stay in sync with policy trainer/inference. Bump VERSION on change.

## The v3 block, and the diagnosis behind it (ROADMAP B1, 2026-07-30)

`features.py` has ALWAYS given the net per-slot HP, damage fraction, attached
energy, prize value and best-estimated damage, for all 12 slots. So the standing
description of the blind spot -- "the net cannot see HP" -- was **wrong**, or at
least imprecise in the way that matters. The net can see the whole board's HP.

What it could not see is **which option points at which slot**. The v2 per-option
vector was a type one-hot plus eight scalars, of which the only positional
information was *area* flags (`active` / `bench` / `hand`) -- **`opt["index"]` and
`opt["inPlayIndex"]` were never encoded at all.** Consequences, and they explain
every rule in `targeting.py`:

  * Two options pointing at two different Pokemon on the bench got **identical**
    dense vectors, distinguishable only by the card-id embedding.
  * So two copies of the same card -- two Munkidori, two Dwebble -- were
    **exactly indistinguishable**. That is `energy_spread` (bare vs loaded
    Munkidori, measured 143-to-94 *against* the right answer, i.e. worse than a
    coin flip) and `chip_target` (which of their benched Pokemon dies to 30).

**So the gap was never "no HP features"; it was no BINDING between an option and
its target's state.** The rules work by re-deriving that binding by hand. This
block gives it to the net directly, which is the actual B1 experiment.

⚠ **Appended, never inserted.** Indices 0..24 are byte-identical to v2, so a v2
net still reads exactly what it was trained on. `policynet.Net` derives its own
option width from `head_in` and slices; that is what lets the shipped net and a
candidate net run **in the same process** for a head-to-head A/B (HANDOFF rule 4)
across a feature-layout change. **Never insert into the middle of this vector.**
"""
from __future__ import annotations

import numpy as np

VERSION = 3

N_OPTION_TYPES = 17
OPT_DENSE_V2 = N_OPTION_TYPES + 8      # 25 -- the shipped `policy_lw2` layout
N_TARGET_FEATS = 12                    # the v3 block, appended
OPT_DENSE_V3 = OPT_DENSE_V2 + N_TARGET_FEATS   # 37 -- the shipped v5 layout

# --- the v6 card-attribute block (day 20), appended after v3 ----------------
# `cardType` is the strongest thing the rule-14 gate found anywhere (7 distinct,
# modal 0.416, H/Hmax 0.780 at `opt_card`) and it is absent from BOTH vectors
# today. That is a textbook binding failure of the same family as B1: the state
# vector has carried `supporterPlayed` since v1, but every Trainer -- Item,
# Tool, Supporter, Stadium -- shares option type 7, so the net cannot tell
# which options that flag forbids. It has to infer "is this a Supporter" from
# a card-id embedding row, which is exactly the channel that fails on cards the
# corpus never contained.
N_CARD_TYPES = 7                       # 0 Pokemon .. 6 Special Energy
N_OPT_ATTR = N_CARD_TYPES + 2          # + target has ability, target weak to us
OPT_DENSE = OPT_DENSE_V3 + N_OPT_ATTR
# Widths a net may legitimately have been trained at. The dim guard accepts these
# and nothing else -- an unknown width is a stale net, not a new one.
KNOWN_OPT_DENSE = (OPT_DENSE_V2, OPT_DENSE_V3, OPT_DENSE)
N_ATTACK_IDS = 1600  # option_features returns (dense, card_id, attack_id, target_id)

# --- the v5 pooled option-set block (day 13) --------------------------------
# Every option is scored INDEPENDENTLY against one shared state vector, so the
# net has never been able to see the option SET -- it cannot tell whether it is
# choosing among 3 Trainers in hand or 40 cards in the deck, nor how the option
# in front of it compares to its alternatives. That is the same class of defect
# as B1 (no binding between an option and its target) and §8y (no `effect` card
# saying what kind of choice this is), and it is the last one of the class that
# is cheap: a deep-sets encoder in its minimal form.
#
# phi = the per-option encoding the head already builds
#       [opt_dense[:opt_cols], card_emb, atk_emb, tgt_emb]
# pool = elementwise mean and max over the select's options, plus two count
#        scalars (the count alone answers "3 Trainers or 40 deck cards")
# rho  = the existing state MLP, which now takes the pool as input
#
# ⚠ APPENDED to the STATE vector, after the v4 block, never inserted. A v3/v4
# net slices to its own `state_in` and reads byte-identical input, which is what
# keeps two feature generations runnable in one process (HANDOFF rule 4).
N_POOL_SCALARS = 2


def pool_width(opt_cols: int, emb: int) -> int:
    """Width of the v5 pooled block for a net trained at `opt_cols` and `emb`."""
    return 2 * (opt_cols + 3 * emb) + N_POOL_SCALARS


def pool_scalars(n: int) -> np.ndarray:
    """The two option-count scalars. Linear saturates at 40 (deck searches run
    to 60); the log keeps 2-vs-4 legible, which is where most selects live."""
    v = np.zeros(N_POOL_SCALARS, dtype=np.float32)
    v[0] = min(n, 40) / 40.0
    v[1] = float(np.log1p(n) / np.log(41.0))
    return v

# AreaType ints
_DECK, _HAND, _DISCARD, _ACTIVE, _BENCH, _PRIZE, _STADIUM = 1, 2, 3, 4, 5, 6, 7
_LOOKING = 12

CHIP_DAMAGE = 30      # Shadow Bullet's snipe and Adrena-Brain's 3 counters

# Imported lazily and cached: `targeting` pulls in `cards` + `textdmg`, and this
# module is imported by the dataset builder as well as the agent. No cycle exists
# today (targeting does not import optfeat) -- the laziness is to keep the import
# graph one-directional if that ever changes.
_CDB = None
_BEST_DAMAGE = None


def _cdb():
    global _CDB
    if _CDB is None:
        from . import cards
        _CDB = cards
    return _CDB


def _best_damage():
    """`targeting.best_damage` -- weakness- and payability-aware expected damage.
    Exact for this deck (every payable attack is flat damage); under-reads
    elsewhere, which only makes the feature conservative."""
    global _BEST_DAMAGE
    if _BEST_DAMAGE is None:
        from .targeting import best_damage
        _BEST_DAMAGE = best_damage
    return _BEST_DAMAGE


def _card_at(state: dict, sel: dict, player: int, area: int,
             index: int) -> int:
    """Best-effort card id at (player, area, index); 0 if unknown."""
    try:
        if area == _LOOKING:
            look = state.get("looking")
            if look and index < len(look) and look[index]:
                return look[index]["id"]
            return 0
        if area == _DECK:
            deck = sel.get("deck")
            if deck and index < len(deck) and deck[index]:
                return deck[index]["id"]
            return 0
        pl = state["players"][player]
        if area == _HAND:
            hand = pl.get("hand")
            if hand and index < len(hand) and hand[index]:
                return hand[index]["id"]
            return 0
        if area == _DISCARD:
            if index < len(pl["discard"]):
                return pl["discard"][index]["id"]
            return 0
        if area == _ACTIVE:
            act = pl["active"]
            if act and act[0] is not None:
                return act[0]["id"]
            return 0
        if area == _BENCH:
            if index < len(pl["bench"]) and pl["bench"][index] is not None:
                return pl["bench"][index]["id"]
            return 0
        if area == _PRIZE:
            pr = pl["prize"]
            if index < len(pr) and pr[index] is not None:
                return pr[index]["id"]
            return 0
        if area == _STADIUM:
            st = state.get("stadium") or []
            if st:
                return st[0]["id"]
            return 0
    except (KeyError, IndexError, TypeError):
        return 0
    return 0


def _pokemon_at(state: dict, player: int, area: int, index: int) -> dict | None:
    """The Pokemon dict an option points at, or None. Unlike `_card_at` this
    returns the live object, because the v3 block needs its HP and energy."""
    try:
        pl = state["players"][player]
        if area == _ACTIVE:
            act = pl["active"]
            return act[0] if act and act[0] is not None else None
        if area == _BENCH:
            bench = pl["bench"]
            if 0 <= index < len(bench):
                return bench[index]
    except (KeyError, IndexError, TypeError):
        return None
    return None


def _target_pokemon(state: dict, opt: dict, t: int, me: int,
                    player: int) -> tuple[dict | None, int]:
    """Resolve (the Pokemon this option acts on, its owner).

    Three shapes, matching how the engine words each option type:
      * ATTACH / EVOLVE (8, 9) -> the in-play Pokemon at inPlayArea/inPlayIndex,
        always ours.
      * ATTACK (13)            -> their Active, the thing we are about to hit.
      * everything else        -> (playerIndex, area, index), which is how every
        CARD/ABILITY/damage-counter select names a Pokemon.
    """
    if t in (8, 9):
        return _pokemon_at(state, me, opt.get("inPlayArea") or 0,
                           opt.get("inPlayIndex") or 0), me
    if t == 13:
        return _pokemon_at(state, 1 - me, _ACTIVE, 0), 1 - me
    return _pokemon_at(state, player, opt.get("area") or 0,
                       opt.get("index") or 0), player


def option_features(obs: dict, opt: dict) -> tuple[np.ndarray, int, int, int]:
    """-> (dense vector, card_id, attack_id, target_id) for one option.
    target_id = the in-play Pokemon an ATTACH/EVOLVE points at."""
    state = obs["current"]
    sel = obs["select"]
    me = state["yourIndex"]

    dense = np.zeros(OPT_DENSE, dtype=np.float32)
    t = opt.get("type") or 0
    if t < N_OPTION_TYPES:
        dense[t] = 1.0
    x = N_OPTION_TYPES

    card_id = 0
    attack_id = 0
    target_id = 0

    area = opt.get("area")
    index = opt.get("index") or 0
    player = opt.get("playerIndex")
    player = me if player is None else player

    if t == 7:  # PLAY: index into my hand
        card_id = _card_at(state, sel, me, _HAND, opt.get("index") or 0)
    elif t in (8, 9):  # ATTACH / EVOLVE: card at (area,index) onto target
        card_id = _card_at(state, sel, me, area or _HAND, index)
        target_id = _card_at(state, sel, me, opt.get("inPlayArea") or 0,
                             opt.get("inPlayIndex") or 0)
    elif t in (3, 10, 11):  # CARD / ABILITY / DISCARD
        card_id = _card_at(state, sel, player, area or 0, index)
    elif t == 13:  # ATTACK
        attack_id = opt.get("attackId") or 0
        act = state["players"][me]["active"]
        if act and act[0] is not None:
            card_id = act[0]["id"]
        opp_act = state["players"][1 - me]["active"]
        if opp_act and opp_act[0] is not None:
            target_id = opp_act[0]["id"]
    elif t == 15:  # SKILL
        card_id = opt.get("cardId") or 0
    elif t in (4, 5, 6):  # TOOL_CARD / ENERGY_CARD / ENERGY on a pokemon
        card_id = _card_at(state, sel, player, area or 0, index)

    dense[x + 0] = 1.0 if player == me else 0.0
    dense[x + 1] = (opt.get("number") or 0) / 10.0
    dense[x + 2] = 1.0 if area == _ACTIVE else 0.0
    dense[x + 3] = 1.0 if area == _BENCH else 0.0
    dense[x + 4] = 1.0 if area == _HAND else 0.0
    dense[x + 5] = (opt.get("energyIndex") or 0) / 5.0
    ipa = opt.get("inPlayArea")
    dense[x + 6] = 1.0 if ipa == _ACTIVE else 0.0
    dense[x + 7] = 1.0 if ipa == _BENCH else 0.0

    # --- v3: the option's TARGET state (indices 25..36) -------------------
    # Everything above this line is v2 and must not move.
    v = OPT_DENSE_V2
    pk, owner = _target_pokemon(state, opt, t, me, player)
    if pk is not None:
        hp = pk.get("hp")
        mx = pk.get("maxHp") or 1
        energies = pk.get("energies") or []
        if hp is not None:
            dense[v + 0] = 1.0                          # a target was resolved
            dense[v + 1] = hp / 300.0
            dense[v + 2] = mx / 300.0
            dense[v + 3] = 1.0 - hp / mx                # damage taken, fraction
            # The exact predicate `chip_target` ranks on: both our chip effects
            # deal exactly 30, so "dies to 30" is the whole rule in one bit.
            dense[v + 4] = 1.0 if hp <= CHIP_DAMAGE else 0.0
            dense[v + 5] = _cdb().prize_value(pk["id"]) / 3.0
            dense[v + 6] = min(len(energies), 6) / 6.0
            # Own-type energy count -- `energy_spread`'s entire signal. A bare
            # Munkidori and a loaded one differ HERE and nowhere else in v2.
            own = _cdb().card(pk["id"]).get("energyType")
            dense[v + 7] = min(sum(1 for e in energies
                                   if e == own or e >= 10), 4) / 4.0
            dense[v + 8] = 1.0 if owner == me else 0.0
            # What our Active could actually do to this target right now. 0.0 on
            # a damage-prevention wall, which is the `wall_defer` condition read
            # off the board instead of hardcoded by card id.
            try:
                mypl, oppl = state["players"][me], state["players"][1 - me]
                act = mypl["active"][0] if mypl.get("active") else None
                dmg = _best_damage()(act, mypl, oppl, pk) if act else 0.0
            except (KeyError, IndexError, TypeError):
                dmg = 0.0
            dense[v + 9] = min(dmg, 400.0) / 400.0
            dense[v + 10] = 1.0 if dmg >= hp else 0.0   # we can KO it now
    # Index disambiguation, and the single most load-bearing scalar here: WITHOUT
    # it two options naming two different bench slots are identical vectors.
    slot_ix = (opt.get("inPlayIndex") if t in (8, 9) else opt.get("index")) or 0
    dense[v + 11] = (min(slot_ix, 5) + 1) / 6.0

    # --- v6: card attributes (indices 37..45) -----------------------------
    # Everything above this line is v3 and must not move.
    w = OPT_DENSE_V3
    if card_id:
        ct = _cdb().card(card_id).get("cardType")
        if ct is not None and 0 <= ct < N_CARD_TYPES:
            dense[w + ct] = 1.0
    if pk is not None:
        tc = _cdb().card(pk["id"])
        dense[w + N_CARD_TYPES + 0] = 1.0 if tc.get("skills") else 0.0
        # Same predicate as features.attr_feats' weakHit, but bound to THIS
        # option rather than to a slot -- the B1 lesson is that the binding is
        # what the net cannot re-derive for itself.
        weak = tc.get("weakness") or 0
        try:
            mypl = state["players"][me]
            act = mypl["active"][0] if mypl.get("active") else None
            atk_t = _cdb().card(act["id"]).get("energyType") if act else None
        except (KeyError, IndexError, TypeError):
            atk_t = None
        dense[w + N_CARD_TYPES + 1] = 1.0 if (weak and atk_t == weak) else 0.0

    if attack_id >= N_ATTACK_IDS:
        attack_id = 0
    return dense, card_id, attack_id, target_id


In [ ]:
%%writefile agents/sa/routing.py
"""Observable matchup routing for E2 residual adapters.

Routes are inferred only from cards the acting seat can see on the opponent:
active, bench, and discard. Prize, hand, and deck contents are never used, and
post-game census labels are never used at inference.
"""
from __future__ import annotations

from typing import Iterable

import numpy as np

# Stable integer ids used by the trainer and the exported checkpoint.
ROUTE_GENERAL = 0
ROUTE_MIRROR = 1
ROUTE_ALAKAZAM = 2

ROUTE_NAMES = {
    ROUTE_GENERAL: "general",
    ROUTE_MIRROR: "mirror",
    ROUTE_ALAKAZAM: "alakazam",
}
NAME_TO_ROUTE = {name: rid for rid, name in ROUTE_NAMES.items()}

# Visible evolution lines. Alakazam is checked first so a board that somehow
# shows both lines (never observed in pds_v4) prefers the rarer specialist.
ALAKAZAM_IDS = frozenset({741, 742, 743})  # Abra / Kadabra / Alakazam
MIRROR_IDS = frozenset({646, 647, 648})    # Impidimp / Morgrem / Grimmsnarl


def route_from_ids(ids: Iterable[int]) -> int:
    """Map a set of visible opponent card ids to a route id."""
    seen = {int(x) for x in ids if int(x)}
    if seen & ALAKAZAM_IDS:
        return ROUTE_ALAKAZAM
    if seen & MIRROR_IDS:
        return ROUTE_MIRROR
    return ROUTE_GENERAL


def visible_opponent_ids(obs: dict) -> set[int]:
    """Collect opponent active/bench/discard card ids from an observation."""
    state = obs.get("current") or {}
    me = state.get("yourIndex")
    players = state.get("players") or []
    if me not in (0, 1) or len(players) < 2:
        return set()
    op = players[1 - int(me)] or {}
    ids: set[int] = set()

    def add_card(card) -> None:
        if not card:
            return
        if isinstance(card, dict):
            cid = card.get("id")
            if cid:
                ids.add(int(cid))
            for attached in card.get("cards") or []:
                add_card(attached)
        else:
            ids.add(int(card))

    active = op.get("active") or []
    if active:
        add_card(active[0] if isinstance(active, list) else active)
    for pk in op.get("bench") or []:
        add_card(pk)
    for card in op.get("discard") or []:
        add_card(card)
    return ids


def route_from_obs(obs: dict) -> int:
    """Hard route for a live observation."""
    return route_from_ids(visible_opponent_ids(obs))


def routes_from_corpus(slots: np.ndarray, opp_discard_flat: np.ndarray,
                       opp_discard_off: np.ndarray) -> np.ndarray:
    """Vector of route ids for corpus rows.

    `slots` columns 6..11 are the opponent active and five bench card ids,
    matching `features.featurize`. Opponent discard is the only other bag that
    is both observable and stored in the shards.
    """
    n = len(slots)
    routes = np.zeros(n, dtype=np.int64)
    opp_slots = slots[:, 6:].astype(np.int64, copy=False)
    ala = np.array(sorted(ALAKAZAM_IDS), dtype=np.int64)
    mir = np.array(sorted(MIRROR_IDS), dtype=np.int64)
    # Slot hits are cheap to vectorize; discard needs a per-row scan.
    slot_ala = np.isin(opp_slots, ala).any(axis=1)
    slot_mir = np.isin(opp_slots, mir).any(axis=1)
    routes[slot_ala] = ROUTE_ALAKAZAM
    routes[~slot_ala & slot_mir] = ROUTE_MIRROR
    # Fill remaining rows that only reveal the line through discard.
    need = np.where(~slot_ala & ~slot_mir)[0]
    if len(need) and len(opp_discard_flat):
        flat = opp_discard_flat.astype(np.int64, copy=False)
        for i in need:
            a, b = int(opp_discard_off[i]), int(opp_discard_off[i + 1])
            if b <= a:
                continue
            chunk = flat[a:b]
            if np.isin(chunk, ala).any():
                routes[i] = ROUTE_ALAKAZAM
            elif np.isin(chunk, mir).any():
                routes[i] = ROUTE_MIRROR
    return routes


In [ ]:
%%writefile agents/sa/policynet.py
"""Numpy inference for the cloned policy (see scripts/train_policy.py)."""
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np

from .features import attr_feats, extra_feats, featurize
from .optfeat import option_features, pool_scalars, pool_width
from .routing import ROUTE_GENERAL, route_from_obs

# SA_PNET_PATH lets an arena run score a candidate net without overwriting the
# shipped one. Kaggle sets no env vars, so there it is always the bundled npz.
_PATH = Path(os.environ.get("SA_PNET_PATH")
             or Path(__file__).resolve().parent / "policy_net.npz")
# how many options to take on a variable-count select; see Net.choose
COUNT_MODE = os.environ.get("SA_COUNT_MODE", "table")
_BAGS = ("my_hand", "my_discard", "opp_discard")
SEL_DENSE = 14

_net = None
_tried = False


def _sel_features(sel: dict) -> np.ndarray:
    v = np.zeros(SEL_DENSE, dtype=np.float32)
    t = sel.get("type") or 0
    if t < 11:
        v[t] = 1.0
    v[11] = sel.get("minCount", 0) / 5.0
    v[12] = sel.get("maxCount", 0) / 5.0
    v[13] = (sel.get("context") or 0) / 50.0
    return v


class Net:
    def __init__(self, z):
        self.slot_emb = z["slot_emb"]
        self.bag_emb = z["bag_emb"]
        self.card_emb = z["card_emb"]
        self.atk_emb = z["atk_emb"]
        # Layers are stored generically (`sfc{i}_w` / `head{i}_w`) so the net
        # can be made deeper without touching this file. Nets exported before
        # that change used fixed ws/w1/w2 names -- still loadable.
        if "n_sfc" in z:
            self.state_layers = [(z[f"sfc{i}_w"], z[f"sfc{i}_b"])
                                 for i in range(int(z["n_sfc"][0]))]
            self.head_layers = [(z[f"head{i}_w"], z[f"head{i}_b"])
                                for i in range(int(z["n_head"][0]))]
        else:
            self.state_layers = [(z["ws"], z["bs"])]
            self.head_layers = [(z["w1"], z["b1"]), (z["w2"], z["b2"])]
        self.count_frac = z["count_frac"] if "count_frac" in z else None
        # E1 heads are optional and append-only. Legacy checkpoints keep the
        # exact policy path; multitask checkpoints expose these predictions for
        # diagnostics, learned count selection, and later planning.
        self.outcome_head = ((z["outcome_w"], z["outcome_b"])
                             if "outcome_w" in z else None)
        self.count_head = ((z["count_w"], z["count_b"])
                           if "count_w" in z else None)
        # The v5 pooled option-set block, 0 for every net before day 13. Recorded
        # rather than derived: the v4 and v5 state widths are both legal, so
        # `state_in` alone cannot tell them apart.
        self.n_pool = int(z["n_pool"][0]) if "n_pool" in z else 0
        # Which members of the v4 block this net was shown (features.X_GROUPS).
        # Absent = all of them, which is every net before day 13.
        self.x_mask = z["x_mask"] if "x_mask" in z else None
        # E2 residual adapters. Absent keys keep the exact legacy policy path.
        self.adapters: dict[str, list[tuple[np.ndarray, np.ndarray]]] = {}
        self.adapter_route_ids: dict[str, int] = {}
        if "adapter_names" in z:
            names = [str(x) for x in z["adapter_names"].tolist()]
            route_ids = (z["adapter_route_ids"].tolist()
                         if "adapter_route_ids" in z else [])
            for i, name in enumerate(names):
                n_layers = int(z[f"adapter_{name}_n"][0])
                layers = [(z[f"adapter_{name}{j}_w"],
                           z[f"adapter_{name}{j}_b"])
                          for j in range(n_layers)]
                self.adapters[name] = layers
                if i < len(route_ids):
                    self.adapter_route_ids[name] = int(route_ids[i])
                else:
                    from .routing import NAME_TO_ROUTE
                    self.adapter_route_ids[name] = NAME_TO_ROUTE[name]
        # --- and the main-side optional blocks (v6 attr, v7 vocab) ---
        # The v6 card-attribute block, 0 for every net before day 20. Recorded
        # for the same reason as n_pool: with three optional blocks, `state_in`
        # no longer identifies the layout on its own.
        self.n_attr = int(z["n_attr"][0]) if "n_attr" in z else 0
        # Which members of the v6 block this net was shown (features.A_GROUPS).
        self.a_mask = z["a_mask"] if "a_mask" in z else None
        # The v7 vocabulary remap, absent on every net before day 21. Each table
        # was collapsed to the rows the corpus trained: row 0 = PAD, row 1 = UNK,
        # rows 2.. = `vocab_<table>` in order. Without it a card the corpus never
        # contained reads an untrained N(0,1) row whose norm is indistinguishable
        # from a trained one's, so the net cannot tell "unknown" from "known".
        self.lut = None
        if "vocab_slot_emb" in z:
            from .features import N_CARD_IDS
            from .optfeat import N_ATTACK_IDS
            self.lut = {}
            for t in ("slot_emb", "bag_emb", "card_emb", "atk_emb"):
                ids = z[f"vocab_{t}"].astype(np.int64)
                size = N_ATTACK_IDS if t == "atk_emb" else N_CARD_IDS
                lut = np.full(max(size, int(ids[-1]) + 1 if ids.size else size),
                              1, dtype=np.int64)      # 1 = UNK
                lut[0] = 0                            # 0 = PAD
                lut[ids] = np.arange(2, 2 + ids.size, dtype=np.int64)
                self.lut[t] = lut

    def _m(self, table: str, ids):
        """Raw card/attack id -> this net's row. Identity on a pre-v7 net."""
        if self.lut is None:
            return ids
        lut = self.lut[table]
        a = np.asarray(ids)
        # Out-of-range cannot happen -- features.py clamps to 0 -- but an id past
        # the table falls to UNK rather than raising in the middle of a match.
        return np.where(a < len(lut), lut[np.clip(a, 0, len(lut) - 1)], 1)

    @property
    def state_in(self) -> int:
        return self.state_layers[0][0].shape[1]

    @property
    def state_out(self) -> int:
        return self.state_layers[-1][0].shape[0]

    @property
    def head_in(self) -> int:
        return self.head_layers[0][0].shape[1]

    @property
    def opt_in(self) -> int:
        """How many per-option dense features THIS net was trained on.

        Derived rather than read from `optfeat.OPT_DENSE`, because a v2 net
        (25) and a v3 net (37) have to be able to run in the same process for a
        head-to-head A/B across the feature change (HANDOFF rule 4). The v3 block
        is appended, so slicing to this width gives a v2 net byte-identical input
        to what it was trained on."""
        return (self.head_in - self.state_out
                - 2 * self.card_emb.shape[1] - self.atk_emb.shape[1])

    def _forward(self, obs: dict) -> tuple[np.ndarray, np.ndarray | None]:
        """Return option logits and the shared state representation."""
        state = obs["current"]
        sel = obs["select"]
        me = state["yourIndex"]
        opts = sel.get("option") or []
        n = len(opts)
        if n == 0:
            return np.zeros(0, dtype=np.float32), None
        # The per-option encoding is built BEFORE the state, because the v5 pool
        # is a summary of it. Nets without the pool ignore it and slice it off,
        # so this costs them nothing but the loop order.
        emb = self.card_emb.shape[1]
        ow = self.opt_in
        oenc = np.empty((n, ow + 3 * emb), dtype=np.float32)
        for i, o in enumerate(opts):
            od, cid, aid, tid = option_features(obs, o)
            # Slice to the width this net was trained at -- the v3 target block
            # is appended, so a v2 net simply does not see it.
            oenc[i, :ow] = od[:ow]
            oenc[i, ow:ow + emb] = self.card_emb[self._m("card_emb", cid)]
            oenc[i, ow + emb:ow + 2 * emb] = self.atk_emb[
                self._m("atk_emb", aid)]
            oenc[i, ow + 2 * emb:] = self.card_emb[self._m("card_emb", tid)]

        dense, bags = featurize(state, me)
        parts = [dense,
                 self.slot_emb[self._m("slot_emb", bags["slots"])].reshape(-1)]
        for name in _BAGS:
            b = bags[name]
            parts.append(self.bag_emb[self._m("bag_emb", b)].mean(axis=0)
                         if len(b)
                         else np.zeros(self.bag_emb.shape[1],
                                       dtype=np.float32))
        parts.append(_sel_features(sel))
        # The v4 block goes LAST, so slicing to this net's own `state_in` feeds
        # a v3 net byte-identical input (features.py, "APPENDED, NEVER
        # INSERTED"). Same trick as `opt_in` one level up.
        xd, xids = extra_feats(state, sel, me)
        if self.x_mask is not None:     # a drop-one ablation arm (day 13)
            from .features import N_EXTRA
            xd = xd * self.x_mask[:N_EXTRA]
            xids = np.where(self.x_mask[N_EXTRA:] > 0, xids, 0)
        parts.append(xd)
        parts.append(self.slot_emb[self._m("slot_emb", xids)].reshape(-1))
        # ...and the v5 pool goes after v4, same rule (optfeat.pool_width).
        if self.n_pool:
            parts += [oenc.mean(axis=0), oenc.max(axis=0), pool_scalars(n)]
        # ...and the v6 attribute block goes after v5, same rule again. Computed
        # only when the net was trained with it -- attr_feats walks 12 slots and
        # a v5 net would pay for a vector it then slices off.
        if self.n_attr:
            a = attr_feats(state, me)
            parts.append(a * self.a_mask if self.a_mask is not None else a)
        x = np.concatenate(parts)
        srepr = x[:self.state_in]
        for w, b in self.state_layers:      # every state layer is relu'd
            srepr = np.maximum(w @ srepr + b, 0.0)

        sw = len(srepr)
        feats = np.empty((n, self.head_in), dtype=np.float32)
        feats[:, :sw] = srepr
        feats[:, sw:] = oenc
        h = feats
        for j, (w, b) in enumerate(self.head_layers):
            h = h @ w.T + b
            if j < len(self.head_layers) - 1:   # last layer is the raw logit
                h = np.maximum(h, 0.0)
        logits = h.reshape(-1)
        if self.adapters:
            route = route_from_obs(obs)
            if route != ROUTE_GENERAL:
                for name, route_id in self.adapter_route_ids.items():
                    if route_id != route:
                        continue
                    residual = feats
                    layers = self.adapters[name]
                    for j, (w, b) in enumerate(layers):
                        residual = residual @ w.T + b
                        if j < len(layers) - 1:
                            residual = np.maximum(residual, 0.0)
                    logits = logits + residual.reshape(-1)
                    break
        return logits, srepr

    def scores(self, obs: dict) -> np.ndarray:
        """Logit per option of obs['select']."""
        return self._forward(obs)[0]

    @staticmethod
    def _head_value(head, srepr: np.ndarray) -> float | None:
        if head is None:
            return None
        w, b = head
        return float((w @ srepr + b).reshape(-1)[0])

    def win_prob(self, obs: dict) -> float | None:
        """Auxiliary E1 outcome estimate, or None for legacy checkpoints."""
        _, srepr = self._forward(obs)
        if srepr is None:
            return None
        logit = self._head_value(self.outcome_head, srepr)
        if logit is None:
            return None
        return float(1.0 / (1.0 + np.exp(-np.clip(logit, -30.0, 30.0))))

    def choose(self, obs: dict) -> list[int]:
        """Rank options by logit; how MANY to take is the harder half.

        `table` (default): a data-derived per-(selectType, context) mean count
        fraction -- one number for the whole bucket, so it is wrong on every
        select whose true count is bimodal.
        `expect`: sum the per-option sigmoids, i.e. the model's own expected
        number of chosen options. Only meaningful for a net trained with a
        pointwise (BCE) term -- a pure listwise net's logits are not calibrated
        probabilities, only a valid ranking.
        """
        sc, srepr = self._forward(obs)
        return self.pick(obs, sc, srepr)

    def pick(self, obs: dict, sc: np.ndarray,
             srepr: np.ndarray | None) -> list[int]:
        """Rank by `sc` and decide HOW MANY to take.

        Split out of `choose` so an ensemble can supply its own combined
        scores without re-implementing the count rule -- rule 18: do not
        re-derive a statistic the tool already computes. `choose` is exactly
        `pick` applied to this net's own forward pass.
        """
        sel = obs["select"]
        mn = sel.get("minCount", 0)
        mx = sel.get("maxCount", 0)
        order = list(np.argsort(-sc))
        k = mx
        if mx > mn:
            if COUNT_MODE == "expect":
                probs = 1.0 / (1.0 + np.exp(-np.clip(sc, -30.0, 30.0)))
                k = int(round(float(probs.sum())))
            elif COUNT_MODE == "learned" and self.count_head is not None:
                logit = self._head_value(self.count_head, srepr)
                frac = 1.0 / (1.0 + np.exp(-np.clip(logit, -30.0, 30.0)))
                k = mn + int(round(float(frac) * (mx - mn)))
            else:
                frac = 1.0
                if self.count_frac is not None:
                    t = min(sel.get("type") or 0, 10)
                    ctx = min(sel.get("context") or 0, 63)
                    frac = float(self.count_frac[t, ctx])
                k = mn + int(round(frac * (mx - mn)))
            k = max(mn, min(k, mx))
        return [int(i) for i in order[:k]]


class Ensemble:
    """Several independently-trained nets voting on one decision.

    🔴 **Why this is not the closed capacity axis.** §8w made ONE net bigger
    (2.6x and 8.2x the parameters) and bought two decisions out of 12,939 and
    then lost 43 -- the features, not the parameter count, were binding. An
    ensemble does something else: it averages functions that were fitted
    *independently*, which cancels the part of each net's error that is
    idiosyncratic to its own initialisation rather than shared.

    ⚡ **And this project has already measured that the idiosyncratic part is
    large.** §5.6/E8 found two same-recipe nets differing only in `--seed`
    swinging **0.073** against each other in a direct mirror head-to-head,
    against ±0.036 of sampling noise -- they disagree far more than the games
    alone explain, i.e. they make DIFFERENT mistakes. That measurement was
    filed as a warning about our instrument; it is also the precondition for
    averaging to pay.

    ⚠ **Probabilities, not raw logits.** A listwise loss fixes the ranking, not
    the scale: two nets can be equally good and differ by a constant factor in
    logit magnitude, and a raw-logit mean would then be a weighted vote with
    weights nobody chose. Softmax each net over the option set first, then
    average, so every member gets exactly one vote. `--raw` overrides.

    ⚠ **The count comes from the FIRST member**, via its own `pick`. Ensembling
    the count fraction as well would confound "which options" with "how many",
    and the count rule is a per-(type, context) table, not a scored quantity.
    """

    def __init__(self, nets: list["Net"], raw: bool = False):
        if not nets:
            raise ValueError("Ensemble needs at least one net")
        self.nets = nets
        self.raw = raw
        # exposed so callers that introspect a Net (x_mask checks, vocab
        # guards, the flip probe) see the primary member's shape
        self.primary = nets[0]

    def __len__(self) -> int:
        return len(self.nets)

    # Passthroughs so anything that introspects a net (the build smoke, the
    # dim guard's callers) sees the shape it is actually being fed. Every
    # member is verified same-architecture by `load` before it gets here.
    @property
    def opt_in(self) -> int:
        return self.primary.opt_in

    @property
    def state_in(self) -> int:
        return self.primary.state_in

    def scores(self, obs: dict) -> np.ndarray:
        acc = None
        for net in self.nets:
            s = np.asarray(net.scores(obs), dtype=np.float64)
            if not self.raw:
                z = s - float(s.max())
                e = np.exp(z)
                s = e / e.sum()
            acc = s if acc is None else acc + s
        return acc / float(len(self.nets))

    def choose(self, obs: dict) -> list[int]:
        sel = obs.get("select") or {}
        n = len(sel.get("option") or [])
        if n == 0:
            return []
        # srepr is only consulted by the `learned` count mode, which the
        # shipped nets do not use; the primary's own forward supplies it when
        # it is needed rather than being faked.
        srepr = None
        if COUNT_MODE == "learned" and self.primary.count_head is not None:
            srepr = self.primary._forward(obs)[1]
        return self.primary.pick(obs, self.scores(obs), srepr)


def load_ensemble(paths: list[str], raw: bool = False) -> Ensemble | None:
    """Load several nets for voting. Returns None if ANY member fails.

    Strict on purpose: a silently-dropped member is a different agent playing
    under the ensemble's name, which is day 22's defect 2 with extra steps.
    """
    nets = []
    for p in paths:
        net = load(p)
        if net is None:
            return None
        nets.append(net)
    return Ensemble(nets, raw=raw)


def load(path) -> Net | None:
    """Load a specific npz, returning None unless it matches the CURRENT
    feature dims. The guard is what stops a stale net from being used
    silently after a feature change -- never remove it."""
    path = Path(path)
    if not path.exists():
        return None
    try:
        net = Net(np.load(path))
        from .features import DENSE_DIM, N_ATTR, N_EXTRA, N_XSLOT
        from .optfeat import KNOWN_OPT_DENSE
        emb = net.slot_emb.shape[1]
        base = (DENSE_DIM + 12 * emb + 3 * net.bag_emb.shape[1] + SEL_DENSE)
        # Four legitimate state widths now, exactly as with the option block:
        # the v3 layout, + the appended v4 block, + the appended v5 pool, + the
        # appended v6 attribute block. `n_pool` and `n_attr` say which one a net
        # is, and they must AGREE with the width -- a net claiming a block it was
        # not trained with would silently read hundreds of columns of garbage.
        v4 = base + N_EXTRA + N_XSLOT * emb
        v5 = v4 + pool_width(net.opt_in, emb)
        want = v5 if net.n_pool else v4
        if net.n_attr:
            want += N_ATTR
        # A v7 net's tables are sized BY the map that travels with them. If the
        # two disagree, every lookup is off by however far they drifted and the
        # agent plays a scrambled net at full confidence -- the exact failure
        # E6 measured at -0.251. Check them against each other, not against a
        # constant: the row count IS 2 + len(vocab), PAD and UNK.
        if net.lut is not None:
            for t, w in (("slot_emb", net.slot_emb), ("bag_emb", net.bag_emb),
                         ("card_emb", net.card_emb), ("atk_emb", net.atk_emb)):
                if w.shape[0] != int((net.lut[t] > 1).sum()) + 2:
                    return None
        if (net.state_in in (base, want) and net.opt_in in KNOWN_OPT_DENSE
                and net.n_pool in (0, pool_width(net.opt_in, emb))
                and net.n_attr in (0, N_ATTR)
                and (net.a_mask is None or net.a_mask.shape == (N_ATTR,))):
            return net
    except Exception:
        pass
    return None


def get() -> Net | None:
    """The process-wide singleton, loaded from `SA_PNET_PATH` or the bundle.

    🔴 **Announces WHICH net it just loaded, once, on stderr.** E33 rolled out
    with this singleton (`policy_net.npz`, the v2 clone) while its seats played
    `out/policy_v5_s2.npz`, published a calibration verdict, and had to
    withdraw it; `p82` had already warned in writing that scoring one net's
    options against another net's games *"returns a plausible number, not an
    error"*. Which net a probe actually used was recoverable only by reading
    the source and knowing the environment, so nothing in a log could ever
    contradict a wrong assumption. Now every run says so itself.

    ⚠ **stderr, not stdout, and that is load-bearing**: `kaggle/score.py` and
    the p5x drivers parse stdout for the arena's score line and drop the rest,
    so a notice printed there is a notice nobody reads -- the same reasoning
    `arena.build_agent` gives for its deck-mismatch warning.
    """
    global _net, _tried
    if not _tried:
        _tried = True
        _net = load(_PATH)
        try:
            tag = "MISSING"
            if _net is not None:
                import hashlib
                tag = "#" + hashlib.md5(
                    Path(_PATH).read_bytes()).hexdigest()[:8]
            # ⚠ Build this OUTSIDE the f-string. Kaggle's episode runner is
            # Python 3.11 (`kaggle_environments` under python3.11/dist-packages)
            # even though Kaggle *notebooks* are 3.12, and a replacement field
            # that spans lines is PEP 701 -- i.e. 3.12+ ONLY. As a multi-line
            # f-string this raised `SyntaxError: unterminated string literal` at
            # IMPORT, so `main.py`'s `from sa.bcagent import PolicyAgent` never
            # completed, both seats died in 0.04s and submission 55489084 came
            # back "Validation Episode failed." A logging nicety took the whole
            # agent down, on the one path no local smoke could see.
            note = ("" if os.environ.get("SA_PNET_PATH")
                    else "  (repo default -- set SA_PNET_PATH to pin a "
                         "different one)")
            print(f"[policynet] singleton = {_PATH} {tag}{note}",
                  file=sys.stderr, flush=True)
        except Exception:
            pass
    return _net


In [ ]:
%%writefile scripts/build_policy_dataset.py
"""Build policy-cloning shards from replay JSONs.

    python scripts/build_policy_dataset.py --out artifacts/pds/d26 replays/2026-07-26

One row per select with >=2 options: state features + per-option features +
multi-hot chosen mask (from the replay's actual action).

⚠ By default this clones BOTH seats of every game -- every archetype and every
skill level in the dump. `--player NAME` keeps only the seats belonging to the
named team(s), which is how an EXPERT corpus is built from a third-party dump:

    python scripts/build_policy_dataset.py --out artifacts/pds_expert \\
        --player "Raja Biswas" --player "Sixth Sense" \\
        replays/sixth_sense_31-07-2026

`--ratings` tags every row with the LB score of the demonstrator who made that
choice, so a corpus can be reweighted or sliced by demonstrator strength (B7):

    python scripts/build_policy_dataset.py --out artifacts/pds_v3r \\
        --ratings out/lb/pokemon-tcg-ai-battle.zip replays/2026-07-26
"""
from __future__ import annotations

import argparse
import csv
import json
import sys
import zipfile
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[1]
for sub in ("src", "agents", ""):
    p = str(ROOT / sub) if sub else str(ROOT)
    if p not in sys.path:
        sys.path.insert(0, p)

from ptcg.env import sdk  # noqa: E402

sdk.load()

from sa.features import attr_feats, extra_feats, featurize  # noqa: E402
from sa.optfeat import option_features, OPT_DENSE  # noqa: E402

SHARD_ROWS = 60_000
SEL_DENSE = 14
NO_RATING = np.float32("nan")


def load_ratings(path: Path) -> tuple[dict[str, float], dict[int, float]]:
    """Kaggle LB export -> (by team name, by teamId). Accepts the .zip that
    `competition_leaderboard_download` writes or the .csv inside it."""
    if path.suffix == ".zip":
        with zipfile.ZipFile(path) as z:
            names = [n for n in z.namelist() if n.endswith(".csv")]
            if not names:
                raise SystemExit(f"{path}: no .csv inside")
            text = z.read(names[0]).decode("utf-8-sig")
    else:
        text = path.read_text(encoding="utf-8-sig")
    by_name: dict[str, float] = {}
    by_id: dict[int, float] = {}
    by_user: dict[str, float] = {}
    n_teams = 0
    for row in csv.DictReader(text.splitlines()):
        try:
            score = float(row["Score"])
        except (TypeError, ValueError, KeyError):
            continue
        n_teams += 1
        by_name[row["TeamName"]] = score
        try:
            by_id[int(row["TeamId"])] = score
        except (TypeError, ValueError, KeyError):
            pass
        # A replay's TeamNames can hold a MEMBER's username rather than the
        # team's display name (teams merge and rename; `zoroark190` is how the
        # LB's #1 `James Cox & Henry Chao` appears in 07-26 replays). Exact
        # member matches are safe; ambiguous ones are dropped, not guessed.
        for user in (row.get("TeamMemberUserNames") or "").split(","):
            user = user.strip()
            if user:
                by_user[user] = score if user not in by_user else float("nan")
    # Display names always win; a username that collides with some other team's
    # display name, or with a second team, is dropped rather than resolved.
    n_user = 0
    for user, score in by_user.items():
        if user not in by_name and score == score:
            by_name[user] = score
            n_user += 1
    print(f"ratings: {n_teams} teams (+{n_user} member usernames) "
          f"from {path.name}")
    return by_name, by_id


def name_id(name: str) -> int:
    """Stable per-team id for dumps with no `episodes_meta.json` sidecar (the
    daily replay dirs). Deterministic across runs and processes, unlike
    `hash()`, so a corpus rebuilt tomorrow keeps the same ids."""
    import hashlib
    h = hashlib.sha1(name.encode("utf-8")).digest()
    return -int.from_bytes(h[:6], "big")   # negative = derived, not Kaggle's


def load_episode_meta(d: Path) -> dict[int, dict[int, tuple[int, int]]]:
    """`episodes_meta.json` -> {episode_id: {seat: (submissionId, teamId)}}.
    Only targeted per-team dumps carry it; day dumps return {}."""
    p = d / "episodes_meta.json"
    if not p.exists():
        return {}
    out: dict[int, dict[int, tuple[int, int]]] = {}
    for ep in json.loads(p.read_text(encoding="utf-8")):
        seats = {}
        for a in ep.get("agents") or []:
            seats[int(a.get("index") or 0)] = (int(a.get("submissionId") or -1),
                                               int(a.get("teamId") or -1))
        out[int(ep["id"])] = seats
    return out


def sel_features(sel: dict) -> np.ndarray:
    v = np.zeros(SEL_DENSE, dtype=np.float32)
    t = sel.get("type") or 0
    if t < 11:
        v[t] = 1.0
    v[11] = sel.get("minCount", 0) / 5.0
    v[12] = sel.get("maxCount", 0) / 5.0
    v[13] = (sel.get("context") or 0) / 50.0
    return v


class Writer:
    def __init__(self, out_dir: Path):
        self.out_dir = out_dir
        out_dir.mkdir(parents=True, exist_ok=True)
        self.idx = 0
        self.reset()

    def reset(self):
        self.sd, self.slots, self.seld, self.gid = [], [], [], []
        self.xd, self.xslots, self.attr = [], [], []
        self.bags = {"my_hand": [], "my_discard": [], "opp_discard": []}
        self.od, self.ocard, self.oatk, self.otgt, self.chosen = [], [], [], [], []
        self.off = [0]
        self.won = []
        self.rating, self.opp_rating, self.team_id, self.sub_id = [], [], [], []

    def add(self, dense, bags, seld, opts, chosen_mask, gid, won,
            rating=NO_RATING, opp_rating=NO_RATING, team_id=-1, sub_id=-1,
            extra=None, attr=None):
        self.sd.append(dense)
        # The v4 state block (features.extra_feats). Written unconditionally --
        # a trainer that does not want it simply does not read these arrays,
        # which is what makes the v3 control run on the IDENTICAL rows.
        xd, xids = extra
        self.xd.append(xd)
        self.xslots.append(xids)
        # The v6 card-attribute block (features.attr_feats), same contract:
        # always written, so a v5 control trains on the IDENTICAL rows.
        self.attr.append(attr)
        self.slots.append(bags["slots"])
        for k in self.bags:
            self.bags[k].append(bags[k])
        self.seld.append(seld)
        od, oc, oa, ot = opts
        self.od.append(od)
        self.ocard.append(oc)
        self.oatk.append(oa)
        self.otgt.append(ot)
        self.chosen.append(chosen_mask)
        self.off.append(self.off[-1] + len(oc))
        self.gid.append(gid)
        self.won.append(won)
        self.rating.append(rating)
        self.opp_rating.append(opp_rating)
        self.team_id.append(team_id)
        self.sub_id.append(sub_id)
        if len(self.sd) >= SHARD_ROWS:
            self.flush()

    def flush(self):
        if not self.sd:
            return
        arrs = {
            "dense": np.stack(self.sd),
            "slots": np.stack(self.slots),
            "seld": np.stack(self.seld),
            "xdense": np.stack(self.xd),
            "xslots": np.stack(self.xslots),
            "attr": np.stack(self.attr),
            "gid": np.asarray(self.gid, dtype=np.int64),
            "won": np.asarray(self.won, dtype=np.float32),
            # B7: who made this choice, and how good are they? NaN = the team
            # was not on the LB snapshot (renamed, or withdrawn).
            "rating": np.asarray(self.rating, dtype=np.float32),
            "opp_rating": np.asarray(self.opp_rating, dtype=np.float32),
            "team_id": np.asarray(self.team_id, dtype=np.int64),
            "sub_id": np.asarray(self.sub_id, dtype=np.int64),
            "opt_dense": np.concatenate(self.od),
            "opt_card": np.concatenate(self.ocard),
            "opt_attack": np.concatenate(self.oatk),
            "opt_target": np.concatenate(self.otgt),
            "opt_chosen": np.concatenate(self.chosen),
            "opt_off": np.asarray(self.off, dtype=np.int64),
        }
        for k, lists in self.bags.items():
            off = np.zeros(len(lists) + 1, dtype=np.int64)
            for i, a in enumerate(lists):
                off[i + 1] = off[i] + len(a)
            arrs[f"bag_{k}_flat"] = (np.concatenate(lists) if off[-1]
                                     else np.zeros(0, dtype=np.int32))
            arrs[f"bag_{k}_off"] = off
        path = self.out_dir / f"shard_{self.idx:03d}.npz"
        np.savez_compressed(path, **arrs)
        print(f"  wrote {path.name}: {len(self.sd)} rows")
        self.idx += 1
        self.reset()


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("dirs", nargs="+")
    ap.add_argument("--out", required=True)
    ap.add_argument("--player", action="append", default=[],
                    help="keep only seats owned by this team name (repeatable). "
                         "Default: clone both seats of every game.")
    ap.add_argument("--players-file",
                    help="UTF-8 file of team names, one per line, added to "
                         "--player (for name lists the shell cannot quote)")
    ap.add_argument("--exclude", action="append", default=[],
                    help="drop these team names even if --player/--players-file "
                         "names them. ⚠ Always exclude OURSELVES from a control "
                         "population: our own agent's selects are what the net "
                         "was fitted to, so leaving `Scio` in inflates "
                         "agreement toward 100%% for those rows.")
    ap.add_argument("--ratings",
                    help="Kaggle leaderboard export (.zip or .csv). Tags every "
                         "row with the demonstrator's LB score (B7). Without "
                         "it every row's rating is NaN.")
    ap.add_argument("--aliases", default="replays/team_aliases.tsv",
                    help="TSV of `replay name<TAB>LB team name` for teams that "
                         "renamed or merged and cannot be matched exactly. "
                         "Missing file = no aliases.")
    args = ap.parse_args()

    keep = set(args.player)
    if args.players_file:
        keep |= {ln.strip() for ln
                 in Path(args.players_file).read_text(encoding="utf-8").splitlines()
                 if ln.strip() and not ln.startswith("#")}
    # ⚠ An empty filter means "clone both seats of every game" -- silently the
    # OPPOSITE of what was asked. An empty --players-file used to build a
    # whole-dump corpus under an expert corpus's name.
    if (args.player or args.players_file) and not keep:
        raise SystemExit("--player/--players-file given but resolved to zero "
                         "names; refusing to build an unfiltered corpus")
    drop = set(args.exclude)
    if drop:
        hit = keep & drop
        keep -= drop
        print(f"--exclude: dropped {sorted(hit)} from the demonstrator set")
        if (args.player or args.players_file) and not keep:
            raise SystemExit("--exclude removed every demonstrator")
    rate_name, rate_id = ({}, {})
    alias: dict[str, str] = {}
    team_name: dict[int, str] = {}
    ap_path = ROOT / args.aliases
    if ap_path.exists():
        for ln in ap_path.read_text(encoding="utf-8").splitlines():
            if not ln.strip() or ln.startswith("#") or "\t" not in ln:
                continue
            old, new = (s.strip() for s in ln.split("\t", 1))
            alias[old] = new
    if args.ratings:
        rate_name, rate_id = load_ratings(Path(args.ratings))
        n_alias = 0
        for old, new in alias.items():
            if new in rate_name and old not in rate_name:
                rate_name[old] = rate_name[new]
                n_alias += 1
            elif new not in rate_name:
                print(f"  alias target not on the LB, ignored: {new!r}",
                      file=sys.stderr)
        print(f"  aliases: {n_alias} applied from {ap_path.name}")
    writer = Writer(ROOT / args.out)
    n_games = n_rows = n_err = n_skip_game = n_skip_seat = 0
    n_rated = n_unrated = 0
    unrated_names: dict[str, int] = {}
    for d in args.dirs:
        ep_meta = load_episode_meta(Path(d))
        for path in sorted(Path(d).glob("*.json")):
            if path.name == "manifest.json" or not path.stem.isdigit():
                continue   # sidecars: manifest.json, episodes_meta.json, ...
            try:
                rep = json.loads(path.read_text(encoding="utf-8"))
                rewards = rep["rewards"]
                if rewards[0] is None or rewards[1] is None:
                    continue
                names = (rep.get("info") or {}).get("TeamNames") or []
                seats = ({i for i, n in enumerate(names) if n in keep}
                         if keep else
                         ({i for i, n in enumerate(names) if n not in drop}
                          if drop else None))
                if seats is not None and not seats:
                    n_skip_game += 1
                    continue
                vis = rep["steps"][0][0].get("visualize") or []
                try:
                    gid = int(path.stem)
                except ValueError:
                    gid = hash(path.stem) & 0x7FFFFFFF
                n_games += 1
                # Per-seat identity. The sidecar's teamId is authoritative when
                # present -- `info.TeamNames` is a DISPLAY name and teams rename
                # mid-window (§8q: one demonstrator appeared as two).
                meta = ep_meta.get(gid, {})
                seat_rating: dict[int, float] = {}
                seat_team: dict[int, int] = {}
                seat_sub: dict[int, int] = {}
                for i in range(max(len(names), len(meta))):
                    sub, tid = meta.get(i, (-1, -1))
                    r = rate_id.get(tid) if tid >= 0 else None
                    nm = names[i] if i < len(names) else ""
                    if r is None and nm:
                        r = rate_name.get(nm)
                    seat_rating[i] = float(r) if r is not None else float("nan")
                    if tid < 0 and nm:
                        # No sidecar: identify the demonstrator by name. The
                        # alias file has already merged renames, so this does
                        # not split one team in two (§8q).
                        tid = name_id(alias.get(nm, nm))
                        team_name[tid] = alias.get(nm, nm)
                    elif tid >= 0 and nm:
                        team_name[tid] = nm
                    seat_team[i] = tid
                    seat_sub[i] = sub
                    if args.ratings:
                        if r is None:
                            n_unrated += 1
                            nm = names[i] if i < len(names) else f"seat{i}"
                            unrated_names[nm] = unrated_names.get(nm, 0) + 1
                        else:
                            n_rated += 1
                for v in vis:
                    obs = v.get("obs")
                    if not obs or not obs.get("current") or not obs.get("select"):
                        continue
                    state = obs["current"]
                    if state["result"] != -1:
                        continue
                    sel = obs["select"]
                    opts = sel.get("option") or []
                    if len(opts) < 2:
                        continue
                    action = v.get("selected")
                    if action is None:
                        action = v.get("action")
                    if not isinstance(action, list):
                        continue
                    picked = [a for a in action
                              if isinstance(a, int) and 0 <= a < len(opts)]
                    if len(picked) != len(action):
                        continue
                    me = state["yourIndex"]
                    if seats is not None and me not in seats:
                        n_skip_seat += 1
                        continue
                    won = 1.0 if rewards[me] > rewards[1 - me] else 0.0
                    dense, bags = featurize(state, me)
                    od = np.zeros((len(opts), OPT_DENSE), dtype=np.float32)
                    oc = np.zeros(len(opts), dtype=np.int32)
                    oa = np.zeros(len(opts), dtype=np.int32)
                    ot = np.zeros(len(opts), dtype=np.int32)
                    for i, o in enumerate(opts):
                        od[i], oc[i], oa[i], ot[i] = option_features(obs, o)
                    mask = np.zeros(len(opts), dtype=np.float32)
                    mask[picked] = 1.0
                    writer.add(dense, bags, sel_features(sel),
                               (od, oc, oa, ot), mask, gid, won,
                               extra=extra_feats(state, sel, me),
                               attr=attr_feats(state, me),
                               rating=seat_rating.get(me, float("nan")),
                               opp_rating=seat_rating.get(1 - me,
                                                          float("nan")),
                               team_id=seat_team.get(me, -1),
                               sub_id=seat_sub.get(me, -1))
                    n_rows += 1
            except Exception as exc:
                n_err += 1
                if n_err <= 5:
                    print(f"  {path.name}: {type(exc).__name__}: {exc}",
                          file=sys.stderr)
    writer.flush()
    if team_name:
        # `team_id` in the shards is an int; this is how a report table gets a
        # name next to it. Merged with any existing map so a corpus built one
        # day-dir at a time accumulates rather than overwrites.
        tp = ROOT / args.out / "teams.json"
        old = (json.loads(tp.read_text(encoding="utf-8"))
               if tp.exists() else {})
        old.update({str(k): v for k, v in team_name.items()})
        tp.write_text(json.dumps(old, ensure_ascii=False, indent=1),
                      encoding="utf-8")
        print(f"  wrote {tp.name}: {len(old)} demonstrators")
    # ⚠ rule 9: a filter that matches nothing writes an empty corpus and exits 0.
    # A mistyped team name (CJK homoglyph, a rename) looks exactly like this.
    if keep and not n_games:
        raise SystemExit(f"player filter {sorted(keep)} matched ZERO games in "
                         f"{args.dirs}; check the exact team name")
    print(f"games={n_games} rows={n_rows} errors={n_err}")
    if args.ratings:
        tot = n_rated + n_unrated
        print(f"ratings: {n_rated}/{tot} seats matched the LB snapshot "
              f"({n_rated / max(tot, 1):.1%})")
        # ⚠ rule 9: an unmatched name is a SILENT NaN. Name the biggest misses
        # so a rename or an encoding mismatch cannot hide as "sparse data".
        for nm, c in sorted(unrated_names.items(), key=lambda kv: -kv[1])[:8]:
            print(f"  unrated: {nm!r} x{c}")
    if keep:
        print(f"player filter {sorted(keep)}: skipped {n_skip_game} games with "
              f"no matching seat, {n_skip_seat} opponent-seat rows")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile scripts/train_policy.py
"""Train the policy net (behavioral cloning of top players' selects).

    python scripts/train_policy.py --ds artifacts/pds --out agents/sa/policy_net.npz

The state is encoded once per row, then scored against each option.

    state:  dense + slot_emb(12x16) + 3 bag means(16) + seld(14)
            -> MLP(--state-h) -> state_repr
    option: opt_dense + card_emb(16) + atk_emb(16) + tgt_emb(16)
    score:  MLP([state_repr, option], --head-h) -> 1

`--loss listwise` optimizes softmax cross-entropy over each select's option
set, which is what the agent actually does at inference (rank the options and
take the top k). `--loss bce` is the original pointwise objective; it treats
every option independently and does not model "which of these is best".

Layer sizes are exported generically (`sfc{i}_w` / `head{i}_w` + counts), so
sa/policynet.py mirrors any depth without a code change.
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

ROOT = Path(__file__).resolve().parents[1]
for sub in ("src", "agents"):
    sys.path.insert(0, str(ROOT / sub))

from ptcg.env import sdk  # noqa: E402

sdk.load()

from sa.features import (A_GROUPS, DENSE_DIM, N_ATTR,  # noqa: E402
                         N_CARD_IDS, N_EXTRA, N_XSLOT, X_GROUPS)
from sa.optfeat import (OPT_DENSE, OPT_DENSE_V2, N_ATTACK_IDS,  # noqa: E402
                        pool_width)
from sa.routing import (NAME_TO_ROUTE, ROUTE_NAMES,  # noqa: E402
                        routes_from_corpus)

EMB = 16
SEL_DENSE = 14
BAGS = ("my_hand", "my_discard", "opp_discard")
# The four embedding tables and the id space each is indexed by. Used only by
# --vocab; see build_remap.
EMB_TABLES = ("slot_emb", "bag_emb", "card_emb", "atk_emb")
PAD_IX, UNK_IX = 0, 1
# The band the LB's top ~40 teams sit in; `val_top1@1120+` says how well the
# net fits STRONG demonstrators as opposed to the mixture (ROADMAP B7).
VAL_HI_RATING = 1120.0


def build_remap(vocab_path: Path) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    """Per-table id -> row map with a PAD row and a shared UNK row.

    The shipped tables are allocated over the RAW id space (1300 card ids, 1600
    attack ids) but the corpus only ever touches 104/134/135/57 of those rows.
    The other ~90% are exported at their random init -- harmless while they are
    never read, and NOT harmless at inference, where an out-of-vocabulary
    opponent card lands on a random unit-normal vector whose norm (3.91-3.95) is
    indistinguishable from a trained row's (3.97-4.07). The net cannot tell "a
    card I have never seen" from "a card I know", so it reads confident garbage.

    Remapping collapses each table to exactly the rows that got a gradient, and
    routes everything else to ONE row that is trained, by construction, to mean
    "unknown card". Row 0 is PAD (empty slot / no stadium / no effect); with
    `padding_idx=0` it is pinned at zero and takes no gradient, which is where
    the v5 net was heading on its own -- it drove |slot_emb[0]| to 2.337 against
    a 3.958 table mean, the 11th smallest of 1,300 rows, on 25.5% of lookups.

    ⚠ Per-table, not shared: a card seen in hand but never on an opponent's
    board is trained in `bag_emb` and untrained in `slot_emb`. One shared vocab
    would re-introduce the exact defect this removes, just for fewer rows.
    """
    from sa.features import N_CARD_IDS
    from sa.optfeat import N_ATTACK_IDS
    tabs = json.loads(vocab_path.read_text(encoding="utf-8"))["tables"]
    sizes = {"atk_emb": N_ATTACK_IDS}
    out: dict[str, tuple[np.ndarray, np.ndarray]] = {}
    for t in EMB_TABLES:
        if t not in tabs:
            raise SystemExit(f"{vocab_path} has no census for {t}")
        ids = np.array(sorted(int(k) for k in tabs[t] if int(k) != 0),
                       dtype=np.int64)
        size = sizes.get(t, N_CARD_IDS)
        if ids.size and int(ids[-1]) >= size:
            raise SystemExit(f"{t}: census id {int(ids[-1])} >= {size}; the "
                             "vocab was built against a different id space")
        lut = np.full(size, UNK_IX, dtype=np.int64)
        lut[PAD_IX] = PAD_IX
        lut[ids] = np.arange(2, 2 + ids.size, dtype=np.int64)
        out[t] = (ids, lut)
    return out


def apply_remap(data: "Data", remap: dict[str, tuple[np.ndarray, np.ndarray]]
                ) -> None:
    """Rewrite every id column in place. Ids at or past a table's raw size
    cannot appear -- features.py already clamps them to 0 -- but clip anyway so
    a corpus built by an older builder fails to UNK rather than IndexError."""
    def m(t: str, a: np.ndarray) -> np.ndarray:
        lut = remap[t][1]
        return lut[np.clip(a, 0, len(lut) - 1)]
    data.slots = m("slot_emb", data.slots)
    data.xslots = m("slot_emb", data.xslots)
    for nm in BAGS:
        data.bag_flat[nm] = m("bag_emb", data.bag_flat[nm])
    data.opt_card = m("card_emb", data.opt_card)
    data.opt_tgt = m("card_emb", data.opt_tgt)
    data.opt_atk = m("atk_emb", data.opt_atk)
    for t in EMB_TABLES:
        ids = remap[t][0]
        print(f"  {t:9s} {len(remap[t][1]):5d} raw ids -> {ids.size + 2:4d} "
              f"rows (PAD + UNK + {ids.size} seen)")


def load_init(model: PolicyNet, path: Path) -> None:
    """Warm-start from an exported .npz (the fine-tuning arm of B7). Refuses on
    any shape mismatch rather than partially loading -- a silently half-loaded
    net trains fine and measures like a fresh one."""
    z = np.load(path)
    with torch.no_grad():
        for name, emb in (("slot_emb", model.slot_emb), ("bag_emb",
                          model.bag_emb), ("card_emb", model.card_emb),
                         ("atk_emb", model.atk_emb)):
            w = z[name]
            if w.shape != tuple(emb.weight.shape):
                raise SystemExit(f"--init {path.name}: {name} is {w.shape}, "
                                 f"model wants {tuple(emb.weight.shape)}")
            emb.weight.copy_(torch.from_numpy(w))
        for prefix, seq in (("sfc", model.state_fc), ("head", model.head)):
            lins = [m for m in seq if isinstance(m, nn.Linear)]
            n = int(z[f"n_{prefix}"][0])
            if n != len(lins):
                raise SystemExit(f"--init {path.name}: {n} {prefix} layers, "
                                 f"model has {len(lins)}")
            for i, lin in enumerate(lins):
                w, b = z[f"{prefix}{i}_w"], z[f"{prefix}{i}_b"]
                if w.shape != tuple(lin.weight.shape):
                    raise SystemExit(
                        f"--init {path.name}: {prefix}{i}_w is {w.shape}, "
                        f"model wants {tuple(lin.weight.shape)}")
                lin.weight.copy_(torch.from_numpy(w))
                lin.bias.copy_(torch.from_numpy(b))
        # E1 auxiliary heads are append-only. A plain v5 checkpoint has no
        # auxiliary tensors, so warm-starting it deliberately leaves these
        # heads at their seeded initialization while loading the policy
        # byte-for-byte. A later multitask checkpoint restores them as well.
        for prefix, head in (("outcome", model.outcome_head),
                             ("count", model.count_head)):
            if head is None or f"{prefix}_w" not in z:
                continue
            w, b = z[f"{prefix}_w"], z[f"{prefix}_b"]
            if w.shape != tuple(head.weight.shape):
                raise SystemExit(f"--init {path.name}: {prefix}_w is {w.shape}, "
                                 f"model wants {tuple(head.weight.shape)}")
            head.weight.copy_(torch.from_numpy(w))
            head.bias.copy_(torch.from_numpy(b))
        # E2 adapters are append-only. A plain v5 checkpoint has none, so
        # warm-starting leaves the zero-initialized residuals in place.
        if model.adapters is not None and "adapter_names" in z:
            names = [str(x) for x in z["adapter_names"].tolist()]
            for name in names:
                if name not in model.adapters:
                    raise SystemExit(
                        f"--init {path.name}: unknown adapter {name!r}")
                seq = model.adapters[name]
                lins = [m for m in seq if isinstance(m, nn.Linear)]
                n = int(z[f"adapter_{name}_n"][0])
                if n != len(lins):
                    raise SystemExit(
                        f"--init {path.name}: adapter {name} has {n} layers, "
                        f"model has {len(lins)}")
                for i, lin in enumerate(lins):
                    w, b = (z[f"adapter_{name}{i}_w"],
                            z[f"adapter_{name}{i}_b"])
                    if w.shape != tuple(lin.weight.shape):
                        raise SystemExit(
                            f"--init {path.name}: adapter_{name}{i}_w is "
                            f"{w.shape}, model wants "
                            f"{tuple(lin.weight.shape)}")
                    lin.weight.copy_(torch.from_numpy(w))
                    lin.bias.copy_(torch.from_numpy(b))
    print(f"warm-started from {path}")


def _mlp(sizes: list[int], dropout: float, out_dim: int | None) -> nn.Sequential:
    """ReLU MLP over `sizes` hidden widths; `out_dim` appends a linear head."""
    layers: list[nn.Module] = []
    for a, b in zip(sizes[:-1], sizes[1:]):
        layers += [nn.Linear(a, b), nn.ReLU(), nn.Dropout(dropout)]
    if out_dim is not None:
        layers.append(nn.Linear(sizes[-1], out_dim))
    return nn.Sequential(*layers)


def _make_adapter(in_dim: int, hidden: int) -> nn.Sequential:
    """Residual logit MLP; final layer is zero-initialized for v5 equivalence."""
    seq = nn.Sequential(
        nn.Linear(in_dim, hidden),
        nn.ReLU(),
        nn.Linear(hidden, 1),
    )
    nn.init.zeros_(seq[-1].weight)
    nn.init.zeros_(seq[-1].bias)
    return seq


class PolicyNet(nn.Module):
    def __init__(self, state_h: tuple[int, ...] = (256,),
                 head_h: tuple[int, ...] = (128,), dropout: float = 0.1,
                 opt_cols: int = OPT_DENSE, extra: bool = True,
                 pool: bool = False, outcome: bool = False,
                 count: bool = False, adapter_names: list[str] | None = None,
                 adapter_h: int = 64, adapters_off: bool = False,
                 attr: bool = False,
                 rows: dict[str, int] | None = None, pad: bool = False):
        super().__init__()
        self.opt_cols = opt_cols
        self.extra = extra
        self.pool = pool
        self.adapters_off = adapters_off
        self.adapter_h = adapter_h
        self.attr = attr
        # `rows` shrinks each table to its own vocabulary (--vocab); absent, the
        # tables span the raw id space exactly as v3-v6 did. Only the ROW count
        # changes -- EMB is untouched -- so every downstream width, and hence
        # every exported layer shape, is identical to the control's.
        r = rows or {}
        pi = PAD_IX if pad else None
        self.slot_emb = nn.Embedding(r.get("slot_emb", N_CARD_IDS), EMB,
                                     padding_idx=pi)
        self.bag_emb = nn.EmbeddingBag(r.get("bag_emb", N_CARD_IDS), EMB,
                                       mode="mean", include_last_offset=True,
                                       padding_idx=pi)
        self.card_emb = nn.Embedding(r.get("card_emb", N_CARD_IDS), EMB,
                                     padding_idx=pi)
        self.atk_emb = nn.Embedding(r.get("atk_emb", N_ATTACK_IDS), EMB,
                                    padding_idx=pi)
        in_state = DENSE_DIM + 12 * EMB + len(BAGS) * EMB + SEL_DENSE
        if extra:                       # the v4 block, appended (features.py)
            in_state += N_EXTRA + N_XSLOT * EMB
        if pool:                        # the v5 block, appended (optfeat.py)
            in_state += pool_width(opt_cols, EMB)
        if attr:                        # the v6 block, appended (features.py)
            in_state += N_ATTR
        self.state_fc = _mlp([in_state, *state_h], dropout, None)
        in_head = state_h[-1] + opt_cols + 3 * EMB
        self.head = _mlp([in_head, *head_h], dropout, 1)
        # Constructed AFTER every policy parameter. Resetting the seed therefore
        # gives a control and an auxiliary treatment identical policy weights;
        # only the treatment consumes additional RNG after that point.
        self.outcome_head = (nn.Linear(state_h[-1], 1) if outcome else None)
        self.count_head = (nn.Linear(state_h[-1], 1) if count else None)
        # E2 adapters are also append-only and zero-initialized, so an untrained
        # treatment matches the frozen base logits exactly.
        self.adapter_names = list(adapter_names or [])
        self.adapter_route_ids: dict[str, int] = {}
        if self.adapter_names:
            unknown = [n for n in self.adapter_names if n not in NAME_TO_ROUTE
                       or NAME_TO_ROUTE[n] == 0]
            if unknown:
                raise SystemExit(
                    f"adapters must be non-general route names; got {unknown}")
            self.adapter_route_ids = {n: NAME_TO_ROUTE[n]
                                      for n in self.adapter_names}
            self.adapters = nn.ModuleDict({
                n: _make_adapter(in_head, adapter_h)
                for n in self.adapter_names
            })
        else:
            self.adapters = None

    def forward(self, dense, slots, bag_flat, bag_off, seld,
                opt_dense, opt_card, opt_atk, opt_tgt, opt_row,
                xdense=None, xslots=None, attrs=None, routes=None,
                return_state: bool = False):
        # The per-option encoding is built FIRST, because the v5 pool feeds it
        # into the state. It is the same tensor the head consumes below, so the
        # pool costs one reduction and no extra embedding lookups.
        # Slice to `opt_cols`. The v3 target block is APPENDED to the v2 layout,
        # so `--opt-cols 25` trains the exact v2-feature control on the identical
        # rows -- same games, same selects, same labels, only the features differ.
        # That is a cleaner control than comparing against the shipped net, which
        # also differs in corpus (2,810 games vs whatever is on disk now).
        oenc = torch.cat([opt_dense[:, :self.opt_cols],
                          self.card_emb(opt_card),
                          self.atk_emb(opt_atk),
                          self.card_emb(opt_tgt)], dim=1)     # (O, D)
        parts = [dense, self.slot_emb(slots).flatten(1)]
        for name in BAGS:
            parts.append(self.bag_emb(bag_flat[name], bag_off[name]))
        parts.append(seld)
        # v4 goes LAST so that `--no-extra` reproduces the v3 state vector
        # byte-for-byte on the identical rows -- the same control discipline as
        # `--opt-cols 25` for the option block.
        if self.extra:
            parts.append(xdense)
            parts.append(self.slot_emb(xslots).flatten(1))
        # ...and v5 goes after v4, so `pool=False` reproduces the v4 state
        # vector byte-for-byte. Same discipline, third generation.
        if self.pool:
            parts.append(self._pool(oenc, opt_row, dense.shape[0]))
        # ...and v6 goes after v5, so `attr=False` reproduces the v5 state
        # vector byte-for-byte. Same discipline, fourth generation.
        if self.attr:
            parts.append(attrs)
        srepr = self.state_fc(torch.cat(parts, dim=1))       # (B, H)
        per_opt = torch.cat([srepr[opt_row], oenc], dim=1)   # (O, ...)
        logits = self.head(per_opt).squeeze(1)               # (O,)
        if (self.adapters is not None and not self.adapters_off
                and routes is not None):
            residual = torch.zeros_like(logits)
            row_route = routes[opt_row]
            for name, route_id in self.adapter_route_ids.items():
                mask = row_route == route_id
                if mask.any():
                    residual[mask] = self.adapters[name](
                        per_opt[mask]).squeeze(1)
            logits = logits + residual
        return (logits, srepr) if return_state else logits

    def _pool(self, oenc: torch.Tensor, opt_row: torch.Tensor,
              n_rows: int) -> torch.Tensor:
        """Segment mean/max of the option encodings + two count scalars.

        A permutation-invariant summary of the option SET, which is the one
        thing an independently-scored option can never carry. Empty selects
        (none exist in the corpus, but the arena can produce one) pool to zero
        rather than to -inf."""
        d = oenc.shape[1]
        idx = opt_row.unsqueeze(1).expand(-1, d)
        cnt = torch.zeros(n_rows, device=oenc.device).index_add_(
            0, opt_row, torch.ones_like(opt_row, dtype=oenc.dtype))
        mean = torch.zeros(n_rows, d, device=oenc.device).index_add_(
            0, idx[:, 0], oenc) / cnt.clamp_min(1.0).unsqueeze(1)
        mx = torch.full((n_rows, d), -1e30, device=oenc.device).scatter_reduce(
            0, idx, oenc, reduce="amax", include_self=True)
        nz = (cnt > 0).unsqueeze(1)
        mx = torch.where(nz, mx, torch.zeros_like(mx))
        scal = torch.stack([cnt.clamp_max(40.0) / 40.0,
                            torch.log1p(cnt) / float(np.log(41.0))], dim=1)
        return torch.cat([mean, mx, scal], dim=1)


def parse_episode_span(spec: str) -> tuple[float, float]:
    """Parse `--episode-span START:END` into inclusive-exclusive fractions."""
    if ":" not in spec:
        raise SystemExit("--episode-span needs START:END (e.g. 0:0.5)")
    a, b = (s.strip() for s in spec.split(":", 1))
    try:
        start, end = float(a), float(b)
    except ValueError as exc:
        raise SystemExit(f"--episode-span {spec!r}: {exc}") from exc
    if not (0.0 <= start < end <= 1.0):
        raise SystemExit("--episode-span requires 0 <= START < END <= 1")
    return start, end


def episode_span_mask(gid: np.ndarray, start: float, end: float) -> np.ndarray:
    """True for rows in [floor(n*start), floor(n*end)) within each gid.

    Row order is the order they appear in `gid` (shard-concat chronological
    order from build_policy_dataset). Odd-length games put the middle row in
    the second half when start=0.5 (floor splits)."""
    keep = np.zeros(len(gid), dtype=bool)
    # First pass: counts per gid in appearance order, without sorting the
    # whole array (gids are not contiguous across day dirs).
    order: dict[int, list[int]] = {}
    for i, g in enumerate(gid.tolist()):
        order.setdefault(g, []).append(i)
    for idxs in order.values():
        n = len(idxs)
        lo = int(n * start)
        hi = int(n * end)
        for j in idxs[lo:hi]:
            keep[j] = True
    return keep


def listwise_loss(out: torch.Tensor, chosen: torch.Tensor,
                  opt_row: torch.Tensor, n_rows: int,
                  w: torch.Tensor | None = None) -> torch.Tensor:
    """Softmax cross-entropy within each select's option set, averaged over
    the chosen options of that select. This is the objective that matches
    inference: the agent ranks the options and takes the top k.

    `w` is an optional per-ROW weight (ROADMAP B7): the loss becomes a weighted
    mean, so a strong demonstrator's selects pull the mode further than a weak
    one's. Weights are normalised to mean 1 by the caller, which keeps the
    effective step size comparable to the unweighted control."""
    # log-softmax per row, computed with a segmented max for stability
    big = torch.full((n_rows,), -1e30, device=out.device)
    mx = big.scatter_reduce(0, opt_row, out, reduce="amax", include_self=True)
    ex = torch.exp(out - mx[opt_row])
    denom = torch.zeros(n_rows, device=out.device).index_add_(0, opt_row, ex)
    logp = out - mx[opt_row] - torch.log(denom + 1e-12)[opt_row]
    picked = torch.zeros(n_rows, device=out.device).index_add_(
        0, opt_row, logp * chosen)
    cnt = torch.zeros(n_rows, device=out.device).index_add_(0, opt_row, chosen)
    valid = cnt > 0
    per_row = -(picked[valid] / cnt[valid])
    if w is None:
        return per_row.mean()
    wv = w[valid]
    return (per_row * wv).sum() / wv.sum().clamp_min(1e-8)


def count_targets(seld: torch.Tensor, chosen: torch.Tensor,
                  opt_row: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Return target fraction and validity mask for variable-count selects.

    This is the row-level equivalent of `count_fraction_table`: the old table
    averages these targets within `(selectType, context)` buckets, while E1
    asks the shared state representation to predict each row separately.
    """
    n_rows = seld.shape[0]
    picked = torch.zeros(n_rows, dtype=chosen.dtype,
                         device=chosen.device).index_add_(0, opt_row, chosen)
    mn = seld[:, 11] * 5.0
    mx = seld[:, 12] * 5.0
    valid = mx > mn + 1e-6
    target = (picked - mn) / (mx - mn).clamp_min(1e-6)
    return target.clamp(0.0, 1.0), valid


class Data:
    def __init__(self, paths: list[Path], want_attr: bool = True):
        # ⚠ `want_attr=False` skips the v6 attribute block entirely instead of
        # materialising it. It is 276 float32 per row -- 27.6% of this object --
        # and `Model.forward` only reads it when the net was built with
        # `--attr`, so under the v5 recipe every one of those bytes is loaded,
        # copied per batch and moved to the device to be discarded. On the
        # 40.1M-row corpus that is 44.3 GB resident and ~89 GB of load peak for
        # nothing. Verified no-op: train loss identical to 4 dp with and
        # without. Default True so every existing caller is unchanged.
        sd, slots, seld, gid, won, rating = [], [], [], [], [], []
        xd, xs, at = [], [], []
        # B8. Present only in shards written by p26_selfplay_gen.py; a BC
        # corpus gets NaN, which is what `--advantage` refuses to weight.
        margin: list = []
        adv: list = []
        self.rows_per_path: list[int] = []
        od, oc, oa, ot, om = [], [], [], [], []
        self.opt_rows: list[tuple[int, int]] = []  # (start,end) per row
        # ⚠ Bags are kept FLAT (one array + one offset array per bag), exactly
        # as the shards store them. The previous version materialised one small
        # numpy array per row per bag -- 249k rows x 3 bags = ~750k objects --
        # and that allocation, not the model, is what OOM'd this 7.3 GB machine
        # on any net above ~1.5M params. Same semantics, ~1 GB less resident.
        bag_flats: dict[str, list] = {n: [] for n in BAGS}
        bag_lens: dict[str, list] = {n: [] for n in BAGS}
        base = 0
        for p in paths:
            z = np.load(p)
            n = len(z["gid"])
            sd.append(z["dense"])
            slots.append(z["slots"])
            seld.append(z["seld"])
            # Corpora built before day 12 have no v4 block; zeros keep them
            # loadable, and `--extra` on such a corpus is refused in main().
            xd.append(z["xdense"] if "xdense" in z
                      else np.zeros((n, N_EXTRA), dtype=np.float32))
            xs.append(z["xslots"] if "xslots" in z
                      else np.zeros((n, N_XSLOT), dtype=np.int32))
            # Same contract for the v6 block: corpora built before day 20 get
            # zeros so they stay loadable, and `--attr` on such a corpus is
            # refused in main() rather than silently training on nothing.
            at.append(z["attr"] if (want_attr and "attr" in z)
                      else np.zeros((n, N_ATTR if want_attr else 0),
                                    dtype=np.float32))
            gid.append(z["gid"])
            won.append(z["won"])
            self.rows_per_path.append(n)
            margin.append(z["margin"] if "margin" in z
                          else np.full(n, np.nan, dtype=np.float32))
            # E27: the per-decision TD residual written by p92_td_advantage.py.
            # NaN for every corpus built before it, which --advantage-col
            # refuses to weight rather than treating as zero.
            adv.append(z["adv"] if "adv" in z
                       else np.full(n, np.nan, dtype=np.float32))
            # Corpora built before `--ratings` have no per-row demonstrator.
            rating.append(z["rating"] if "rating" in z
                          else np.full(n, np.nan, dtype=np.float32))
            # Pre-v6 corpora store OPT_DENSE_V3 (37) cols; current builds store
            # OPT_DENSE (46). Pad the short layout with zeros so mixed --ds
            # unions concatenate; --opt-cols then slices the prefix it needs.
            od_arr = z["opt_dense"]
            w = int(od_arr.shape[1])
            if w < OPT_DENSE:
                od_arr = np.pad(od_arr, ((0, 0), (0, OPT_DENSE - w)))
            elif w > OPT_DENSE:
                od_arr = od_arr[:, :OPT_DENSE]
            od.append(od_arr)
            oc.append(z["opt_card"])
            oa.append(z["opt_attack"])
            ot.append(z["opt_target"] if "opt_target" in z
                      else np.zeros_like(z["opt_card"]))
            om.append(z["opt_chosen"])
            off = z["opt_off"]
            for i in range(n):
                self.opt_rows.append((base + off[i], base + off[i + 1]))
            base += off[-1]
            for nm in BAGS:
                flat = z[f"bag_{nm}_flat"]
                boff = z[f"bag_{nm}_off"]
                bag_flats[nm].append(flat.astype(np.int64, copy=False))
                bag_lens[nm].append(np.diff(boff).astype(np.int64))
        self.dense = np.concatenate(sd)
        self.slots = np.concatenate(slots).astype(np.int64)
        self.seld = np.concatenate(seld)
        self.xdense = np.concatenate(xd)
        self.xslots = np.concatenate(xs).astype(np.int64)
        self.attr = np.concatenate(at)
        self.has_extra = all("xdense" in np.load(p) for p in paths)
        self.has_attr = all("attr" in np.load(p) for p in paths)
        self.gid = np.concatenate(gid)
        self.won = np.concatenate(won)
        self.rating = np.concatenate(rating)
        self.margin = np.concatenate(margin)
        self.adv = np.concatenate(adv)
        self.w = np.ones(len(self.gid), dtype=np.float32)
        self.opt_dense = np.concatenate(od)
        self.opt_card = np.concatenate(oc).astype(np.int64)
        self.opt_atk = np.concatenate(oa).astype(np.int64)
        self.opt_tgt = np.concatenate(ot).astype(np.int64)
        self.opt_chosen = np.concatenate(om)
        # Flats concatenate in row order, so a cumsum over the per-row lengths
        # gives GLOBAL offsets across shards.
        self.bag_flat: dict[str, np.ndarray] = {}
        self.bag_off: dict[str, np.ndarray] = {}
        for nm in BAGS:
            lens = np.concatenate(bag_lens[nm])
            off = np.zeros(len(lens) + 1, dtype=np.int64)
            np.cumsum(lens, out=off[1:])
            self.bag_off[nm] = off
            self.bag_flat[nm] = (np.concatenate(bag_flats[nm]) if off[-1]
                                 else np.zeros(0, dtype=np.int64))
        self.n = len(self.gid)
        # E2 route labels from observable opponent slots + discard only.
        self.routes = routes_from_corpus(
            self.slots, self.bag_flat["opp_discard"],
            self.bag_off["opp_discard"])

    def batches(self, idx: np.ndarray, bs: int,
                rng: np.random.Generator | None):
        order = rng.permutation(idx) if rng is not None else idx
        for i in range(0, len(order), bs):
            sel = order[i:i + bs]
            bag_flat, bag_off = {}, {}
            for nm in BAGS:
                go = self.bag_off[nm]
                lens = go[sel + 1] - go[sel]
                off = np.zeros(len(sel) + 1, dtype=np.int64)
                np.cumsum(lens, out=off[1:])
                if off[-1]:
                    idx = np.concatenate([np.arange(go[k], go[k + 1])
                                          for k in sel if go[k + 1] > go[k]])
                    gathered = self.bag_flat[nm][idx]
                else:
                    gathered = np.zeros(0, dtype=np.int64)
                bag_flat[nm] = torch.from_numpy(gathered)
                bag_off[nm] = torch.from_numpy(off)
            spans = [self.opt_rows[k] for k in sel]
            opt_idx = np.concatenate([np.arange(a, b) for a, b in spans])
            opt_row = np.concatenate(
                [np.full(b - a, j) for j, (a, b) in enumerate(spans)])
            yield (torch.from_numpy(self.dense[sel]),
                   torch.from_numpy(self.slots[sel]),
                   bag_flat, bag_off,
                   torch.from_numpy(self.seld[sel]),
                   torch.from_numpy(self.opt_dense[opt_idx]),
                   torch.from_numpy(self.opt_card[opt_idx]),
                   torch.from_numpy(self.opt_atk[opt_idx]),
                   torch.from_numpy(self.opt_tgt[opt_idx]),
                   torch.from_numpy(opt_row),
                   torch.from_numpy(self.opt_chosen[opt_idx]),
                   spans,
                   torch.from_numpy(self.w[sel]),
                   sel,
                   torch.from_numpy(self.xdense[sel]),
                   torch.from_numpy(self.xslots[sel]),
                   torch.from_numpy(self.attr[sel]),
                   torch.from_numpy(self.routes[sel]))


def td_advantage_weights(data: "Data", is_rl: np.ndarray, beta: float,
                         anchor_w: float) -> np.ndarray:
    """E27: AWR over the PER-DECISION TD residual, not the game result.

    `w = exp(beta * A / sd(A))` on RL rows. **The normalisation by sd(A) is the
    part that has to be argued, and it is frozen in the pre-registration rather
    than tuned**: a TD residual has sd ~0.07 while B8's `won - baseline` has
    sd ~0.5, so passing the same beta to both would make this reweighting ~7x
    gentler than the one that already measured null (§8ao). Dividing by sd puts
    beta in units of "standard deviations of advantage", and **beta = 0.5
    reproduces B8's weight RATIO of e ~ 2.72 between a +1sd and a -1sd
    decision** -- so E27 differs from B8 in the SIGNAL, not in how hard it
    pushes. Tuning beta afterwards is the shopping B8 was denied.

    ⚠ Rows whose advantage is NaN (any corpus not passed through
    `p92_td_advantage.py`) are refused outright rather than silently weighted 1,
    because a half-annotated corpus would train mostly on unweighted rows and
    report a perfectly ordinary loss curve.
    """
    if not is_rl.any():
        raise SystemExit("--advantage-col needs RL shards from "
                         "p26_selfplay_gen.py; every row came from --anchor-ds")
    a = data.adv
    bad = int(np.isnan(a[is_rl]).sum())
    if bad:
        raise SystemExit(
            f"{bad:,} of {int(is_rl.sum()):,} RL rows have no `adv` column. "
            f"Run scripts/p92_td_advantage.py over the corpus first -- "
            f"training on a partly-annotated corpus is a silent null.")
    sd = float(np.std(a[is_rl]))
    if sd <= 0:
        raise SystemExit("advantage column has zero variance -- nothing to weight")
    w = np.ones(data.n, dtype=np.float32)
    z = (a[is_rl] - float(np.mean(a[is_rl]))) / sd
    # 🔴 Clip the STANDARDISED advantage at +/-2 sd, chosen on a property of
    # the data and not of any score (E25's discipline for picking tau). The
    # advantage distribution is a spike at zero plus a heavy tail -- the same
    # shape §8by found for rollout value -- so an unclipped exponent hands a
    # handful of outlier transitions enormous weight: at +/-4 the effective
    # sample size measured 50.8% against B8's 91.3%, i.e. half the corpus
    # thrown away to variance before any signal is asked for. At +/-2 the FULL
    # weight range is [e^-1, e^+1], which is exactly B8's win-row/loss-row
    # range end to end.
    w[is_rl] = np.exp(beta * np.clip(z, -2.0, 2.0))
    w[~is_rl] = anchor_w
    w = (w / w.mean()).astype(np.float32)
    print(f"--advantage-col {beta}: adv mean={np.mean(a[is_rl]):+.5f} "
          f"sd={sd:.4f}; weights [{w.min():.4f}, {w.max():.3f}]; "
          f"ratio(+1sd/-1sd)={np.exp(2 * beta):.2f}")
    print(f"  anchor rows {int((~is_rl).sum()):,} at {anchor_w}")
    ess = w[is_rl].sum() ** 2 / np.square(w[is_rl]).sum()
    print(f"  effective sample size {ess:,.0f} of {int(is_rl.sum()):,} RL rows "
          f"({ess / max(int(is_rl.sum()), 1):.1%})")
    return w


def advantage_weights(data: "Data", is_rl: np.ndarray, beta: float,
                      anchor_w: float, margin_max: float) -> np.ndarray:
    """B8: advantage-weighted regression over our OWN recorded outcomes.

    The weight on an RL row is `exp((won - baseline) / beta)`, so a select from
    a game we won is cloned harder than one from a game we lost. This is
    deliberately NOT a policy gradient with negative steps: an AWR weight is
    always positive, so the update can only ever re-weight behaviour the clone
    already produces. That is the property that makes it safe to run on a net
    worth ~942 on the ladder -- the downside is bounded by how far a reweighting
    can move it, not by an unbounded ascent direction.

    ⚠ **`--winners-only` is not this.** It scored 0.375 (§1) by filtering OTHER
    people's games and discarding half the corpus. Here nothing is discarded,
    the games are our own, and the losing rows still train -- at lower weight.
    The distinction is the whole reason B8 is not a repeat of that measurement.

    `anchor_w` is the weight held by corpus rows, which keeps the fine-tune
    tethered to the clone (rule: the thing being risked is a working agent).
    `margin_max` restricts the reweighting to selects where the net's top-1
    logit lead was small. §8u measured that agreement with the FIELD predicts
    strength, so a confident select is one we have a positive reason not to
    disturb; and an outcome signal can only change a decision that was close.
    Rows above the threshold fall back to weight 1 -- still trained, just not
    re-weighted.
    """
    if not is_rl.any():
        raise SystemExit("--advantage needs RL shards (a corpus built by "
                         "p26_selfplay_gen.py); every row came from --anchor-ds")
    won = data.won
    base = float(won[is_rl].mean())
    w = np.ones(data.n, dtype=np.float32)
    gate = is_rl.copy()
    if margin_max > 0:
        m = data.margin
        # NaN margins are BC rows; they are never gated here anyway.
        gate &= np.nan_to_num(m, nan=np.inf) <= margin_max
    w[gate] = np.exp((won[gate] - base) / max(beta, 1e-6))
    w[~is_rl] = anchor_w
    # Normalise over the rows that actually carry the objective, so the step
    # size stays comparable to the byte-identical control's (§8z's discipline).
    w = (w / w.mean()).astype(np.float32)
    n_gate = int(gate.sum())
    print(f"--advantage {beta}: baseline won={base:.4f}; "
          f"{n_gate:,} of {int(is_rl.sum()):,} RL rows re-weighted "
          f"({n_gate / max(int(is_rl.sum()), 1):.1%}"
          + (f", margin<={margin_max}" if margin_max > 0 else "") + ")")
    print(f"  weights [{w.min():.4f}, {w.max():.3f}]; "
          f"anchor rows {int((~is_rl).sum()):,} at {anchor_w}")
    ess = w[is_rl].sum() ** 2 / np.square(w[is_rl]).sum()
    print(f"  effective sample size {ess:,.0f} of {int(is_rl.sum()):,} RL rows "
          f"({ess / max(int(is_rl.sum()), 1):.1%})")
    return w


def apply_freeze(model: "PolicyNet", spec: str) -> None:
    """Train only the named top-level parameter groups; freeze the rest.

    B8's pre-registered form is "fine-tune a SMALL parameter set", and the
    reason is §8w: 8.2x the parameters bought -43 decisions, so capacity is not
    what is missing and a full-net update on a much smaller, much noisier
    corpus is the way to lose the clone rather than improve it.
    """
    keep = {s.strip() for s in spec.split(",") if s.strip()}
    known = {n.split(".", 1)[0] for n, _ in model.named_parameters()}
    unknown = keep - known
    if unknown:
        raise SystemExit(f"--freeze-except names {sorted(unknown)}; "
                         f"parameter groups are {sorted(known)}")
    n_train = n_frozen = 0
    for name, p in model.named_parameters():
        if name.split(".", 1)[0] in keep:
            n_train += p.numel()
        else:
            p.requires_grad_(False)
            n_frozen += p.numel()
    print(f"--freeze-except {sorted(keep)}: training {n_train:,} params, "
          f"froze {n_frozen:,} ({n_train / max(n_train + n_frozen, 1):.1%} live)")


class StreamData:
    """Shard-at-a-time corpus, for a corpus that does not fit in RAM.

    `Data` concatenates everything: ~4.0 KB/row resident and ~7.8 KB/row at the
    load peak, which is 160 GB / 313 GB on the 40.1M-row host corpus against a
    34 GB box. This holds only the SMALL per-row columns globally --
    gid/won/rating/margin/adv/w/routes, ~40 B/row, so ~1.6 GB at 40M rows -- and
    loads the big ones (dense, opt_*, bags) one BUFFER of shards at a time.

    It exposes the same surface `Data` does, so nothing in the training loop
    changes and `--stream` is purely additive.

    ⚠ THE ONE REAL DIFFERENCE, AND IT MUST BE DECLARED IN ANY A/B: batches are
    drawn from a buffer of `--stream-buffer` shards, not from the whole corpus.
    Shard order is reshuffled every epoch and rows are shuffled across the
    buffer, which is the standard approximation, but it is NOT the global
    shuffle the in-RAM path does. A net trained with --stream and one trained
    without differ in more than the corpus.
    """

    def __init__(self, paths: list[Path], buffer: int = 4,
                 want_attr: bool = False):
        self.paths = list(paths)
        self.buffer = max(1, buffer)
        self.want_attr = want_attr
        gid, won, rating, margin, adv, routes = [], [], [], [], [], []
        self.rows_per_path: list[int] = []
        self.has_extra = True
        self.has_attr = True
        t0 = time.time()
        for i, p in enumerate(self.paths):
            with np.load(p) as z:
                n = len(z["gid"])
                gid.append(z["gid"])
                won.append(z["won"])
                nan = np.full(n, np.nan, dtype=np.float32)
                rating.append(z["rating"] if "rating" in z else nan)
                margin.append(z["margin"] if "margin" in z else nan)
                adv.append(z["adv"] if "adv" in z else nan)
                self.rows_per_path.append(n)
                self.has_extra &= "xdense" in z
                self.has_attr &= "attr" in z
                # Routes are per-row ints derived from slots + the opponent's
                # discard bag. Computed HERE, once, because the training loop
                # reads `data.routes` outside of any batch.
                routes.append(routes_from_corpus(
                    z["slots"].astype(np.int64),
                    z["bag_opp_discard_flat"].astype(np.int64),
                    z["bag_opp_discard_off"]))
            if (i + 1) % 100 == 0:
                print(f"  scanned {i + 1}/{len(self.paths)} shards "
                      f"({time.time() - t0:.0f}s)", flush=True)
        self.gid = np.concatenate(gid)
        self.won = np.concatenate(won)
        self.rating = np.concatenate(rating)
        self.margin = np.concatenate(margin)
        self.adv = np.concatenate(adv)
        self.routes = np.concatenate(routes)
        self.n = len(self.gid)
        self.w = np.ones(self.n, dtype=np.float32)
        off = np.zeros(len(self.paths) + 1, dtype=np.int64)
        np.cumsum(self.rows_per_path, out=off[1:])
        self._off = off
        print(f"--stream: {len(self.paths)} shards, {self.n:,} rows indexed in "
              f"{time.time() - t0:.0f}s; big columns stay on disk "
              f"(buffer={self.buffer} shards)", flush=True)

    # `Data` materialises these; nothing in the loop reads them when streaming,
    # so fail loudly rather than return something silently wrong.
    def _refuse(self, what: str):
        raise SystemExit(f"--stream does not support {what}; it needs the whole "
                         f"corpus resident, which is the thing --stream exists "
                         f"to avoid")

    @property
    def attr(self):
        self._refuse("--drop-a / whole-corpus attr access")

    def _groups(self, idx: np.ndarray):
        """Global row indices -> [(shard, its rows)], in shard order."""
        sh = np.searchsorted(self._off, idx, side="right") - 1
        order = np.argsort(sh, kind="stable")
        idx_s, sh_s = idx[order], sh[order]
        bounds = np.searchsorted(sh_s, np.arange(len(self.paths) + 1))
        return [(k, idx_s[bounds[k]:bounds[k + 1]])
                for k in range(len(self.paths)) if bounds[k + 1] > bounds[k]]

    def batches(self, idx: np.ndarray, bs: int,
                rng: "np.random.Generator | None"):
        groups = self._groups(np.asarray(idx))
        gorder = (rng.permutation(len(groups)) if rng is not None
                  else np.arange(len(groups)))
        for i in range(0, len(gorder), self.buffer):
            picks = [groups[g] for g in gorder[i:i + self.buffer]]
            sub = Data([self.paths[k] for k, _ in picks],
                       want_attr=self.want_attr)
            # Global row ids of every row in the loaded shards, in load order.
            gids = np.concatenate([np.arange(self._off[k], self._off[k + 1])
                                   for k, _ in picks])
            # Carry the globally-computed per-row columns onto the sub-corpus so
            # weighting and route accounting stay identical to the in-RAM path.
            sub.w = self.w[gids]
            sub.rating = self.rating[gids]
            sub.routes = self.routes[gids]
            # Selected rows, remapped global -> local within the loaded buffer.
            local, base = [], 0
            for k, rows in picks:
                local.append(rows - self._off[k] + base)
                base += self.rows_per_path[k]
            sel = np.concatenate(local)
            for b in sub.batches(sel, bs, rng):
                # ⚠ The loop indexes `data.rating[vsel]` with the yielded sel,
                # so it MUST come back out as global ids, not buffer-local ones.
                b = list(b)
                b[13] = gids[b[13]]
                yield tuple(b)
            del sub


def count_fraction_table_stream(data: "StreamData", idx: np.ndarray
                                ) -> np.ndarray:
    """count_fraction_table over a streamed corpus: same arithmetic, one shard
    of `seld`/`opt_chosen` resident at a time."""
    num = np.zeros((11, 64))
    den = np.zeros((11, 64))
    for k, rows in data._groups(np.asarray(idx)):
        local = rows - data._off[k]
        with np.load(data.paths[k]) as z:
            seld_all = z["seld"]
            off = z["opt_off"]
            chosen = z["opt_chosen"]
        for j in local:
            seld = seld_all[j]
            t = int(np.argmax(seld[:11]))
            ctx = min(int(round(seld[13] * 50.0)), 63)
            mn = int(round(seld[11] * 5.0))
            mx = int(round(seld[12] * 5.0))
            if mx <= mn:
                continue
            c = float(chosen[off[j]:off[j + 1]].sum())
            num[t, ctx] += (c - mn) / (mx - mn)
            den[t, ctx] += 1.0
    frac = np.where(den > 0, num / np.maximum(den, 1), 1.0)
    return frac.astype(np.float32)


def count_fraction_table(data: "Data", idx: np.ndarray) -> np.ndarray:
    """(11, 64) mean of (chosen-min)/(max-min) per (selectType, context) over
    variable-count selects. Unseen cells default to 1.0 (take the max), which
    matches top play for searches/benching."""
    num = np.zeros((11, 64))
    den = np.zeros((11, 64))
    for k in idx:
        a, b = data.opt_rows[k]
        seld = data.seld[k]
        t = int(np.argmax(seld[:11]))
        ctx = int(round(seld[13] * 50.0))
        ctx = min(ctx, 63)
        mn = int(round(seld[11] * 5.0))
        mx = int(round(seld[12] * 5.0))
        if mx <= mn:
            continue
        chosen = float(data.opt_chosen[a:b].sum())
        num[t, ctx] += (chosen - mn) / (mx - mn)
        den[t, ctx] += 1.0
    frac = np.where(den > 0, num / np.maximum(den, 1), 1.0)
    return frac.astype(np.float32)


def apply_x_drop(data: "Data", names: list[str]) -> np.ndarray:
    """Zero the named members of the v4 state block and return the mask.

    Zeroing rather than deleting keeps the layer widths, the parameter count and
    the weight init identical to the full-block run, so a drop-one arm differs
    from it in the CONTENT of a few columns and nothing else. An xslot set to 0
    is the same "no card" row the embedding already uses for an absent stadium.
    """
    mask = np.ones(N_EXTRA + N_XSLOT, dtype=np.float32)
    for nm in names:
        if nm not in X_GROUPS:
            raise SystemExit(f"--drop-x {nm}: known groups are "
                             f"{', '.join(X_GROUPS)}")
        for i in X_GROUPS[nm]:
            mask[i] = 0.0
    data.xdense = data.xdense * mask[:N_EXTRA]
    data.xslots = np.where(mask[N_EXTRA:] > 0, data.xslots, 0)
    print(f"--drop-x {','.join(names)}: zeroed xdense cols "
          f"{[i for i in range(N_EXTRA) if mask[i] == 0]} and xslot cols "
          f"{[i for i in range(N_XSLOT) if mask[N_EXTRA + i] == 0]}")
    return mask


def apply_a_drop(data: "Data", names: list[str]) -> np.ndarray:
    """Zero the named members of the v6 attribute block and return the mask.

    Same discipline as `apply_x_drop`: the block ships whole, so without a
    drop-one arm nothing would say WHICH of energyType / weakness / ability /
    resist / weakHit paid for the result. Zeroing keeps widths, parameter count
    and init identical, so an arm differs only in column content.
    """
    mask = np.ones(N_ATTR, dtype=np.float32)
    for nm in names:
        if nm not in A_GROUPS:
            raise SystemExit(f"--drop-a {nm}: known groups are "
                             f"{', '.join(A_GROUPS)}")
        for i in A_GROUPS[nm]:
            mask[i] = 0.0
    data.attr = data.attr * mask
    print(f"--drop-a {','.join(names)}: zeroed {int((mask == 0).sum())} of "
          f"{N_ATTR} attr columns")
    return mask


def export_npz(model: PolicyNet, path: Path, count_frac: np.ndarray,
               x_mask: np.ndarray | None = None,
               a_mask: np.ndarray | None = None,
               remap: dict[str, tuple[np.ndarray, np.ndarray]] | None = None):
    """Export every Linear generically, so inference mirrors any depth."""
    def arr(t: torch.Tensor) -> np.ndarray:
        return t.detach().cpu().numpy()

    out: dict[str, np.ndarray] = {
        "slot_emb": arr(model.slot_emb.weight),
        "bag_emb": arr(model.bag_emb.weight),
        "card_emb": arr(model.card_emb.weight),
        "atk_emb": arr(model.atk_emb.weight),
        "count_frac": count_frac,
        # Width of the v5 pooled block, 0 if the net has none. Inference cannot
        # derive this from `state_in` alone (the v4 and v5 widths are both
        # legal), so it is recorded explicitly. Nets exported before day 13 have
        # no such key and are read as 0.
        "n_pool": np.array([pool_width(model.opt_cols, EMB) if model.pool
                            else 0], dtype=np.int64),
        # Width of the v6 attribute block, 0 if the net has none. Same reason as
        # n_pool: `state_in` alone no longer identifies the layout once three
        # optional blocks exist. Nets exported before day 20 lack this key and
        # are read as 0.
        "n_attr": np.array([N_ATTR if model.attr else 0], dtype=np.int64),
    }
    if remap is not None:
        # The raw ids this net's rows stand for, in row order after PAD and UNK.
        # Inference rebuilds the lookup from these, so the map can never drift
        # from the tables it was trained with -- they travel in one file.
        for t in EMB_TABLES:
            out[f"vocab_{t}"] = remap[t][0].astype(np.int64)
    if a_mask is not None:
        out["a_mask"] = a_mask
    if x_mask is not None:
        # Which members of the v4 block this net was actually shown. Inference
        # applies it, so an ablation arm can never be fed a column it never saw.
        out["x_mask"] = x_mask
    for prefix, seq in (("sfc", model.state_fc), ("head", model.head)):
        n = 0
        for mod in seq:
            if isinstance(mod, nn.Linear):
                out[f"{prefix}{n}_w"] = arr(mod.weight)
                out[f"{prefix}{n}_b"] = arr(mod.bias)
                n += 1
        out[f"n_{prefix}"] = np.array([n], dtype=np.int64)
    for prefix, head in (("outcome", model.outcome_head),
                         ("count", model.count_head)):
        if head is not None:
            out[f"{prefix}_w"] = arr(head.weight)
            out[f"{prefix}_b"] = arr(head.bias)
    if model.adapters is not None:
        out["adapter_names"] = np.asarray(model.adapter_names)
        out["adapter_h"] = np.array([model.adapter_h], dtype=np.int64)
        out["adapter_route_ids"] = np.asarray(
            [model.adapter_route_ids[n] for n in model.adapter_names],
            dtype=np.int64)
        for name, seq in model.adapters.items():
            n = 0
            for mod in seq:
                if isinstance(mod, nn.Linear):
                    out[f"adapter_{name}{n}_w"] = arr(mod.weight)
                    out[f"adapter_{name}{n}_b"] = arr(mod.bias)
                    n += 1
            out[f"adapter_{name}_n"] = np.array([n], dtype=np.int64)
    np.savez_compressed(path, **out)
    print(f"exported -> {path}")


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--ds", default="artifacts/pds",
                    help="shard dir; comma-separated for several")
    ap.add_argument("--anchor-ds", default="",
                    help="B8: corpus shard dir(s) mixed in as the ANCHOR term, "
                         "so the fine-tune cannot drift off the clone it "
                         "started from. Rows from here are never "
                         "advantage-weighted.")
    ap.add_argument("--advantage-col", type=float, default=0.0,
                    help="E27: AWR beta over the per-decision TD residual "
                         "written by p92_td_advantage.py, in units of sd(adv). "
                         "0.5 reproduces B8's weight ratio (e ~ 2.72).")
    ap.add_argument("--advantage", type=float, default=0.0,
                    help="B8: AWR temperature. Weight = exp((won-baseline)/B) "
                         "on --ds rows. 0 disables (plain cloning).")
    ap.add_argument("--anchor-w", type=float, default=1.0,
                    help="weight held by --anchor-ds rows")
    ap.add_argument("--primary-mass", type=float, default=0.0,
                    help="E3: target fraction of total supervised loss assigned "
                         "to --ds rows, with --anchor-ds supplying the remaining "
                         "mass. For example 0.1 gives curated DAgger labels 10%% "
                         "and the frozen BC corpus 90%%. 0 disables.")
    ap.add_argument("--margin-max", type=float, default=0.0,
                    help="B8: only re-weight selects whose top1-top2 logit "
                         "margin was <= this. 0 = re-weight every RL row.")
    ap.add_argument("--export-last", action="store_true",
                    help="export the FINAL epoch instead of the best-val one. "
                         "Required on both arms of any A/B where one arm's "
                         "objective is not corpus fit (rule 3) -- otherwise "
                         "the arms export different epochs and the comparison "
                         "is confounded by training length.")
    ap.add_argument("--freeze-except", default="",
                    help="B8: comma-separated top-level parameter groups to "
                         "train; everything else is frozen (e.g. 'head')")
    ap.add_argument("--epochs", type=int, default=4)
    ap.add_argument("--keep-gids", default="",
                    help="file of episode ids (one per line, or a CSV whose "
                         "first column is the id) -- keep only rows whose gid "
                         "is in it. `gid` IS the episode id, so this is how a "
                         "rating cut or an archetype filter is applied to an "
                         "already-built corpus WITHOUT rebuilding it.")
    ap.add_argument("--stream", action="store_true",
                    help="load shards a buffer at a time instead of "
                         "concatenating the corpus. Required above ~4M rows. "
                         "⚠ shuffling becomes buffer-local; declare it.")
    ap.add_argument("--stream-buffer", type=int, default=4,
                    help="shards held resident per buffer under --stream")
    ap.add_argument("--max-hours", type=float, default=0.0,
                    help="stop after the first epoch that finishes past this "
                         "many hours and exit 0. For hosted runs with a hard "
                         "wall-clock cap that discard output on kill.")
    ap.add_argument("--bs", type=int, default=1024)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=1e-5)
    ap.add_argument("--winners-only", action="store_true")
    ap.add_argument("--episode-span", default="",
                    help="keep only a chronological fraction of each game's "
                         "rows: START:END in [0,1] (e.g. 0:0.5 = first half, "
                         "0.5:1 = last half). Split is by decision-row count "
                         "per gid, in shard-concat order. Empty = full episode.")
    ap.add_argument("--loss", choices=("bce", "listwise", "both"),
                    default="listwise")
    ap.add_argument("--state-h", default="256",
                    help="comma-separated hidden widths for the state MLP")
    ap.add_argument("--head-h", default="128",
                    help="comma-separated hidden widths for the scoring MLP")
    ap.add_argument("--dropout", type=float, default=0.1)
    ap.add_argument("--device", choices=("cpu", "cuda"), default="cpu",
                    help="training device. Default cpu preserves historical "
                         "recipes; use cuda for the private E1 GPU sweep.")
    ap.add_argument("--aux-outcome-w", type=float, default=0.0,
                    help="E1: weight of win/loss BCE on the shared state "
                         "representation. 0 disables the outcome head.")
    ap.add_argument("--aux-count-w", type=float, default=0.0,
                    help="E1: weight of soft-label BCE for the selected-count "
                         "fraction on variable-count rows. 0 disables the head.")
    ap.add_argument("--out", default="agents/sa/policy_net.npz")
    ap.add_argument("--rating-temp", type=float, default=0.0,
                    help="ROADMAP B7: weight each row by "
                         "exp((rating - max) / T), normalised to mean 1. Small "
                         "T = clone the best demonstrators only; 0 (default) = "
                         "uniform, the standing control. Needs a corpus built "
                         "with `--ratings`.")
    ap.add_argument("--rating-min", type=float, default=0.0,
                    help="drop rows whose demonstrator is below this LB score")
    ap.add_argument("--init",
                    help="warm-start from an exported .npz (fine-tuning). "
                         "Shapes must match the arch flags.")
    ap.add_argument("--opt-cols", type=int, default=OPT_DENSE,
                    help="per-option feature columns to use. Default = all "
                         f"({OPT_DENSE}). Pass {OPT_DENSE_V2} to train the "
                         "v2-feature CONTROL on identical rows (ROADMAP B1).")
    ap.add_argument("--seed", type=int, default=0,
                    help="torch/numpy seed. Vary it to SIZE run-to-run "
                         "variance, which is the confound behind every "
                         "net-vs-net A/B in this repo (§8z).")
    ap.add_argument("--drop-x", default="",
                    help="comma-separated members of the v4 state block to "
                         "ABLATE (features.X_GROUPS: "
                         f"{','.join(X_GROUPS)}). The surviving mask is stored "
                         "in the npz and applied at inference.")
    ap.add_argument("--attr", action="store_true",
                    help="the v6 block: append per-slot CARD ATTRIBUTES "
                         "(energyType, weakness, ability, resistance, "
                         "weak-to-facing-type) to the STATE vector. These come "
                         "from the card DB, which covers all 1,267 cards, so "
                         "unlike an embedding row they transfer to cards the "
                         "corpus never contained (E6). Default off = the v5 "
                         "state vector, byte-for-byte, on identical rows.")
    ap.add_argument("--drop-a", default="",
                    help="comma-separated members of the v6 attribute block to "
                         "ABLATE (features.A_GROUPS: "
                         f"{','.join(A_GROUPS)}). The surviving mask is stored "
                         "in the npz and applied at inference.")
    ap.add_argument("--vocab", default="",
                    help="the v7 block: an out/emb/vocab.json census. Collapses "
                         "each embedding table to the rows the corpus actually "
                         "trained, with row 0 = PAD and row 1 = a shared UNK "
                         "that every out-of-vocabulary card routes to. Implies "
                         "--pad. Default off = the v3-v6 raw id space, i.e. an "
                         "unseen card reads a random untrained row.")
    ap.add_argument("--pad", action="store_true",
                    help="pin embedding row 0 to zero and give it no gradient "
                         "(padding_idx). Alone, this is the v7 block's SECOND "
                         "half only -- the arm that isolates 'id 0 is "
                         "overloaded' from 'unseen cards read noise'.")
    ap.add_argument("--pool", action="store_true",
                    help="the v5 block: append a mean/max pool of the option "
                         "encodings + two count scalars to the STATE vector "
                         "(optfeat.pool_width). Default off = the v4 state "
                         "vector byte-for-byte, i.e. the control.")
    ap.add_argument("--no-extra", action="store_true",
                    help="ignore the v4 state block (features.extra_feats). "
                         "This is the day-12 CONTROL: identical rows, "
                         "identical recipe, the v3 state vector byte-for-byte.")
    ap.add_argument("--adapters", default="",
                    help="E2: comma-separated residual adapter names "
                         "(mirror,alakazam). Append-only; zero-initialized so "
                         "an untrained treatment matches the frozen base.")
    ap.add_argument("--adapter-h", type=int, default=64,
                    help="E2: hidden width of each residual adapter MLP")
    ap.add_argument("--adapters-off", action="store_true",
                    help="E2 control: keep adapters in the checkpoint but do "
                         "not add their residual during forward/training")
    args = ap.parse_args()
    adapter_names = [s.strip() for s in args.adapters.split(",") if s.strip()]
    if not 1 <= args.opt_cols <= OPT_DENSE:
        raise SystemExit(f"--opt-cols must be in 1..{OPT_DENSE}")
    if args.aux_outcome_w < 0 or args.aux_count_w < 0:
        raise SystemExit("auxiliary loss weights must be non-negative")
    if args.adapter_h < 1:
        raise SystemExit("--adapter-h must be positive")
    if args.adapters_off and not adapter_names:
        raise SystemExit("--adapters-off requires --adapters")
    if not 0.0 <= args.primary_mass < 1.0:
        raise SystemExit("--primary-mass must be in [0, 1)")
    if args.primary_mass > 0 and not args.anchor_ds:
        raise SystemExit("--primary-mass needs --anchor-ds; otherwise there is "
                         "no anchor mass to preserve")
    if args.primary_mass > 0 and (args.advantage > 0 or args.rating_temp > 0):
        raise SystemExit("--primary-mass, --advantage, and --rating-temp each "
                         "define row weights; choose one")
    if (args.aux_outcome_w > 0 or args.aux_count_w > 0) and not args.export_last:
        raise SystemExit("E1 auxiliary treatments require --export-last so "
                         "control and treatment export the same epoch")
    if adapter_names and not args.export_last:
        raise SystemExit("E2 adapter arms require --export-last so control "
                         "and treatment export the same epoch")

    device = torch.device(args.device)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise SystemExit("--device cuda requested but torch.cuda.is_available() "
                         "is false")
    if device.type == "cpu":
        torch.set_num_threads(max(1, torch.get_num_threads() - 1))
    # Seeded so that a control/treatment pair (e.g. --opt-cols 25 vs 37, ROADMAP
    # B1) differs in its FEATURES and not in dropout masks or batch order. Weight
    # init still differs where the layer widths differ, which cannot be avoided.
    torch.manual_seed(args.seed)
    paths: list[Path] = []
    for d in args.ds.split(","):
        d = d.strip()
        if not d:
            continue
        got = sorted((ROOT / d).rglob("shard_*.npz"))
        if not got:
            raise SystemExit(f"no shards under {ROOT / d}")
        paths += got
    n_primary = len(paths)
    for d in args.anchor_ds.split(","):
        d = d.strip()
        if not d:
            continue
        got = sorted((ROOT / d).rglob("shard_*.npz"))
        if not got:
            raise SystemExit(f"no shards under {ROOT / d}")
        paths += got
    if args.stream:
        # ⛔ Everything below needs whole-corpus arrays. Refusing is the point:
        # a silently-ignored ablation flag would produce a net that is not the
        # arm it claims to be.
        for flag, name in ((args.drop_x, "--drop-x"), (args.drop_a, "--drop-a"),
                           (args.vocab, "--vocab"),
                           (args.advantage, "--advantage"),
                           (args.advantage_col, "--advantage-col"),
                           (args.primary_mass, "--primary-mass"),
                           (args.rating_temp, "--rating-temp"),
                           (args.anchor_ds, "--anchor-ds")):
            if flag:
                raise SystemExit(f"--stream does not support {name} yet")
        data = StreamData(paths, buffer=args.stream_buffer,
                          want_attr=args.attr)
    else:
        data = Data(paths, want_attr=args.attr)
    # Which rows came from --ds rather than --anchor-ds. Built from the
    # per-path row counts Data records, so it cannot drift from the actual
    # concatenation order.
    is_rl = np.zeros(data.n, dtype=bool)
    is_rl[:sum(data.rows_per_path[:n_primary])] = True
    if args.anchor_ds:
        print(f"--anchor-ds: {int(is_rl.sum()):,} primary rows + "
              f"{int((~is_rl).sum()):,} anchor rows")
    remap = None
    rows = None
    if args.vocab:
        vp = ROOT / args.vocab
        if not vp.exists():
            raise SystemExit(f"{vp} missing -- run scripts/p53_emb_vocab.py")
        remap = build_remap(vp)
        print(f"--vocab {args.vocab}:")
        apply_remap(data, remap)
        rows = {t: int(remap[t][0].size) + 2 for t in EMB_TABLES}
        args.pad = True
    x_mask = None
    if args.drop_x:
        if args.no_extra:
            raise SystemExit("--drop-x ablates the v4 block; --no-extra "
                             "already removes all of it")
        x_mask = apply_x_drop(data, [s.strip() for s in args.drop_x.split(",")
                                     if s.strip()])
    a_mask = None
    if args.drop_a:
        a_mask = apply_a_drop(data, [s.strip() for s in args.drop_a.split(",")
                                     if s.strip()])
    keep = np.ones(data.n, dtype=bool)
    if args.keep_gids:
        kp = Path(args.keep_gids)
        if not kp.is_absolute():
            kp = ROOT / kp
        want = set()
        for ln in kp.read_text(encoding="utf-8-sig").splitlines():
            tok = ln.split(",", 1)[0].strip()
            if tok and tok.lstrip("-").isdigit():
                want.add(int(tok))
        if not want:
            raise SystemExit(f"{kp}: parsed zero episode ids")
        sel = np.isin(data.gid, np.fromiter(want, dtype=np.int64, count=len(want)))
        keep &= sel
        # ⚠ rule 9: a filter that matches almost nothing must not look like a
        # small corpus. Say what fraction survived, in games as well as rows.
        print(f"--keep-gids {kp.name}: {len(want):,} ids -> "
              f"{int(sel.sum()):,} of {data.n:,} rows kept "
              f"({int(sel.sum())/max(data.n,1):.1%}), "
              f"{len(np.unique(data.gid[sel])):,} episodes matched")
        if not sel.any():
            raise SystemExit("--keep-gids matched ZERO rows; the id space is "
                             "wrong (gid is the episode id)")
    if args.episode_span:
        start, end = parse_episode_span(args.episode_span)
        span = episode_span_mask(data.gid, start, end)
        keep &= span
        print(f"--episode-span {start:g}:{end:g}: {int(span.sum())} of "
              f"{data.n} rows kept ({int(span.sum()) / max(data.n, 1):.1%} "
              f"by decision count per gid)")
    if args.winners_only:
        keep &= data.won > 0.5
    rated = ~np.isnan(data.rating)
    if args.rating_min > 0:
        keep &= rated & (data.rating >= args.rating_min)
        print(f"--rating-min {args.rating_min}: {int(keep.sum())} of {data.n} "
              f"rows kept")
    if args.rating_temp > 0:
        if not rated.any():
            raise SystemExit("--rating-temp needs a corpus built with "
                             "`--ratings`; every row's rating is NaN")
        # An unrated demonstrator gets the MEDIAN rating's weight rather than
        # 0 or 1: dropping them silently changes the corpus, and weighting them
        # 1.0 would make the unknown teams the most-cloned ones once the
        # exponential pushes everyone else down.
        r = np.where(rated, data.rating, np.nanmedian(data.rating))
        w = np.exp((r - np.nanmax(data.rating)) / args.rating_temp)
        w = (w / w[keep].mean()).astype(np.float32)
        data.w = w
        ess = w[keep].sum() ** 2 / np.square(w[keep]).sum()
        print(f"--rating-temp {args.rating_temp}: weights "
              f"[{w[keep].min():.4f}, {w[keep].max():.3f}], "
              f"effective sample size {ess:,.0f} of {int(keep.sum()):,} rows "
              f"({ess / max(int(keep.sum()), 1):.1%})")
        print(f"  {int((~rated & keep).sum())} unrated rows held at the median "
              f"rating ({np.nanmedian(data.rating):.1f})")
    if args.advantage_col > 0:
        if args.advantage > 0:
            raise SystemExit("--advantage-col and --advantage both set data.w; "
                             "pick the terminal-outcome signal or the TD one")
        if args.rating_temp > 0:
            raise SystemExit("--advantage-col and --rating-temp both set data.w")
        data.w = td_advantage_weights(data, is_rl, args.advantage_col,
                                      args.anchor_w)
    if args.advantage > 0:
        if args.rating_temp > 0:
            raise SystemExit("--advantage and --rating-temp both set data.w; "
                             "pick one")
        if args.winners_only:
            # §1's 0.375 result IS --winners-only. Running both would produce a
            # net that is neither intervention and would be reported as B8.
            raise SystemExit("--advantage subsumes --winners-only (it keeps "
                             "the losing rows at lower weight); refusing both")
        data.w = advantage_weights(data, is_rl, args.advantage,
                                   args.anchor_w, args.margin_max)
    if args.primary_mass > 0:
        n_primary_rows = int(is_rl.sum())
        n_anchor_rows = int((~is_rl).sum())
        if not n_primary_rows or not n_anchor_rows:
            raise SystemExit("--primary-mass needs non-empty primary and anchor "
                             "datasets")
        primary_w = (args.primary_mass / (1.0 - args.primary_mass)
                     * n_anchor_rows / n_primary_rows)
        data.w = np.ones(data.n, dtype=np.float32)
        data.w[is_rl] = primary_w
        print(f"--primary-mass {args.primary_mass:g}: "
              f"{n_primary_rows:,} primary rows at {primary_w:.3f} + "
              f"{n_anchor_rows:,} anchor rows at 1.0")
    val_mask = (data.gid % 20) == 0
    train_idx = np.where(keep & ~val_mask)[0]
    val_idx = np.where(keep & val_mask)[0]
    print(f"{len(paths)} shards, {data.n} rows -> {len(train_idx)} train / "
          f"{len(val_idx)} val ({len(np.unique(data.gid))} games)")

    count_frac = (count_fraction_table_stream(data, train_idx) if args.stream
                  else count_fraction_table(data, train_idx))

    state_h = tuple(int(x) for x in args.state_h.split(","))
    head_h = tuple(int(x) for x in args.head_h.split(","))
    if not args.no_extra and not data.has_extra:
        raise SystemExit(f"{args.ds} was built before the v4 state block; "
                         "rebuild it or pass --no-extra")
    if args.attr and not data.has_attr:
        raise SystemExit(f"{args.ds} was built before the v6 attribute block; "
                         "rebuild it with scripts/build_policy_dataset.py")
    if args.drop_a and not args.attr:
        raise SystemExit("--drop-a ablates the v6 block; it needs --attr")
    if args.pool and args.no_extra:
        raise SystemExit("--pool is the v5 block and is defined as v4 + pool; "
                         "inference only knows the v4 and v5 state widths")
    model = PolicyNet(state_h=state_h, head_h=head_h, dropout=args.dropout,
                      opt_cols=args.opt_cols, extra=not args.no_extra,
                      pool=args.pool, outcome=args.aux_outcome_w > 0,
                      count=args.aux_count_w > 0,
                      adapter_names=adapter_names or None,
                      adapter_h=args.adapter_h,
                      adapters_off=args.adapters_off,
                      attr=args.attr, rows=rows, pad=args.pad).to(device)
    if args.init:
        load_init(model, ROOT / args.init)
    if args.freeze_except:
        if not args.init:
            raise SystemExit("--freeze-except without --init trains a few "
                             "layers on top of RANDOM embeddings; that is not "
                             "a fine-tune of anything")
        apply_freeze(model, args.freeze_except)
    if args.adapters_off and model.adapters is not None:
        # Control: adapters exist for export shape parity but must not train.
        for p in model.adapters.parameters():
            p.requires_grad_(False)
    if adapter_names:
        route_counts = {
            ROUTE_NAMES[rid]: int((data.routes == rid).sum())
            for rid in sorted(set(data.routes.tolist()))
        }
        print(f"E2 routes: {route_counts} adapters={adapter_names} "
              f"h={args.adapter_h} off={args.adapters_off}")
    print(f"arch: state{list(state_h)} head{list(head_h)} loss={args.loss} "
          f"opt_cols={args.opt_cols}/{OPT_DENSE} "
          f"extra={not args.no_extra} pool={args.pool} attr={args.attr} "
          f"vocab={bool(args.vocab)} pad={args.pad} "
          f"emb_params={sum(e.weight.numel() for e in (model.slot_emb, model.bag_emb, model.card_emb, model.atk_emb)):,} "
          f"aux_outcome={args.aux_outcome_w:g} "
          f"aux_count={args.aux_count_w:g} "
          f"adapters={adapter_names or []} "
          f"device={device.type} "
          f"params={sum(p.numel() for p in model.parameters())}")
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = (torch.optim.AdamW(trainable, lr=args.lr, weight_decay=args.wd)
           if trainable else None)
    bcef = nn.BCEWithLogitsLoss()
    bce_none = nn.BCEWithLogitsLoss(reduction="none")
    rng = np.random.default_rng(args.seed)
    best = -1.0

    t_start = time.time()
    for epoch in range(args.epochs):
        model.train()
        t0 = time.time()
        tot = seen = 0.0
        for batch in data.batches(train_idx, args.bs, rng):
            (dense, slots, bf, bo, seld, odn, ocd, oat, otg, orow, om,
             spans, rw, _sel, xd, xs, at, routes) = batch
            dense, slots, seld = (dense.to(device), slots.to(device),
                                  seld.to(device))
            bf = {k: v.to(device) for k, v in bf.items()}
            bo = {k: v.to(device) for k, v in bo.items()}
            odn, ocd, oat = odn.to(device), ocd.to(device), oat.to(device)
            otg, orow, om = otg.to(device), orow.to(device), om.to(device)
            rw, xd, xs = rw.to(device), xd.to(device), xs.to(device)
            at = at.to(device)
            routes = routes.to(device)
            if opt is not None:
                opt.zero_grad()
            need_state = args.aux_outcome_w > 0 or args.aux_count_w > 0
            result = model(dense, slots, bf, bo, seld, odn, ocd, oat, otg,
                           orow, xd, xs, at, routes=routes,
                           return_state=need_state)
            if need_state:
                out, srepr = result
            else:
                out, srepr = result, None
            loss = torch.zeros((), dtype=out.dtype)
            wrow = rw if (args.rating_temp > 0 or args.advantage > 0
                          or args.primary_mass > 0) else None
            if args.loss in ("bce", "both"):
                if wrow is None:
                    loss = loss + bcef(out, om)
                else:   # the row's weight applies to each of its options
                    per = bce_none(out, om) * wrow[orow]
                    loss = loss + per.sum() / wrow[orow].sum().clamp_min(1e-8)
            if args.loss in ("listwise", "both"):
                loss = loss + listwise_loss(out, om, orow, len(spans), wrow)
            if args.aux_outcome_w > 0:
                pred = model.outcome_head(srepr).squeeze(1)
                target = torch.from_numpy(data.won[_sel]).to(
                    device=device, dtype=pred.dtype)
                per = bce_none(pred, target)
                aux = (per.mean() if wrow is None else
                       (per * rw).sum() / rw.sum().clamp_min(1e-8))
                loss = loss + args.aux_outcome_w * aux
            if args.aux_count_w > 0:
                pred = model.count_head(srepr).squeeze(1)
                target, valid = count_targets(seld, om, orow)
                if valid.any():
                    per = bce_none(pred[valid], target[valid])
                    aux = (per.mean() if wrow is None else
                           (per * rw[valid]).sum()
                           / rw[valid].sum().clamp_min(1e-8))
                    loss = loss + args.aux_count_w * aux
            if opt is not None and loss.requires_grad:
                loss.backward()
                opt.step()
            tot += float(loss.detach()) * len(om)
            seen += len(om)
        # val: top-1 accuracy on single-choice rows
        model.eval()
        hit = tries = 0
        hit_hi = tries_hi = 0
        route_hit = {name: 0 for name in ROUTE_NAMES.values()}
        route_tries = {name: 0 for name in ROUTE_NAMES.values()}
        out_bce = out_ok = out_n = 0.0
        count_abs = count_n = 0.0
        with torch.no_grad():
            for batch in data.batches(val_idx, args.bs, None):
                (dense, slots, bf, bo, seld, odn, ocd, oat, otg, orow, om,
                 spans, _rw, vsel, xd, xs, at, routes) = batch
                dense, slots, seld = (dense.to(device), slots.to(device),
                                      seld.to(device))
                bf = {k: v.to(device) for k, v in bf.items()}
                bo = {k: v.to(device) for k, v in bo.items()}
                odn, ocd, oat = odn.to(device), ocd.to(device), oat.to(device)
                otg, orow, om = otg.to(device), orow.to(device), om.to(device)
                xd, xs = xd.to(device), xs.to(device)
                at = at.to(device)
                routes = routes.to(device)
                need_state = args.aux_outcome_w > 0 or args.aux_count_w > 0
                result = model(dense, slots, bf, bo, seld, odn, ocd, oat, otg,
                               orow, xd, xs, at, routes=routes,
                               return_state=need_state)
                if need_state:
                    out, srepr = result
                else:
                    out, srepr = result, None
                if args.aux_outcome_w > 0:
                    pred = model.outcome_head(srepr).squeeze(1)
                    target = torch.from_numpy(data.won[vsel]).to(
                        device=device, dtype=pred.dtype)
                    out_bce += float(bce_none(pred, target).sum())
                    out_ok += float(((pred >= 0) == (target >= 0.5)).sum())
                    out_n += len(target)
                if args.aux_count_w > 0:
                    pred = model.count_head(srepr).squeeze(1)
                    target, valid = count_targets(seld, om, orow)
                    if valid.any():
                        count_abs += float(
                            (torch.sigmoid(pred[valid]) - target[valid])
                            .abs().sum())
                        count_n += float(valid.sum())
                out = out.cpu().numpy()
                om = om.cpu().numpy()
                routes_np = routes.cpu().numpy()
                pos = 0
                for j, (a, b) in enumerate(spans):
                    k = b - a
                    sc = out[pos:pos + k]
                    ch = om[pos:pos + k]
                    pos += k
                    if ch.sum() == 1:
                        ok = ch[np.argmax(sc)] == 1
                        hit += ok
                        tries += 1
                        rname = ROUTE_NAMES.get(int(routes_np[j]), "general")
                        route_hit[rname] += int(ok)
                        route_tries[rname] += 1
                        # Same rows, restricted to strong demonstrators. Rule 3
                        # still holds -- neither number predicts strength; this
                        # one just says WHOSE policy is being fit.
                        if data.rating[vsel[j]] >= VAL_HI_RATING:
                            hit_hi += ok
                            tries_hi += 1
        acc = hit / max(tries, 1)
        hi = hit_hi / max(tries_hi, 1)
        aux_msg = ""
        if args.aux_outcome_w > 0:
            aux_msg += (f" val_out_bce={out_bce / max(out_n, 1):.4f}"
                        f" val_out_acc={out_ok / max(out_n, 1):.4f}")
        if args.aux_count_w > 0:
            aux_msg += f" val_count_mae={count_abs / max(count_n, 1):.4f}"
        if adapter_names:
            for rname in ("mirror", "alakazam", "general"):
                rt = route_tries[rname]
                if rt:
                    aux_msg += (f" val_top1_{rname}="
                                f"{route_hit[rname] / rt:.4f}(n={rt})")
        print(f"epoch {epoch}: train={tot / seen:.4f} val_top1={acc:.4f} "
              f"val_top1@{VAL_HI_RATING:.0f}+={hi:.4f} (n={tries_hi})"
              f"{aux_msg} "
              f"({time.time() - t0:.0f}s)")
        # 🔴 rule 3. The default here SELECTS THE CHECKPOINT BY `val_top1`, and
        # that metric is measured not to predict strength in either direction
        # (§8z moved it by 8 decisions for +37 Elo; §8aa moved it by 214 for
        # +14 -- a 70x exchange-rate difference). It is tolerable for a plain
        # clone, whose objective IS corpus fit. It is NOT tolerable for any arm
        # whose objective deliberately departs from corpus fit: an
        # advantage-weighted arm will fit the corpus worse ON PURPOSE, stop
        # improving `acc` earlier, and export an EARLIER EPOCH than its control
        # -- so the A/B would be comparing epoch counts, not the intervention.
        # `--export-last` is mandatory for both arms of such a pair.
        if args.export_last:
            if epoch == args.epochs - 1:
                export_npz(model, ROOT / args.out, count_frac, x_mask, a_mask,
                           remap)
        elif acc > best:
            best = acc
            export_npz(model, ROOT / args.out, count_frac, x_mask, a_mask,
                           remap)
        # 🔴 A HOSTED RUN THAT IS KILLED COMMITS NOTHING. Kaggle terminates a
        # batch kernel at its 12 h cap and discards /kaggle/working with it, so
        # a checkpoint written every epoch still reaches nobody -- the process
        # has to EXIT CLEANLY for the output to be saved. `--max-hours` turns
        # "trained 11 h, delivered nothing" into "trained N epochs, exported
        # the best one, exit 0". Set it BELOW the platform cap, not at it.
        if args.max_hours and (time.time() - t_start) > args.max_hours * 3600:
            print(f"--max-hours {args.max_hours:g} reached after epoch {epoch} "
                  f"({(time.time() - t_start) / 3600:.2f} h). Stopping cleanly "
                  f"so the export is committed; chain the rest with "
                  f"`--init {args.out}`.")
            break
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile scripts/filter_corpus.py
"""Rewrite a policy corpus keeping only the rows of selected episodes.

WHY THIS EXISTS (EVIDENCE §1a follow-up). `train_policy.py --keep-gids` filters
rows *after* the loader has already materialised them, so under `--stream` every
epoch still opens and fully decompresses every shard that holds at least one
surviving row. A global top-10% cut is spread across all 59 days, so that is
essentially all 701 shards: the gradient steps drop to 10% but the I/O does not
move, and "10% of the data" does not buy 10% of the time.

It also quietly shrinks the shuffle. `StreamData` buffers `--stream-buffer`
whole shards and trains on whatever survives the mask inside them, so a 10% cut
turns a 458k-row shuffle pool into a ~46k-row one at the same `--stream-buffer`
-- a confound §1a's measured 0.0012 bound does NOT cover, because that was
measured with no mask in play.

This script does the cut ONCE, off the critical path, and repacks the survivors
into full-size shards. The training run then reads a corpus that is 10% of the
bytes and buffers the same number of trainable rows per buffer as the unfiltered
run did, so `--stream-buffer` keeps its old meaning.

    python -X utf8 scripts/filter_corpus.py \
        --ds artifacts/pds_hostall --keep-gids out/keep_gids.txt \
        --out artifacts/pds_top10

⚠ `gid` IS the episode id (EVIDENCE §1a), which is what makes this a row mask
over the existing shards rather than a rebuild.
"""
from __future__ import annotations

import argparse
import glob
import time
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[1]

# Kept in sync with train_policy.BAGS. Imported rather than re-declared would be
# better, but this script has to run inside a Kaggle kernel where only the
# embedded files exist, and it must not drag the trainer's torch import in.
BAGS = ("my_hand", "my_discard", "opp_discard")

# Ragged blocks: offset key -> value keys it indexes. Values are addressed by
# the offsets, not by row, so both sides have to be rebuilt under a mask.
RAGGED: dict[str, list[str]] = {
    "opt_off": ["opt_dense", "opt_card", "opt_attack", "opt_target",
                "opt_chosen"],
}
RAGGED.update({f"bag_{nm}_off": [f"bag_{nm}_flat"] for nm in BAGS})

# Every key the ragged rebuild owns. Anything else in the shard is per-row and
# is carried through generically -- ⚠ do NOT reintroduce a hardcoded row-key
# list here. The first version of this script had one and silently dropped
# `opp_rating`, `sub_id` and `team_id`, which is the join from a row back to
# its demonstrator's leaderboard rating.
_RAGGED_KEYS = set(RAGGED) | {v for vs in RAGGED.values() for v in vs}


def slice_shard(z, keep: np.ndarray) -> dict[str, np.ndarray]:
    """One shard's arrays, restricted to the rows `keep` selects."""
    n = len(keep)
    out: dict[str, np.ndarray] = {}
    for off_key, val_keys in RAGGED.items():
        if off_key not in z:
            continue
        off = z[off_key].astype(np.int64)
        lens = np.diff(off)[keep]
        total = int(lens.sum())
        # Global positions of every element belonging to a kept row, in row
        # order. `cumsum(lens) - lens` is the exclusive prefix sum, so this maps
        # each new element back to its original slot across the gaps the mask
        # opens up.
        starts = off[:-1][keep]
        idx = (np.repeat(starts - (np.cumsum(lens) - lens), lens)
               + np.arange(total, dtype=np.int64))
        new_off = np.zeros(len(lens) + 1, dtype=np.int64)
        np.cumsum(lens, out=new_off[1:])
        out[off_key] = new_off
        for vk in val_keys:
            if vk in z:
                if len(z[vk]) != off[-1]:
                    raise SystemExit(
                        f"{vk} has {len(z[vk]):,} elements but {off_key} ends "
                        f"at {off[-1]:,}; the shard's ragged block is "
                        f"inconsistent and slicing it would corrupt it")
                out[vk] = z[vk][idx]
    for k in z.files:
        if k in _RAGGED_KEYS:
            continue
        a = z[k]
        if a.ndim >= 1 and a.shape[0] == n:
            out[k] = a[keep]
        else:
            # rule 9: never write a corpus with a column we did not understand.
            raise SystemExit(
                f"shard key {k!r} has shape {a.shape}, which is neither ragged "
                f"nor per-row against {n:,} rows. Teach filter_corpus.py what "
                f"it is rather than dropping or mis-slicing it.")
    return out


def merge(parts: list[dict[str, np.ndarray]]) -> dict[str, np.ndarray]:
    """Concatenate sliced shards into one, rebuilding every offset array."""
    if len(parts) == 1:
        return parts[0]
    out: dict[str, np.ndarray] = {}
    for off_key, val_keys in RAGGED.items():
        if off_key not in parts[0]:
            continue
        lens = np.concatenate([np.diff(p[off_key]) for p in parts])
        off = np.zeros(len(lens) + 1, dtype=np.int64)
        np.cumsum(lens, out=off[1:])
        out[off_key] = off
        for vk in val_keys:
            present = [p[vk] for p in parts if vk in p]
            if present:
                out[vk] = np.concatenate(present)
    for k in parts[0]:
        if k not in _RAGGED_KEYS:
            out[k] = np.concatenate([p[k] for p in parts])
    return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--ds", required=True, help="corpus dir to filter")
    ap.add_argument("--keep-gids", required=True,
                    help="one episode id per line (leading column of a CSV ok)")
    ap.add_argument("--out", required=True, help="corpus dir to write")
    ap.add_argument("--rows-per-shard", type=int, default=57000,
                    help="target rows per output shard. The default matches the "
                         "hostall corpus's own 57,243, so --stream-buffer keeps "
                         "the meaning it had in the unfiltered run.")
    args = ap.parse_args()

    src = Path(args.ds)
    if not src.is_absolute():
        src = ROOT / src
    shards = sorted(glob.glob(f"{src}/**/shard_*.npz", recursive=True))
    if not shards:
        raise SystemExit(f"no shards under {src}")

    kp = Path(args.keep_gids)
    if not kp.is_absolute():
        kp = ROOT / kp
    want: set[int] = set()
    for ln in kp.read_text(encoding="utf-8-sig").splitlines():
        tok = ln.split(",", 1)[0].strip()
        if tok and tok.lstrip("-").isdigit():
            want.add(int(tok))
    if not want:
        raise SystemExit(f"{kp}: parsed zero episode ids")
    want_arr = np.fromiter(want, dtype=np.int64, count=len(want))

    dst = Path(args.out)
    if not dst.is_absolute():
        dst = ROOT / dst
    dst.mkdir(parents=True, exist_ok=True)

    print(f"{len(shards)} shards under {src}", flush=True)
    print(f"--keep-gids {kp.name}: {len(want):,} episode ids", flush=True)

    buf: list[dict[str, np.ndarray]] = []
    buf_rows = 0
    n_out = 0
    tot_in = tot_kept = 0
    eps: set[int] = set()
    t0 = time.time()

    def flush() -> None:
        nonlocal buf, buf_rows, n_out
        if not buf:
            return
        merged = merge(buf)
        path = dst / f"shard_{n_out:04d}.npz"
        np.savez_compressed(path, **merged)
        n_out += 1
        buf, buf_rows = [], 0

    for i, p in enumerate(shards):
        with np.load(p) as z:
            gid = z["gid"]
            tot_in += len(gid)
            keep = np.isin(gid, want_arr)
            k = int(keep.sum())
            if not k:
                continue
            tot_kept += k
            eps.update(np.unique(gid[keep]).tolist())
            buf.append(slice_shard(z, keep))
            buf_rows += k
        if buf_rows >= args.rows_per_shard:
            flush()
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(shards)} shards, {tot_kept:,} rows kept "
                  f"({time.time() - t0:.0f}s)", flush=True)
    flush()

    if not tot_kept:
        raise SystemExit("kept ZERO rows; the id space is wrong (gid is the "
                         "episode id)")
    # rule 9: a filter that matches almost nothing must not look like a small
    # corpus. Report what survived in rows AND in episodes, against the ask.
    print(f"\n{tot_kept:,} of {tot_in:,} rows kept ({tot_kept / tot_in:.1%}), "
          f"{len(eps):,} of {len(want):,} requested episodes matched")
    print(f"wrote {n_out} shards to {dst} in {time.time() - t0:.0f}s")
    if len(eps) < len(want):
        print(f"⚠ {len(want) - len(eps):,} requested episodes are not in this "
              f"corpus at all")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/label_archetypes.py
"""Tag each episode in a policy corpus with the archetype the DEMONSTRATOR played.

The host corpus carries no archetype label, and RUN 2 ("top 20% of Grimmsnarl
games", EVIDENCE §1a follow-up) needs one. This recovers it from the shards
themselves -- no replay re-read -- because `slots` already holds the card ids of
everything in play.

⚠ SIDE MATTERS. `slots` is 12 wide: 0..5 are the demonstrator's own active and
bench, 6..11 are the opponent's (agents/sa/features.py, `slot_ids`). We are
cloning the demonstrator, so an episode is "Grimmsnarl" when GRIMMSNARL IS ON
SLOTS 0..5. Reading all 12 would label every game *against* Grimmsnarl as a
Grimmsnarl game, which in a field where it is the dominant deck is most of them.

Detection is by signature card, not by decklist equality: the Marnie's line has
half a dozen variants in `decks/` (grimmsnarl, _boss, _budew, _xerosic, _g4 ...)
and they are all the same archetype for this purpose.

    python -X utf8 scripts/label_archetypes.py --ds artifacts/pds_hostall \
        --out out/episode_archetypes.csv

Writes `episode_id,archetype`, one row per episode, plus a census to stdout.
"""
from __future__ import annotations

import argparse
import csv
import glob
import time
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[1]

# Signature card ids per archetype. A card belongs here only if seeing it on a
# player's own side is on its own strong evidence of the archetype -- so the
# evolution line, never the generic trainers (Rare Candy, Boss's Orders and
# Night Stretcher are in half the decks in `decks/`).
ARCHETYPES: dict[str, set[int]] = {
    # decks/grimmsnarl.py: Marnie's Impidimp / Morgrem / Grimmsnarl ex.
    # Impidimp is the basic of the line, so it reaches the bench in essentially
    # every game the deck actually plays -- which is what makes slot recall high.
    "grimmsnarl": {646, 647, 648},
}

# The demonstrator's own slots. See the module docstring -- this is the whole
# correctness argument of the script.
MY_SLOTS = slice(0, 6)


def label_corpus(shards: list[str], archetypes: dict[str, set[int]],
                 use_bags: bool = False, verbose: bool = True
                 ) -> tuple[dict[int, set[str]], int]:
    """episode id -> set of archetypes seen on the demonstrator's side."""
    hits: dict[int, set[str]] = {}
    seen: set[int] = set()
    sig = {name: np.fromiter(ids, dtype=np.int32, count=len(ids))
           for name, ids in archetypes.items()}
    t0 = time.time()
    for i, p in enumerate(shards):
        with np.load(p) as z:
            gid = z["gid"]
            seen.update(gid.tolist())
            cards = [z["slots"][:, MY_SLOTS]]
            if use_bags:
                # my_hand / my_discard are the demonstrator's too, and they
                # catch a line that was drawn or discarded without ever being
                # benched. Ragged, so they are matched per-row via the offsets.
                for nm in ("my_hand", "my_discard"):
                    flat, off = z[f"bag_{nm}_flat"], z[f"bag_{nm}_off"]
                    for name, ids in sig.items():
                        m = np.isin(flat, ids)
                        if not m.any():
                            continue
                        rows = np.searchsorted(off, np.flatnonzero(m),
                                               side="right") - 1
                        for g in np.unique(gid[rows]).tolist():
                            hits.setdefault(g, set()).add(name)
            block = np.concatenate(cards, axis=1)
            for name, ids in sig.items():
                m = np.isin(block, ids).any(axis=1)
                if m.any():
                    for g in np.unique(gid[m]).tolist():
                        hits.setdefault(g, set()).add(name)
        if verbose and (i + 1) % 100 == 0:
            print(f"  scanned {i + 1}/{len(shards)} shards "
                  f"({time.time() - t0:.0f}s)", flush=True)
    return hits, len(seen)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--ds", required=True, help="corpus dir to label")
    ap.add_argument("--out", default="", help="CSV to write (episode_id,archetype)")
    ap.add_argument("--bags", action="store_true",
                    help="also match my_hand/my_discard, not just the board. "
                         "Higher recall, much slower.")
    args = ap.parse_args()

    src = Path(args.ds)
    if not src.is_absolute():
        src = ROOT / src
    shards = sorted(glob.glob(f"{src}/**/shard_*.npz", recursive=True))
    if not shards:
        raise SystemExit(f"no shards under {src}")
    print(f"{len(shards)} shards under {src}", flush=True)

    hits, n_seen = label_corpus(shards, ARCHETYPES, use_bags=args.bags)

    # rule 9: report the census, in episodes, against the total -- a labeller
    # that matches almost nothing must not be mistaken for a rare archetype.
    print(f"\n{n_seen:,} episodes scanned")
    for name in ARCHETYPES:
        k = sum(1 for v in hits.values() if name in v)
        print(f"  {name:14s} {k:7,} ({k / max(n_seen, 1):6.1%})")
    multi = sum(1 for v in hits.values() if len(v) > 1)
    if multi:
        print(f"  ⚠ {multi:,} episodes matched more than one archetype")
    unlabelled = n_seen - len(hits)
    print(f"  {'other':14s} {unlabelled:7,} ({unlabelled / max(n_seen, 1):6.1%})")

    if args.out:
        op = Path(args.out)
        if not op.is_absolute():
            op = ROOT / op
        op.parent.mkdir(parents=True, exist_ok=True)
        with open(op, "w", encoding="utf-8", newline="") as fh:
            w = csv.writer(fh)
            w.writerow(["episode_id", "archetype"])
            for g in sorted(hits):
                w.writerow([g, "|".join(sorted(hits[g]))])
        print(f"\nwrote {len(hits):,} labelled episodes -> {op}")


if __name__ == "__main__":
    main()


## The engine


In [ ]:
# ── the `cg` engine ──────────────────────────────────────────────────────────
# `sa/cards.py` reads the card and attack tables out of the engine, so the
# FEATURIZER needs it even though this notebook never plays a game.
# `kiyotah/cg-lib` is the package the host sample notebooks ship (same api.py,
# sim.py and libcg.so as the competition's sample_submission).
import glob, logging, kagglehub, kagglehub.clients as _khc

logging.getLogger("kagglehub").setLevel(logging.ERROR)   # 23,600 log lines otherwise


class _QuietBar:                # kagglehub draws a tqdm bar per FILE; that is
    def __init__(self, *a, **k): pass          # one per episode. Kill it or the
    def __enter__(self): return self           # notebook output eats the browser.
    def __exit__(self, *a): return False
    def update(self, *a, **k): pass


_khc.tqdm = _QuietBar

# ⚠ The engine is MOUNTED too, not downloaded. A runtime kagglehub attach dies
# with "New Datasets cannot be attached in non-interactive sessions" in a batch
# kernel, and the HTTP fallback spends the same 429 budget the episodes need.
# `kiyotah/cg-lib` is declared in dataset_sources alongside the episode days.
cg_dir = None
for _c in glob.glob("/kaggle/input/**/cg/api.py", recursive=True):
    cg_dir = os.path.dirname(os.path.dirname(_c))
    break
if cg_dir is None:
    # Last resort: one HTTP request, which the budget can afford.
    os.environ["DISABLE_KAGGLE_CACHE"] = "1"
    print("cg-lib not mounted; falling back to a single HTTP fetch")
    cg_dir = kagglehub.dataset_download("kiyotah/cg-lib")
# `ptcg.config.find_sdk_dir()` globs data/**/cg/api.py, so drop it where the
# repo already looks rather than special-casing sys.path.
dst = f"{ROOT}/data/cg_sdk"
if not os.path.exists(f"{dst}/cg/api.py"):
    os.makedirs(dst, exist_ok=True)
    shutil.copytree(os.path.join(cg_dir, "cg"), f"{dst}/cg", dirs_exist_ok=True)
print("cg engine ->", dst)


In [ ]:
# ── the featurizer must agree with the engine before anything else runs ──────
from ptcg.env import sdk
sdk.load()
from sa import cards as cdb
from sa.features import DENSE_DIM, N_ATTR, N_EXTRA
from sa.optfeat import OPT_DENSE

print(f"cards={len(cdb.cards())}  attacks={len(cdb.attacks())}")
print(f"DENSE_DIM={DENSE_DIM}  N_EXTRA={N_EXTRA}  N_ATTR={N_ATTR}  OPT_DENSE={OPT_DENSE}")
assert len(cdb.cards()) > 1000, "engine loaded but the card table is empty"
assert OPT_DENSE >= OPT_COLS, f"corpus writes {OPT_DENSE} option cols, recipe slices {OPT_COLS}"


## The corpus


In [ ]:
# ── what actually got built, and will it fit in RAM ──────────────────────────
import glob
import numpy as np

shards = sorted(glob.glob(f"{CORPUS_DIR}/**/shard_*.npz", recursive=True))
rows = opts = 0
games = set()
for p in shards:
    z = np.load(p)
    rows += len(z["gid"])
    opts += len(z["opt_chosen"])
    games.update(np.unique(z["gid"]).tolist())

disk = sum(os.path.getsize(p) for p in shards)
print(f"{len(shards)} shards under {CORPUS_DIR}")
print(f"{rows:,} decisions from {len(games):,} games, {opts:,} options "
      f"({opts / max(rows, 1):.1f} per decision)")
print(f"{disk / 1e6:.1f} MB on disk")

# ⚠ The kill happens while the corpus LOADS, not while it trains.
# `train_policy.Data` appends every shard's arrays to per-key lists, then
# concatenates -- and the lists stay referenced until __init__ returns, so both
# copies are resident at once. Measured on artifacts/pds_all: 4.0 KB/row
# resident, ~7.8 KB/row at that peak.
steady, peak = rows * 4.0e-6, rows * 7.8e-6
try:
    import psutil
    have = psutil.virtual_memory().total / 1e9
except Exception:
    have = float("nan")
print(f"\ncorpus load peaks at ~{peak:.1f} GB (settling to ~{steady:.1f} GB); "
      f"this machine has {have:.0f} GB")
if peak > 0.80 * have:
    print("⚠ TOO TIGHT. Lower EPISODES_PER_DAY, delete the shard dirs for the "
          "days you want thinned, and re-run the fetch cell -- the trainer "
          "would otherwise be OOM-killed partway through loading, after you "
          "have already paid for the downloads.")
assert rows > 0, "empty corpus"


## Training


In [ ]:

# ── the rating cut: which episodes survive ───────────────────────────────────
# ⚡ The 0.440 result (EVIDENCE §1a) says UNFILTERED volume is negative and the
# suspect is composition: v5_s2 cloned the top ~400/day (cutoff ~1150) while the
# full corpus reaches down to avg_score ~700-900. This applies a quality bar to
# the SAME shards, so composition moves and nothing else does.
import csv as _csv, glob as _glob

if TOP_PCT and TOP_PCT > 0:
    cand = _glob.glob("/kaggle/input/**/episode_ratings.csv", recursive=True)
    if not cand:
        raise SystemExit("TOP_PCT set but episode_ratings.csv is not attached; "
                         "add siamrahman29/ptcg-episode-ratings")
    with open(cand[0], encoding="utf-8-sig", newline="") as fh:
        rows = [(int(r["episode_id"]), float(r["avg_score"]))
                for r in _csv.DictReader(fh)]
    if ARCHETYPE:
        # ⚠ RUN 2 ranks WITHIN the archetype ("top N% OF Grimmsnarl games"), not
        # globally-then-filtered. The consequence, and it must be recorded with
        # the result: the quality bar FLOATS to wherever this archetype's own
        # rating distribution sits, so the cutoff below is NOT the global one
        # and the two runs are not comparable on cutoff alone.
        from scripts.label_archetypes import label_corpus, ARCHETYPES as _ARCH
        if ARCHETYPE not in _ARCH:
            raise SystemExit(f"unknown ARCHETYPE {ARCHETYPE!r}; "
                             f"known: {sorted(_ARCH)}")
        _sh = sorted(_glob.glob(f"{CORPUS_DIR}/**/shard_*.npz", recursive=True))
        _hits, _seen = label_corpus(_sh, {ARCHETYPE: _ARCH[ARCHETYPE]})
        keep_arch = {g for g, v in _hits.items() if ARCHETYPE in v}
        share = len(keep_arch) / max(_seen, 1)
        print(f"ARCHETYPE {ARCHETYPE}: {len(keep_arch):,} of {_seen:,} "
              f"episodes on the DEMONSTRATOR's side ({share:.1%})")
        # Pre-registered gate. The two ways a label silently produces a wrong
        # corpus are matching nothing and matching everything -- neither is a
        # cut, and both would train happily and read as a real result.
        if not 0.02 <= share <= 0.98:
            raise SystemExit(
                f"archetype share {share:.1%} is outside [2%, 98%]: the "
                f"labeller is not discriminating. Refusing to train.")
        rows = [r for r in rows if r[0] in keep_arch]
        print(f"  ranking within the archetype: {len(rows):,} rated episodes")
    rows.sort(key=lambda r: -r[1])
    k = max(1, int(len(rows) * TOP_PCT / 100.0))
    KEEP_GIDS = f"{ROOT}/keep_gids.txt"
    with open(KEEP_GIDS, "w", encoding="utf-8") as fh:
        # chr(10), not a backslash escape: this cell is embedded in the
        # generator as a string literal, where a newline escape is one more
        # level of quoting to get wrong.
        fh.write(chr(10).join(str(e) for e, _ in rows[:k]))
    scope = f"within {ARCHETYPE}" if ARCHETYPE else "global"
    print(f"TOP_PCT {TOP_PCT}% ({scope}): {k:,} of {len(rows):,} episodes, "
          f"avg_score cutoff {rows[k-1][1]:.0f} -> {KEEP_GIDS}")
    print(f"  (for reference v5_s2's own corpus cut at ~1150)")
    if ARCHETYPE:
        print(f"  ⚠ this cutoff is the {ARCHETYPE} distribution's own, NOT the "
              f"global one -- record it with the result")
else:
    print("TOP_PCT 0: every episode in the corpus is kept")


In [ ]:

# ── apply the cut to the SHARDS, once, before any epoch runs ─────────────────
# ⚠ WHY THIS CELL EXISTS. `--keep-gids` masks rows AFTER the loader has
# materialised them, so under --stream every epoch still opens and fully
# decompresses every shard holding at least one surviving row. A GLOBAL top-N%
# cut is spread across all 59 days, so that is essentially all 701 shards: the
# gradient steps drop to N% of the rows and the I/O does not move at all.
# "10% of the data" would not have bought 10% of the time.
#
# It also silently shrinks the shuffle. StreamData buffers WHOLE shards and
# trains on whatever survives the mask inside them, so at the same
# --stream-buffer a 10% cut holds ~10x fewer TRAINABLE rows than the unfiltered
# run did -- a confound EVIDENCE §1a's measured 0.0012 bound does NOT cover,
# because it was measured with no mask in play.
#
# Cutting the shards once fixes both at once: the corpus the epochs read is N%
# of the BYTES, and --stream-buffer keeps the meaning it had in the unfiltered
# run. Verified against `Data()` masked to the same gids -- all 23 shard keys
# preserved, every array bit-identical.
if KEEP_GIDS:
    FILTERED = f"artifacts/pds_top{TOP_PCT}" + (f"_{ARCHETYPE}" if ARCHETYPE else "")
    rc = subprocess.run([sys.executable, "-X", "utf8",
                         "scripts/filter_corpus.py",
                         "--ds", CORPUS_DIR,
                         "--keep-gids", KEEP_GIDS,
                         "--out", f"{ROOT}/{FILTERED}"], cwd=ROOT).returncode
    if rc != 0:
        raise SystemExit(f"filter_corpus.py exited {rc}")
    CORPUS, CORPUS_DIR = FILTERED, f"{ROOT}/{FILTERED}"
    # The cut is baked into the shards now. Passing it again would be a no-op
    # that still costs an np.isin over every row of every buffer, and it would
    # make the log read as though the mask were doing the work.
    KEEP_GIDS = ""
    print(f"training corpus is now {CORPUS_DIR}")
else:
    print("TOP_PCT 0: no cut, training on the corpus as built")


In [ ]:
# ── train: the v5_s2 command, unchanged except --ds and --out ────────────────
import torch

device = DEVICE
if device == "cuda":
    # ⚠ `torch.cuda.is_available()` is NOT sufficient on Kaggle. A session that
    # gets a **Tesla P100** reports True and then fails on the first kernel
    # launch: the image ships torch 2.10+cu128, which supports sm_70..sm_120,
    # and the P100 is sm_60. Measured, not assumed -- a GPU probe kernel came
    # back "Tesla P100 ... is not compatible with the current PyTorch install".
    # Crashing here costs a corpus build; crashing an hour in costs the session.
    cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None
    if cap is None:
        print("⚠ no CUDA in this session; using cpu (which is what v5_s2 used)")
        device = "cpu"
    elif cap[0] < 7:
        name = torch.cuda.get_device_name(0)
        print(f"⚠ {name} is compute {cap[0]}.{cap[1]}; this image's torch needs "
              f">= 7.0. Using cpu. Pick the T4 x2 accelerator, not P100, if you "
              f"want the GPU.")
        device = "cpu"
    else:
        print(f"cuda: {torch.cuda.get_device_name(0)} (compute {cap[0]}.{cap[1]})")

cmd = [sys.executable, "-X", "utf8", "scripts/train_policy.py",
       "--ds", CORPUS,
       "--epochs", str(EPOCHS),
       "--bs", str(BS),
       "--loss", LOSS,
       "--state-h", STATE_H,
       "--head-h", HEAD_H,
       "--opt-cols", str(OPT_COLS),
       "--seed", str(SEED),
       "--max-hours", str(MAX_HOURS),
       *(["--stream", "--stream-buffer", str(STREAM_BUFFER)] if STREAM else []),
       *(["--keep-gids", KEEP_GIDS] if KEEP_GIDS else []),
       "--device", device,
       "--out", f"out/{NET_NAME}"]
if POOL:
    cmd.append("--pool")
if INIT_NET:
    import glob as _g
    hits = [p for p in _g.glob(f"/kaggle/input/**/{INIT_NET}", recursive=True)]
    if not hits:
        raise SystemExit(f"INIT_NET {INIT_NET!r} not found under /kaggle/input; "
                         f"attach the dataset that holds it")
    # ⚠ rule 20: say which FILE, not just which name. Two datasets can carry the
    # same basename and the log is the only record of which one trained this net.
    print(f"--init {hits[0]} (warm start: WEIGHTS only, optimizer state resets)")
    cmd += ["--init", hits[0]]

if SKIP_TRAIN:
    print("SKIP_TRAIN: corpus is built; stopping before training.")
    raise SystemExit

print(" ".join(cmd), "\n" + "-" * 78, flush=True)

log = []
with subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as pr:
    for line in pr.stdout:
        print(line, end="", flush=True)
        log.append(line)
    rc = pr.wait()

open(f"{ROOT}/out/{NET_NAME}.train.log", "w", encoding="utf-8").writelines(log)
assert rc == 0, f"trainer exited {rc}"


In [ ]:
# ── verify the export the way the ARENA loads it ─────────────────────────────
# ⚠ HANDOFF rule 20: a path is not an identity. This cell checks the export
# against the SHAPES of the shipped `out/policy_v5_s2.npz`, key by key -- that
# is what "exactly the v5_s2 agent, different corpus" has to mean at the
# artefact level -- and prints the fingerprint to carry into the arena.
import hashlib
import importlib
import numpy as np

import sa.policynet as pn
importlib.reload(pn)

# taken from out/policy_v5_s2.npz at notebook-generation time
V5_S2_SHAPES = {
    'atk_emb': [1600, 16],
    'bag_emb': [1300, 16],
    'card_emb': [1300, 16],
    'count_frac': [11, 64],
    'head0_b': [256],
    'head0_w': [256, 341],
    'head1_b': [128],
    'head1_w': [128, 256],
    'head2_b': [1],
    'head2_w': [1, 128],
    'n_attr': [1],
    'n_head': [1],
    'n_pool': [1],
    'n_sfc': [1],
    'sfc0_b': [512],
    'sfc0_w': [512, 708],
    'sfc1_b': [256],
    'sfc1_w': [256, 512],
    'slot_emb': [1300, 16],
}

path = f"{ROOT}/out/{NET_NAME}"
net = pn.load(path)
assert net is not None, "policynet.load() refused the export"

z = np.load(path)
got = {k: list(z[k].shape) for k in z.files}
params = sum(int(z[k].size) for k in z.files if z[k].dtype == np.float32)
digest = hashlib.sha256(open(path, "rb").read()).hexdigest()[:8]

print(f"{NET_NAME}  #{digest}  {os.path.getsize(path) / 1e6:.2f} MB")
print(f"state MLP input {got['sfc0_w'][1]}   head input {got['head0_w'][1]}   "
      f"n_pool={int(z['n_pool'][0])}   n_attr={int(z['n_attr'][0])}")
print(f"{params:,} float params in {len(z.files)} arrays")

missing = sorted(set(V5_S2_SHAPES) - set(got))
extra = sorted(set(got) - set(V5_S2_SHAPES))
diff = sorted(k for k in set(got) & set(V5_S2_SHAPES) if got[k] != V5_S2_SHAPES[k])
if missing or extra or diff:
    for k in missing:
        print(f"  MISSING {k} {V5_S2_SHAPES[k]}")
    for k in extra:
        print(f"  EXTRA   {k} {got[k]}")
    for k in diff:
        print(f"  SHAPE   {k}: {got[k]} != v5_s2's {V5_S2_SHAPES[k]}")
    raise SystemExit(
        "this net is NOT architecturally v5_s2. It cannot be A/B'd against "
        "out/policy_v5_s2.npz as a corpus change, and it cannot share an "
        "ensemble with policy_v5. Check --pool / --opt-cols / --state-h / "
        "--head-h in the CONFIG cell.")

print("\n✅ byte-for-byte the same architecture as out/policy_v5_s2.npz "
      "(every array name and shape matches). The ONLY thing that differs "
      "between this net and the shipped one is the corpus it was fitted to.")


In [ ]:
# ── leave everything downloadable in /kaggle/working ─────────────────────────
OUT = "/kaggle/working"

shutil.copy(f"{ROOT}/out/{NET_NAME}", f"{OUT}/{NET_NAME}")
shutil.copy(f"{ROOT}/out/{NET_NAME}.train.log", f"{OUT}/{NET_NAME}.train.log")

# The corpus is small and expensive to rebuild -- ship it so a re-train (another
# seed, an ablation) does not spend another hour of downloads.
corpus_zip = shutil.make_archive(f"{OUT}/{os.path.basename(CORPUS)}", "zip",
                                 root_dir=f"{ROOT}/{CORPUS}")

# The loose shards are in the zip now; the engine stays so the cells above can
# be re-run (another seed, an ablation) without re-downloading it.
shutil.rmtree(f"{ROOT}/artifacts", ignore_errors=True)

for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    if os.path.isfile(p):
        print(f"{os.path.getsize(p) / 1e6:9.2f} MB  {f}")


---

## Back in the repo

Download `policy_v5_s2_hostall.npz` and `pds_hostall.zip` from the output pane, then:

```powershell
Copy-Item <downloads>\policy_v5_s2_hostall.npz out\policy_v5_s2_hostall.npz
Expand-Archive <downloads>\pds_hostall.zip artifacts\pds_hostall

# the A/B that matters: same rules, same deck, byte-identical config, net swapped
python -X utf8 scripts/arena.py play `
    "bc:hostall,net=out/policy_v5_s2_hostall.npz,noChip,noSpread,noSrc" `
    "bc:v5s2ship,net=out/policy_v5_s2.npz,noChip,noSpread,noSrc" `
    --matches 2000 --deck-a grimmsnarl --deck-b grimmsnarl `
    --archive out/arena/hostall_vs_v5s2.jsonl
```

⚠ **Read the result against the noise floor, not against 0.500.** Two
identical-recipe nets differing only in `--seed` measure **0.482 [0.460, 0.504]**
against each other (EVIDENCE §8z), so anything inside roughly ±0.025 is a null
and *"more data helped"* is not a claim this A/B can support at n=2,000.

⚠ **`--device cuda` is not `v5_s2`'s numerics.** If the arena reads a difference
you want to attribute to the corpus, re-run the training with `DEVICE = "cpu"`
before believing it — otherwise the corpus change and the device change are
confounded.
